<details>
<summary><h2 style="display:inline;">🛠️ Setup Instructions & Pipeline Overview (Click to Expand)</h2></summary>

**Welcome to the Smart OPI-ORCA V03-01 Spectroscopic Pipeline.**

This Jupyter Notebook automates the discovery, optimization, and high-accuracy spectral prediction of molecular isomers and Van der Waals complexes with extreme academic rigor. It features explicit Monte Carlo error propagation, structural MCMC fitting, and topological data generation.

**Prerequisites:**
1. **Anaconda** or **Miniconda** installed.
2. **ORCA 6.1.1** downloaded from the [FACCTs Portal](https://faccts.de/download/).
3. **PyTorch/libtorch** C++ extensions for ORCA downloaded.

**How to Start:**
Simply run **Stage 0**. The script will automatically scan your computer. If it cannot find ORCA, it will open your browser to the download page, ask where you put the files, and use Windows PowerShell to permanently inject the paths for you. It will also build the necessary Python environment.
</details>

🛠️ Stage 0: Environment Setup

Purpose: Scans the OS, enforces strict dependencies, verifies hardware (GPU/CPU), and generates the Digital Passport and opi_system_config.json.

In [2]:
#!/usr/bin/env python3

# ==============================================================================
# 00-ENVCHK-AA
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage intelligently scans the Linux operating system to ensure the user 
# is operating within the correct "TOPOS" environment. 
# It utilizes a STRICT "Test -> Install -> Re-Test -> Local File -> Fail" 
# architecture to verify Conda dependencies, Pip modules, and ORCA External Tools.
# It features a Dynamic Dependency Resolver that iteratively upgrades/downgrades 
# to the highest mutually compatible versions of Python, NumPy, and C++ bindings.
# 
# 2. Use instructions:
# Execute this script directly in your terminal or Jupyter Notebook. 
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a rigorous Digital Passport, verifies CUDA matrices, 
# enforces OpenMPI limits, resolves C++ API collisions, and produces 
# `opi_system_config.json` alongside a reproducible `requirements_lock.txt`.
# ==============================================================================

import os
import sys
import subprocess
import shutil
import json
import multiprocessing
import tarfile
import glob
import urllib.request
import logging
import hashlib
import platform
import re

try:
    import resource
except ImportError:
    resource = None

# ==============================================================================
# TERMINAL FORMATTING & LOGGING PROTOCOLS
# ==============================================================================

class Colors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'

logging.basicConfig(
    filename='Stage0_Environment.log', 
    level=logging.DEBUG,
    format='%(asctime)s [%(levelname)s] %(message)s'
)

def print_status(msg, status="info"):
    if status == "success": 
        print(f"{Colors.OKGREEN}✅ {msg}{Colors.ENDC}")
        logging.info(f"SUCCESS: {msg}")
    elif status == "error": 
        print(f"{Colors.FAIL}❌ {msg}{Colors.ENDC}")
        logging.error(msg)
    elif status == "warning": 
        print(f"{Colors.WARNING}⚠️ {msg}{Colors.ENDC}")
        logging.warning(msg)
    else: 
        print(f"{Colors.OKCYAN}➡️ {msg}{Colors.ENDC}")
        logging.info(msg)

def hard_fail(component, reason, url=None, instruction=None):
    print(f"\n{Colors.FAIL}{Colors.BOLD}🚨 STRICT ENFORCEMENT FAILURE: {component} 🚨{Colors.ENDC}")
    print(f"Reason: {reason}")
    print(f"The automated installation failed, and no valid local package was found.")
    logging.critical(f"HARD FAIL - {component}: {reason}")
    
    if url:
        print(f"\n{Colors.OKCYAN}MANUAL INTERVENTION REQUIRED:{Colors.ENDC}")
        print(f" 1. Go to: {url}")
        if instruction:
            print(f" 2. {instruction}")
        print(f" 3. Place the file EXACTLY in this directory: {os.getcwd()}")
        print(f" 4. Re-run Stage 0. The strict scanner will detect the local file and install it.")
    sys.exit(1)

def download_reporthook(count, block_size, total_size):
    if total_size > 0:
        downloaded = count * block_size
        percent = int(downloaded * 100 / total_size)
        mb_down = downloaded / (1024 * 1024)
        mb_total = total_size / (1024 * 1024)
        sys.stdout.write(f"\r{Colors.OKCYAN}➡️ Downloading... {percent}% [{mb_down:.1f} MB / {mb_total:.1f} MB]{Colors.ENDC}")
        sys.stdout.flush()

def get_sha256(filepath):
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

# ==============================================================================
# HARDWARE & SYSTEM SCANNERS
# ==============================================================================

def get_hardware_info():
    print(f"\n{Colors.BOLD}--- Hardware Acceleration Auto-Detect ---{Colors.ENDC}")
    
    # OS Memory Allocation (ulimit) Verification & Auto-Expansion
    if resource:
        soft_stack, hard_stack = resource.getrlimit(resource.RLIMIT_STACK)
        if soft_stack != resource.RLIM_INFINITY and soft_stack < 83886080: # ~80 MB
            try:
                # Attempt to automatically maximize the soft limit to the system's hard limit
                new_soft = hard_stack if hard_stack != resource.RLIM_INFINITY else resource.RLIM_INFINITY
                resource.setrlimit(resource.RLIMIT_STACK, (new_soft, hard_stack))
                print_status("Dynamically expanded OS Stack Size (ulimit -s) to maximum allowed.", "success")
                logging.info(f"Dynamically updated ulimit -s from {soft_stack} to {new_soft}")
            except Exception as e:
                print_status(f"OS Stack Size (ulimit -s) is dangerously restricted ({soft_stack // 1024} KB).", "warning")
                print_status("Automated memory limit expansion failed (insufficient privileges).", "warning")
                print(f"\n{Colors.OKCYAN}MANUAL INTERVENTION RECOMMENDED:{Colors.ENDC}")
                print("To prevent Coupled-Cluster memory segmentation faults, run this command in your Linux Mint terminal:")
                print(f"  {Colors.BOLD}ulimit -s unlimited{Colors.ENDC}")
                print("\nIf you want to make this permanent across system reboots, run:")
                print(f"  {Colors.BOLD}echo '* soft stack unlimited' | sudo tee -a /etc/security/limits.conf{Colors.ENDC}")
                print("Then completely close and restart your Jupyter Notebook.\n")
                logging.warning(f"Restricted ulimit -s detected: {soft_stack}. Auto-expansion failed: {e}")
        else:
            print_status("OS Memory Allocation (Stack Size) is mathematically healthy.", "success")
            
    cpu_cores = multiprocessing.cpu_count()
    print_status(f"CPU Detected: {cpu_cores} threads available", "success")
    
    try:
        mem_bytes = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
        mem_gb = mem_bytes / (1024.**3)
    except Exception: mem_gb = 0.0
    print_status(f"Memory Detected: {mem_gb:.2f} GB available", "success")
    
    gpu_type = "UNKNOWN"
    torch_wheel = "cpu"
    
    try:
        if shutil.which("nvidia-smi"): 
            gpu_type = "NVIDIA"
            nvidia_out = subprocess.check_output("nvidia-smi", shell=True, text=True)
            match = re.search(r"CUDA Version:\s+(\d+\.\d+)", nvidia_out)
            if match:
                cuda_v = float(match.group(1))
                if cuda_v >= 12.4: torch_wheel = "cu124"
                elif cuda_v >= 12.1: torch_wheel = "cu121"
                elif cuda_v >= 11.8: torch_wheel = "cu118"
                else: torch_wheel = "cu118"
                print_status(f"CUDA Host Version: {cuda_v} (Routing to PyTorch wheel {torch_wheel})", "success")
        else:
            lspci_out = subprocess.check_output("lspci | grep -i -E 'vga|3d|display'", shell=True, text=True).lower()
            if "amd" in lspci_out or "radeon" in lspci_out: 
                gpu_type = "AMD"
                torch_wheel = "rocm6.0"
            elif "intel" in lspci_out: gpu_type = "INTEL"
    except Exception as e: 
        logging.error(f"Hardware scan error: {e}")
    
    if gpu_type == "UNKNOWN" or gpu_type == "INTEL":
        print_status("GPU Detection: No dedicated Tensor/CUDA GPU found. Defaulting to CPU.", "warning")
        gpu_type = "CPU"
    else:
        print_status(f"GPU Detected: {gpu_type} Architecture", "success")
        
    return {"gpu_type": gpu_type, "cpu_threads": cpu_cores, "memory_gb": mem_gb, "torch_wheel": torch_wheel}

def test_orca_execution():
    print(f"\n{Colors.BOLD}--- Deep ORCA Engine Pre-Flight Check ---{Colors.ENDC}")
    orca_cmd = shutil.which("orca")
    if orca_cmd == "/usr/bin/orca": orca_cmd = None 
        
    if not orca_cmd:
        fast_checks = [
            os.path.expanduser("~/orca_6_1_1_avx2/orca"),
            os.path.expanduser("~/orca/orca"),
            "/opt/orca/orca",
            "/usr/local/orca/orca"
        ]
        for fc in fast_checks:
            if os.path.exists(fc) and os.access(fc, os.X_OK):
                orca_cmd = fc
                break
                
    if not orca_cmd:
        print_status("Jupyter cannot find 'orca' natively.", "warning")
        try: orca_cmd = input(f"Please enter the absolute path to your ORCA executable: \n > ").strip()
        except: sys.exit(1)
        if not os.path.exists(orca_cmd):
            hard_fail("ORCA Binary", f"Path not found: {orca_cmd}")

    dummy_inp = "dummy_orca_test.inp"
    with open(dummy_inp, "w") as f:
        f.write("! r2SCAN-3c\n*xyz 0 1\nO 0 0 0\nH 0 0.75 0.5\nH 0 -0.75 0.5\n*")

    try:
        result = subprocess.run([orca_cmd, dummy_inp], capture_output=True, text=True, timeout=30)
        
        orca_version = "Unknown"
        match = re.search(r"Program Version (\d+\.\d+\.\d+)", result.stdout)
        if match: orca_version = match.group(1)
        
        if "ORCA TERMINATED NORMALLY" in result.stdout:
            print_status(f"ORCA {orca_version} validated (Basis sets & Property modules OK) at: {orca_cmd}", "success")
            for ext in [".inp", ".out", ".densities", ".gbw", "_property.txt"]:
                f_path = dummy_inp.replace(".inp", ext)
                if os.path.exists(f_path): os.remove(f_path)
            return orca_cmd, orca_version
        else:
            print_status("ORCA executed but crashed! (Missing Basis Libraries or OpenMPI?)", "error")
            logging.critical(f"ORCA Deep Check Failed:\n{result.stderr}\n{result.stdout[-1500:]}")
            raise ValueError("Abnormal Termination")
    except Exception as e:
        hard_fail("ORCA Execution", f"Deep module check failed: {e}")

# ==============================================================================
# DYNAMIC DEPENDENCY RESOLVER & STRICT VERIFICATION
# ==============================================================================

def find_conda():
    conda_path = shutil.which("conda")
    if conda_path: return conda_path
    for path in [os.path.expanduser("~/anaconda3/bin/conda"), os.path.expanduser("~/miniconda3/bin/conda"), "/opt/anaconda3/bin/conda"]:
        if os.path.exists(path) and os.access(path, os.X_OK): return path
    return None

def dynamic_dependency_resolver(conda_exe, hw_info):
    """
    Iteratively hunts for the highest compatible version of packages.
    Parses collision flags (Numpy C-API, Python distutils) and gracefully upgrades/downgrades.
    """
    print(f"\n{Colors.BOLD}--- Phase 3: Dynamic Dependency Resolution & Version Alignment ---{Colors.ENDC}")
    base_packages = {
        "numpy": "numpy", "scipy": "scipy", "ase": "ase", 
        "networkx": "networkx", "psutil": "psutil", "tblite": "tblite", 
        "pyscf": "pyscf", "pyscf-dispersion": "pyscf.dispersion", 
        "geometric": "geometric", "ipykernel": "ipykernel", "matplotlib": "matplotlib"
    }
    
    constraints = {}
    max_retries = 4
    
    # Pre-Flight: Python version flag check
    py_ver_res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", "import sys; print(sys.version_info.minor)"], capture_output=True, text=True)
    try:
        if int(py_ver_res.stdout.strip()) >= 12:
            print_status("Python >= 3.12 detected. Gathering error flags: 'distutils' deprecation limits C++ bindings.", "error")
            print_status("Gracefully downgrading environment to optimal Python 3.11...", "warning")
            subprocess.run([conda_exe, "install", "-n", "TOPOS", "python=3.11", "-y"], capture_output=True)
    except Exception: pass

    for attempt in range(1, max_retries + 1):
        print_status(f"Resolution Cycle {attempt}/{max_retries}: Aligning highest compatible versions...", "info")
        install_args = []
        for pkg in base_packages.keys():
            install_args.append(f"{pkg}{constraints.get(pkg, '')}")
                
        if hw_info["gpu_type"] == "NVIDIA":
            gpu_pkg = "gpu4pyscf-cuda12x" if "12" in hw_info["torch_wheel"] else "gpu4pyscf-cuda11x"
            install_args.append(f"{gpu_pkg}{constraints.get('gpu4pyscf', '')}")
        
        # Execute Bulk Pip Install
        pip_cmd = [conda_exe, "run", "-n", "TOPOS", "pip", "install", "--upgrade"] + install_args
        inst_res = subprocess.run(pip_cmd, capture_output=True, text=True)
        
        # Deep Integration Test
        import_str = "import " + ", ".join(set(base_packages.values()))
        if hw_info["gpu_type"] == "NVIDIA": import_str += "; import gpu4pyscf"
        
        test_res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", import_str], capture_output=True, text=True)
        
        if test_res.returncode == 0:
            print_status("Matrix alignment successful! Dependencies seamlessly upgraded/downgraded to optimal configurations.", "success")
            return constraints
            
        err_log = (test_res.stderr + "\n" + inst_res.stderr).lower()
        re_run = False
        
        # Parse Flag 1: Numpy 2.0 C-API Collision
        if "api version" in err_log or ("numpy" in err_log and ("c-api" in err_log or "runtimeerror" in err_log or "incompatible" in err_log)):
            print_status("Collision Flag: Numpy 2.x ABI breaks C++ tight-binding (tblite/PySCF).", "error")
            print_status("Downgrading Numpy to highest stable 1.x and locking tblite C++ dependencies...", "warning")
            constraints["numpy"] = "<=1.26.4"
            constraints["tblite"] = "<=0.3.0"
            re_run = True
            
        # Parse Flag 2: Missing Deprecated modules
        if "distutils" in err_log:
            print_status("Collision Flag: Missing distutils. Injecting setuptools constraint...", "warning")
            subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", "setuptools"], capture_output=True)
            re_run = True

        if not re_run:
            print_status("Unresolved module error or pip PyPI download failure. Falling back to individual local file evaluation...", "warning")
            logging.error(f"Unresolved Matrix Error:\n{test_res.stderr}\n{inst_res.stderr}")
            break

    print_status("Max automated loop retries reached. Triggering single-package strict enforcement...", "info")
    return constraints

def enforce_pip_dependency(conda_exe, pkg_base, import_name, constraint="", url="https://pypi.org/"):
    """Strictly enforces a pip module via import testing and local fallback."""
    pkg_full = f"{pkg_base}{constraint}"
    
    res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", f"import {import_name}"], capture_output=True, text=True)
    if res.returncode == 0:
        print_status(f"Dependency '{pkg_full}' is fully functional.", "success")
        return
        
    print_status(f"Automated download failed. Scanning for local wheels for {pkg_full}...", "warning")
    search_name = pkg_base.split('==')[0].split('[')[0].replace("-", "_")
    local_files = glob.glob(f"*{search_name}*.whl") + glob.glob(f"*{search_name}*.tar.gz")
    if not local_files:
        search_name = pkg_base.split('==')[0].split('[')[0]
        local_files = glob.glob(f"*{search_name}*.whl") + glob.glob(f"*{search_name}*.tar.gz")
        
    if local_files:
        print_status(f"Found local file {local_files[0]}. Installing...", "info")
        logging.info(f"Checksum for {local_files[0]}: {get_sha256(local_files[0])}")
        
        install_res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", local_files[0], "--force-reinstall"], capture_output=True, text=True)
        res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", f"import {import_name}"], capture_output=True, text=True)
        
        if res.returncode == 0:
            print_status(f"'{pkg_full}' installed successfully from local package.", "success")
            return
        else:
            logging.error(f"Local install failed. \nSTDOUT: {install_res.stdout}\nSTDERR: {install_res.stderr}\nIMPORT ERR: {res.stderr}")
            print(f"\n{Colors.WARNING}--- Local Install Error Logs ---{Colors.ENDC}\nSee Stage0_Environment.log for detailed stack traces.")
            
    hard_fail(pkg_full, "Pip install and local package evaluation failed.", url, f"Download exactly '{pkg_full}' and place the .whl or .tar.gz locally.")

def enforce_conda_dependency(conda_exe, pkg_name, check_cmd):
    res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "which", check_cmd], capture_output=True)
    if res.returncode == 0:
        print_status(f"Binary '{check_cmd}' ({pkg_name}) is present.", "success")
        return
        
    print_status(f"'{pkg_name}' missing. Attempting automated Conda install...", "warning")
    subprocess.run([conda_exe, "run", "-n", "TOPOS", "conda", "install", "-c", "conda-forge", "-y", pkg_name], capture_output=True)
    
    res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "which", check_cmd], capture_output=True)
    if res.returncode == 0:
        print_status(f"'{pkg_name}' successfully installed and verified.", "success")
        return
        
    hard_fail(pkg_name, "Conda failed to acquire the binary.", f"https://anaconda.org/conda-forge/{pkg_name}")

def enforce_pytorch(conda_exe, hw_info):
    print(f"\n{Colors.BOLD}--- Phase 2: Hardware & MLFF Libraries ---{Colors.ENDC}")
    res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", "import torch; import mace"], capture_output=True)
    if res.returncode == 0:
        print_status("PyTorch & MACE-OFF23 are fully functional.", "success")
        return
        
    print_status("PyTorch/MACE missing or broken. Attempting hardware-aware install...", "warning")
    torch_wheel = hw_info["torch_wheel"]
    gpu_type = hw_info["gpu_type"]
    
    # 1. Upgrade pip to ensure wheel building doesn't fail
    print_status("Upgrading pip to prepare for MLFFs...", "info")
    subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", "--upgrade", "pip"], capture_output=True)
    
    if gpu_type == "NVIDIA":
        subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", "torch", "torchvision", "torchaudio", "--index-url", f"https://download.pytorch.org/whl/{torch_wheel}"], capture_output=True)
    elif gpu_type == "AMD":
        subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/rocm6.0"], capture_output=True)
    else:
        subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cpu"], capture_output=True)
        
    print_status("Installing MACE via PyPI...", "info")
    mace_res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", "mace-torch", "cuequivariance", "cuequivariance-torch"], capture_output=True, text=True)

    res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", "import torch; import mace"], capture_output=True)
    
    # 2. PyPI -> Source Fallback
    if res.returncode != 0:
        print_status("PyPI MACE installation failed. Falling back to Source Installation...", "warning")
        mace_dir = os.path.abspath("mace_source")
        if not os.path.exists(mace_dir):
            subprocess.run(["git", "clone", "https://github.com/ACEsuit/mace.git", mace_dir], capture_output=True)
        subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", mace_dir], capture_output=True)
        subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", "cuequivariance", "cuequivariance-torch"], capture_output=True)

    res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", "import torch; import mace"], capture_output=True, text=True)
    if res.returncode == 0:
        print_status("PyTorch & MACE-OFF23 successfully installed and verified.", "success")
        return
        
    logging.error(f"PyTorch/MACE Final Install Error:\n{res.stderr}\n{mace_res.stderr}")
    hard_fail("PyTorch / MACE-OFF23", "Automated hardware-specific installation failed.", url="https://github.com/ACEsuit/mace", instruction="Check the Stage0_Environment.log for detailed traceback errors.")

def enforce_opi(conda_exe):
    print(f"\n{Colors.BOLD}--- Phase 3.5: ORCA Python Interface (OPI) ---{Colors.ENDC}")
    res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", "import opi"], capture_output=True)
    if res.returncode == 0:
        print_status("OPI >=2.0.0 is present and functional.", "success")
        return
        
    print_status("Scanning for local proprietary OPI package...", "info")
    local_files = glob.glob("opi-*.tar.gz") + glob.glob("opi-*.whl")
    
    if local_files:
        target_file = local_files[0]
        print_status(f"Found local file {target_file}.", "success")
        logging.info(f"OPI Checksum ({target_file}): {get_sha256(target_file)}")
        
        install_res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "install", target_file, "--force-reinstall"], capture_output=True, text=True)
        res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", "import opi"], capture_output=True)
        if res.returncode == 0:
            print_status("OPI strictly verified.", "success")
            return
        else:
            logging.error(f"OPI Local Install Error:\n{install_res.stderr}")
            hard_fail("OPI 2.0", "Local package failed to import.", url="https://faccts.de/download/")
    else:
        hard_fail("OPI >=2.0.0", "Proprietary package missing.", url="https://faccts.de/download/", instruction="Download 'opi-2.0.0.tar.gz' for Python 3.11 from the FACCTs portal.")

def enforce_oet(conda_exe, hw_info, constraints):
    print(f"\n{Colors.BOLD}--- Phase 4: ORCA External Tools (OET) ---{Colors.ENDC}")
    oet_dir = os.path.abspath("orca-external-tools")
    oet_venv = os.path.join(oet_dir, "OET_VENV")
    oet_script_dir = os.path.join(oet_dir, "bin")
    
    def find_oet_mace_bin():
        candidates = [
            os.path.join(oet_script_dir, "oet_mace"),
            os.path.join(oet_venv, "bin", "oet_mace"),
        ]
        for candidate in candidates:
            if os.path.exists(candidate) and os.access(candidate, os.X_OK):
                return candidate
        return None
    
    def test_oet():
        oet_mace_bin = find_oet_mace_bin()
        if oet_mace_bin:
            venv_python = os.path.join(oet_venv, "bin", "python")
            res = subprocess.run([venv_python, "-c", "import mace; import tblite"], capture_output=True, text=True)
            if res.returncode == 0: return True, ""
            return False, f"VENV Python Import Error:\n{res.stderr}"
        return False, (
            f"oet_mace binary not found in either {oet_script_dir} or {os.path.join(oet_venv, 'bin')}. "
            "The OET installation did not produce the expected executable."
        )
        
    success, err_msg = test_oet()
    if success:
        print_status("ORCA External Tools (MACE enabled): Present and Functional.", "success")
        return oet_venv
        
    print_status("OET missing or outdated (v2.0.0 lacks MACE). Enforcing strict main branch installation...", "warning")
    
    if os.path.exists(oet_dir): shutil.rmtree(oet_dir)
    
    print_status("Cloning latest OET from GitHub to guarantee MACE support...", "info")
    clone_res = subprocess.run(["git", "clone", "https://github.com/faccts/orca-external-tools.git", oet_dir], capture_output=True, text=True)
    if clone_res.returncode != 0:
        logging.error(f"Git clone failed:\n{clone_res.stderr}")
        hard_fail("ORCA External Tools", "Git clone failed. Ensure internet connection.", url="https://github.com/faccts/orca-external-tools")
    
    print_status("Running FACCTs Virtual Environment installer...", "info")
    oet_inst = subprocess.run(
        [conda_exe, "run", "-n", "TOPOS", "python", "install.py", "--venv-dir", oet_venv, "--script-dir", oet_script_dir], 
        cwd=oet_dir, capture_output=True, text=True
    )
    if oet_inst.returncode != 0:
        logging.error(f"OET install.py failed:\n{oet_inst.stderr}")
        print_status("OET install.py threw an error. See log.", "warning")
    
    venv_pip = os.path.join(oet_venv, "bin", "pip")
    if os.path.exists(venv_pip):
        print_status("Injecting hardware-aware MLFF dependencies into OET VENV...", "info")
        torch_wheel = hw_info["torch_wheel"]
        gpu_type = hw_info["gpu_type"]
        
        subprocess.run([venv_pip, "install", "--upgrade", "pip", "wheel", "setuptools"], capture_output=True)
        
        if gpu_type == "NVIDIA":
            t_res = subprocess.run([venv_pip, "install", "torch", "torchvision", "torchaudio", "--index-url", f"https://download.pytorch.org/whl/{torch_wheel}"], capture_output=True, text=True)
        elif gpu_type == "AMD":
            t_res = subprocess.run([venv_pip, "install", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/rocm6.0"], capture_output=True, text=True)
        else:
            t_res = subprocess.run([venv_pip, "install", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cpu"], capture_output=True, text=True)
        if t_res.returncode != 0: logging.error(f"Torch VENV install failed: {t_res.stderr}")

        print_status("Enforcing dynamic version alignment for VENV C++ bindings...", "info")
        mace_wheels = glob.glob("*mace*.whl")
        mace_pkg = mace_wheels[0] if mace_wheels else "mace-torch"
        
        venv_deps = ["cuequivariance", "cuequivariance-torch"]
        for p in ["tblite", "numpy", "scipy", "ase"]:
            venv_deps.append(f"{p}{constraints.get(p, '')}")

        m_res = subprocess.run([venv_pip, "install", mace_pkg] + venv_deps + ["--force-reinstall"], capture_output=True, text=True)
        
        if m_res.returncode != 0: 
            logging.error(f"MACE VENV PyPI install failed: {m_res.stderr}")
            print_status("VENV MACE PyPI installation failed. Falling back to source...", "warning")
            mace_dir = os.path.abspath("mace_source")
            if not os.path.exists(mace_dir):
                subprocess.run(["git", "clone", "https://github.com/ACEsuit/mace.git", mace_dir], capture_output=True)
            subprocess.run([venv_pip, "install", mace_dir], capture_output=True)
            subprocess.run([venv_pip, "install"] + venv_deps, capture_output=True)
        
    success, err_msg = test_oet()
    if success:
        print_status("ORCA External Tools compiled and strictly verified.", "success")
        return oet_venv
    else:
        logging.error(f"OET Import Test Error:\n{err_msg}")
        hard_fail("ORCA External Tools", f"VENV Python Import Error:\n{err_msg.strip()}", url="https://github.com/faccts/orca-external-tools")

def enforce_molsym(conda_exe):
    print(f"\n{Colors.BOLD}--- Phase 5: MolSym Library ---{Colors.ENDC}")
    molsym_dir = os.path.abspath("MolSym")
    molsym_zip = "master.zip"
    
    res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", "import sys; sys.path.append('MolSym'); import molsym"], capture_output=True)
    if res.returncode == 0:
        print_status("MolSym Library: Present and Functional.", "success")
        return molsym_dir

    print_status("MolSym missing. Enforcing installation...", "warning")
    if not os.path.exists(molsym_dir):
        if not os.path.exists(molsym_zip):
            print_status(f"'{molsym_zip}' not found. Downloading from GitHub...", "info")
            try: 
                urllib.request.urlretrieve("https://github.com/NASymmetry/MolSym/archive/refs/heads/master.zip", molsym_zip, reporthook=download_reporthook)
                print()
            except KeyboardInterrupt:
                if os.path.exists(molsym_zip): os.remove(molsym_zip)
                sys.exit(1)
            
        if os.path.exists(molsym_zip):
            logging.info(f"MolSym Zip Checksum: {get_sha256(molsym_zip)}")
            import zipfile
            with zipfile.ZipFile(molsym_zip, 'r') as zip_ref: zip_ref.extractall(".")
            for d in os.listdir("."):
                if d.startswith("MolSym-") and os.path.isdir(d):
                    os.rename(d, molsym_dir)
                    break
                    
    res = subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-c", "import sys; sys.path.append('MolSym'); import molsym"], capture_output=True)
    if res.returncode == 0:
        print_status("MolSym configured successfully.", "success")
        return molsym_dir
    else:
        hard_fail("MolSym", "Failed to extract or import MolSym.", url="https://github.com/NASymmetry/MolSym")

def check_fast_pass(hw_info):
    if not os.path.exists("opi_system_config.json"): return False
    try:
        with open("opi_system_config.json", 'r') as f: config = json.load(f)
        orca_cmd = config['paths']['orca_binary']
        subprocess.run([orca_cmd, "--version"], capture_output=True, check=True)
        import ase, networkx, scipy, tblite, mace, torch, opi
        print_status("Fast-Pass Matrix Active: Environment verified natively. Bypassing Subprocess checks.", "success")
        return config
    except Exception as e:
        logging.info(f"Fast-Pass failed: {e}")
        return False

def setup_environment(hw_info):
    conda_exe = find_conda()
    if not conda_exe:
        hard_fail("Anaconda/Miniconda", "Conda path not found.")

    envs = subprocess.check_output([conda_exe, "env", "list"], text=True)
    if "TOPOS" not in envs:
        print_status("Constructing Python 3.11 environment...")
        subprocess.run([conda_exe, "create", "-n", "TOPOS", "python=3.11", "-y"], check=True)

    print(f"\n{Colors.BOLD}--- Phase 1: Core Conda/Pip Dependencies ---{Colors.ENDC}")
    enforce_conda_dependency(conda_exe, "openmpi", "mpirun")
    enforce_conda_dependency(conda_exe, "git", "git")
    
    mpi_check = subprocess.run([conda_exe, "run", "-n", "TOPOS", "which", "mpirun"], capture_output=True, text=True)
    mpi_path = mpi_check.stdout.strip()
    
    mpi_ver = subprocess.run([mpi_path, "--version"], capture_output=True, text=True)
    if "4.1." in mpi_ver.stdout or "4.1." in mpi_ver.stderr:
        print_status(f"OpenMPI 4.1.x Strictly Verified (ORCA 6.1.1 Optimal) at: {mpi_path}", "success")
    else:
        print_status("OpenMPI version is NOT 4.1.x! Automating downgrade/upgrade to 4.1.6...", "warning")
        logging.warning(f"OpenMPI Mismatch: {mpi_ver.stdout.strip()[:100]}. Auto-correcting...")
        subprocess.run([conda_exe, "run", "-n", "TOPOS", "conda", "install", "-c", "conda-forge", "openmpi=4.1.6", "-y"], capture_output=True)
        mpi_check = subprocess.run([conda_exe, "run", "-n", "TOPOS", "which", "mpirun"], capture_output=True, text=True)
        mpi_path = mpi_check.stdout.strip()
        print_status(f"OpenMPI automatically corrected to 4.1.6 at: {mpi_path}", "success")

    enforce_pytorch(conda_exe, hw_info)

    constraints = dynamic_dependency_resolver(conda_exe, hw_info)

    base_packages = {
        "numpy": "numpy", "scipy": "scipy", "ase": "ase", 
        "networkx": "networkx", "psutil": "psutil", "tblite": "tblite", 
        "pyscf": "pyscf", "pyscf-dispersion": "pyscf.dispersion", 
        "geometric": "geometric", "ipykernel": "ipykernel", "matplotlib": "matplotlib"
    }

    pip_reqs = [
        ("numpy==1.26.4", "numpy", "https://pypi.org/project/numpy/"),
        ("scipy==1.13.1", "scipy", "https://pypi.org/project/scipy/"),
        ("ase==3.23.0", "ase", "https://pypi.org/project/ase/"),
        ("networkx==3.3", "networkx", "https://pypi.org/project/networkx/"),
        ("psutil==5.9.8", "psutil", "https://pypi.org/project/psutil/"),
        ("tblite==0.3.0", "tblite", "https://pypi.org/project/tblite/"),
        ("pyscf==2.6.2", "pyscf", "https://pypi.org/project/pyscf/"),
        ("pyscf-dispersion==0.5.0", "pyscf.dispersion", "https://pypi.org/project/pyscf-dispersion/"),
        ("geometric==1.0.2", "geometric", "https://pypi.org/project/geometric/"), 
        ("ipykernel", "ipykernel", "https://pypi.org/project/ipykernel/"),
        ("matplotlib==3.9.0", "matplotlib", "https://pypi.org/project/matplotlib/"),
    ]
    
    for pkg, imp, url in pip_reqs:
        parts = pkg.split("==", 1)
        base_pkg_name = parts[0]
        dyn_constraint = constraints.get(base_pkg_name, "")
        if dyn_constraint:
            enforce_pip_dependency(conda_exe, base_pkg_name, imp, dyn_constraint, url)
        else:
            pinned_constraint = f"=={parts[1]}" if len(parts) == 2 and parts[1] else ""
            enforce_pip_dependency(conda_exe, base_pkg_name, imp, pinned_constraint, url)
        
    if hw_info["gpu_type"] == "NVIDIA":
        gpu_pkg = "gpu4pyscf-cuda12x" if "12" in hw_info["torch_wheel"] else "gpu4pyscf-cuda11x"
        enforce_pip_dependency(conda_exe, gpu_pkg, "gpu4pyscf", constraints.get("gpu4pyscf", ""), "https://pypi.org/project/gpu4pyscf/")

    enforce_opi(conda_exe)

    print_status("Registering Kernel to Jupyter...", "info")
    subprocess.run([conda_exe, "run", "-n", "TOPOS", "python", "-m", "ipykernel", "install", "--user", "--name", "TOPOS", "--display-name", "Python (TOPOS)"], capture_output=True)

    oet_venv = enforce_oet(conda_exe, hw_info, constraints)
    molsym_path = enforce_molsym(conda_exe)
    
    print_status("Generating exact requirements_lock.txt for SI reproducibility...", "info")
    with open("requirements_lock.txt", "w") as f:
        subprocess.run([conda_exe, "run", "-n", "TOPOS", "pip", "freeze"], stdout=f, text=True)

    return mpi_path, oet_venv, molsym_path

def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}   TOPOS SPECTROSCOPIC PIPELINE - STAGE 0  {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    logging.info("--- NEW EXECUTION LAUNCHED ---")
    
    hw_info = get_hardware_info()
    fast_config = check_fast_pass(hw_info)
    
    if fast_config:
        orca_path = fast_config['paths']['orca_binary']
        orca_version = fast_config.get('passport', {}).get('ORCA_Version', 'Unknown')
        mpi_path = fast_config['paths']['openmpi_binary']
        oet_venv = fast_config['paths']['oet_venv']
        molsym_path = fast_config['paths']['molsym_path']
    else:
        orca_path, orca_version = test_orca_execution()
        mpi_path, oet_venv, molsym_path = setup_environment(hw_info)
    
    passport = {
        "OS_Platform": platform.platform(),
        "OS_Release": platform.release(),
        "Python_Version": sys.version.split()[0],
        "ORCA_Version": orca_version,
        "CUDA_Target": hw_info["torch_wheel"]
    }
    
    config = {
        "hardware": hw_info,
        "passport": passport,
        "paths": {
            "orca_binary": orca_path,
            "openmpi_binary": mpi_path,
            "oet_venv": oet_venv,
            "molsym_path": molsym_path
        }
    }
    
    with open("opi_system_config.json", "w") as f:
        json.dump(config, f, indent=4)
    
    print(f"\n{Colors.BOLD}--- Stage 0 Execution Summary ---{Colors.ENDC}")
    print_status(f"Digital Passport Secured: ORCA {orca_version} on {passport['OS_Platform']}", "success")
    print_status("Pipeline State JSON generated (opi_system_config.json)", "success")
    
    current_env = os.environ.get("CONDA_DEFAULT_ENV", "")
    if current_env != "TOPOS" and "TOPOS" not in sys.executable:
        print(f"\n{Colors.WARNING}{Colors.BOLD}⚠️  KERNEL RESTART REQUIRED ⚠️{Colors.ENDC}")
        print("Please refresh your browser, click 'Kernel' -> 'Change Kernel', and select 'Python (TOPOS)'.")
    else:
        print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 0 COMPLETE: You are actively inside TOPOS. Proceed to Stage 1.{Colors.ENDC}")

if __name__ == "__main__":
    main()



   TOPOS SPECTROSCOPIC PIPELINE - STAGE 0  

--- Hardware Acceleration Auto-Detect ---
✅ OS Memory Allocation (Stack Size) is mathematically healthy.
✅ CPU Detected: 24 threads available
✅ Memory Detected: 62.55 GB available
✅ CUDA Host Version: 13.2 (Routing to PyTorch wheel cu124)
✅ GPU Detected: NVIDIA Architecture

--- Deep ORCA Engine Pre-Flight Check ---
✅ ORCA 6.1.1 validated (Basis sets & Property modules OK) at: /home/joshua/orca_6_1_1_avx2/orca

--- Phase 1: Core Conda/Pip Dependencies ---
✅ Binary 'mpirun' (openmpi) is present.
✅ Binary 'git' (git) is present.
✅ OpenMPI 4.1.x Strictly Verified (ORCA 6.1.1 Optimal) at: /home/joshua/anaconda3/envs/TOPOS/bin/mpirun

--- Phase 2: Hardware & MLFF Libraries ---
✅ PyTorch & MACE-OFF23 are fully functional.

--- Phase 3: Dynamic Dependency Resolution & Version Alignment ---
➡️ Resolution Cycle 1/4: Aligning highest compatible versions...
✅ Matrix alignment successful! Dependencies seamlessly upgraded/downgraded to optimal configura

/usr/bin/bash: /home/joshua/anaconda3/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)


📥 Stage 1.1: Ingestion & Config

Purpose: Prompts for system variables, scans xyz files, analyzes topology, and mathematically routes downstream DFT to GPU or CPU.

In [8]:
#!/usr/bin/env python3


# ==============================================================================
# 01-INGEST-GC (Stage 1.1)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage acts as the pipeline's "Intelligent Entry Point." It prompts the user 
# for core molecular variables (Charge, Multiplicity), scans the ingestion folder 
# for starting geometries, and performs a deep topological analysis. It verifies 
# MACE-OFF23 neural network compatibility and calculates the theoretical VRAM 
# footprint to mathematically route downstream DFT calculations to either GPU (PySCF) 
# or CPU (ORCA) to prevent Out-Of-Memory crashes.
# 
# 2. Use instructions:
# Execute this script. It will interactively ask for project variables. Place 
# your starting `.xyz` files into the generated `Isomer_xyz_Initial_Files` folder, 
# then re-run the script to finalize ingestion.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: The script maps the files, verifies MACE compatibility, calculates 
# basis function VRAM footprint, and saves all flags to `opi_user_config.json`.
# ==============================================================================

import os
import sys
import glob
import json
import shutil
import warnings
import psutil
import numpy as np

# Attempt to load GPU frameworks for dynamic VRAM checks
try:
    import cupy as cp
except ImportError:
    cp = None

warnings.filterwarnings("ignore")

try:
    from ase.io import read
    from ase.neighborlist import natural_cutoffs, NeighborList
    import networkx as nx
except ImportError:
    print("❌ ERROR: Required libraries (ASE, NetworkX) not found. Please run Stage 0.")
    sys.exit(1)

class Colors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'

CONFIG_FILE = "opi_user_config.json"
INGEST_DIR = "Isomer_xyz_Initial_Files"

def print_status(msg, status="info"):
    if status == "success": print(f"{Colors.OKGREEN}✅ {msg}{Colors.ENDC}")
    elif status == "error": print(f"{Colors.FAIL}❌ {msg}{Colors.ENDC}")
    elif status == "warning": print(f"{Colors.WARNING}⚠️ {msg}{Colors.ENDC}")
    else: print(f"{Colors.OKCYAN}➡️ {msg}{Colors.ENDC}")

# ==============================================================================
# HARDWARE ROUTING & ESTIMATION PROTOCOLS
# ==============================================================================

def estimate_basis_functions(atoms, basis_family="def2-TZVP"):
    """
    Empirically estimates the number of basis functions for def2-TZVP to 
    inform the VRAM allocation router.
    """
    bf_count = 0
    for atom in atoms:
        sym = atom.symbol
        if sym in ['H', 'He']: bf_count += 5
        elif sym in ['Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne']: bf_count += 14
        elif sym in ['Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar']: bf_count += 18
        elif sym in ['K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Br', 'Kr']: bf_count += 32
        else: bf_count += 45 
    return bf_count

def evaluate_quantum_hardware_routing(num_atoms, num_basis_functions, calculation_type="frequency"):
    """
    Evaluates whether to use GPU-PySCF or CPU-ORCA based on hardware limits.
    Assumes Target: RTX 3090 (24GB VRAM) vs Host CPU (48GB System RAM, 12 Cores).
    """
    vram_total_gb = 24.0
    if cp is not None:
        try:
            vram_total_gb = cp.cuda.Device(0).attributes['TotalGlobalMem'] / (1024**3)
        except Exception: pass
        
    system_ram_gb = psutil.virtual_memory().total / (1024**3)
    cpu_cores = psutil.cpu_count(logical=False) or os.cpu_count()

    if calculation_type.lower() == "frequency":
        estimated_vram_needed = (num_basis_functions ** 2.3) * 8 / (1024**3) * 1.5
    else:
        estimated_vram_needed = (num_basis_functions ** 2) * 8 / (1024**3) * 1.2

    print(f"\n{Colors.OKBLUE}--- Hardware Tensor Routing Matrix ---{Colors.ENDC}")
    print(f" Atoms: {num_atoms} | Estimated Basis Functions (def2-TZVP): {num_basis_functions}")
    print(f" Estimated VRAM Needed: {Colors.WARNING}{estimated_vram_needed:.2f} GB{Colors.ENDC} ({calculation_type})")
    print(f" Available VRAM: {vram_total_gb:.2f} GB | System RAM: {system_ram_gb:.2f} GB | CPU Cores: {cpu_cores}")

    if estimated_vram_needed >= (vram_total_gb * 0.85):
        print_status("ROUTE TO ORCA (CPU): PySCF will likely hit a GPU Out-Of-Memory (OOM) crash.", "error")
        if num_atoms > 150:
            print_status("Recommendation: Run ORCA with 'r²SCAN-3c' to save CPU cycles on this large system.", "info")
        else:
            print_status("Recommendation: Run ORCA with 'ωB97X-3c' if electronic accuracy is critical.", "info")
        return "ORCA_CPU"
    else:
        print_status("ROUTE TO PySCF (GPU): System fits comfortably inside VRAM. Brute force speed is optimal.", "success")
        return "PySCF_GPU"

# ==============================================================================
# INGESTION & TOPOLOGY PROTOCOLS
# ==============================================================================

def load_or_query_config():
    """Interactively provisions the foundational project configuration."""
    if os.path.exists(CONFIG_FILE):
        with open(CONFIG_FILE, 'r') as f: config = json.load(f)
    else:
        config = {}

    if "molecule_name" not in config or not config["molecule_name"]:
        print(f"\n{Colors.OKCYAN}Enter the base name for this molecule/project (e.g., SO2-H2):{Colors.ENDC}")
        config["molecule_name"] = input(" > ").strip()
    else:
        print_status(f"Project Name automatically set to: {config['molecule_name']}")

    if "charge" not in config:
        print(f"\n{Colors.OKCYAN}Enter the total formal charge of the system [Default: 0]:{Colors.ENDC}")
        c_input = input(" > ").strip()
        config["charge"] = int(c_input) if c_input else 0
    else:
        print_status(f"System Charge automatically set to: {config['charge']}")

    if "multiplicity" not in config:
        print(f"\n{Colors.OKCYAN}Enter the spin multiplicity (1=Singlet, 2=Doublet, etc.) [Default: 1]:{Colors.ENDC}")
        m_input = input(" > ").strip()
        config["multiplicity"] = int(m_input) if m_input else 1
    else:
        print_status(f"Spin Multiplicity automatically set to: {config['multiplicity']}")

    if "temperature" not in config: config["temperature"] = 298.15

    with open(CONFIG_FILE, 'w') as f: json.dump(config, f, indent=4)
    return config

def check_ingestion_folder():
    """Validates the presence of initial coordinate data."""
    if not os.path.exists(INGEST_DIR):
        os.makedirs(INGEST_DIR)
        print_status(f"Created '{INGEST_DIR}' folder.", "info")
        
    xyz_files = glob.glob(os.path.join(INGEST_DIR, "*.xyz"))
    if not xyz_files:
        print(f"\n{Colors.FAIL}{Colors.BOLD}🚨 NO XYZ FILES DETECTED 🚨{Colors.ENDC}")
        print("ACTION REQUIRED:")
        print(f"Please place your starting .xyz structural files into the '{INGEST_DIR}' directory.")
        print("Once the files are in place, re-run this stage.\n")
        sys.exit(0)
        
    print_status(f"Detected {len(xyz_files)} initial isomer file(s) for processing.", "success")
    return xyz_files

def analyze_topology(xyz_file, config):
    """Uses ASE and NetworkX to categorize fragments, MACE constraints, and Hardware routing."""
    try:
        mol = read(xyz_file)
        num_atoms = len(mol)
        
        size_class = "Small" if num_atoms < 15 else "Medium" if num_atoms <= 50 else "Large"
        
        cutoffs = [c * 1.2 for c in natural_cutoffs(mol)]
        nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
        nl.update(mol)
        matrix = nl.get_connectivity_matrix()
        graph = nx.from_scipy_sparse_array(matrix)
        fragments = list(nx.connected_components(graph))
        num_frags = len(fragments)
        
        min_dist = 0.0
        if num_frags == 1:
            complex_type = "Single Covalently Bound Molecule"
        else:
            min_dist = float('inf')
            positions = mol.get_positions()
            for i in range(num_frags):
                for j in range(i + 1, num_frags):
                    for atom_i in fragments[i]:
                        for atom_j in fragments[j]:
                            d = np.linalg.norm(positions[atom_i] - positions[atom_j])
                            if d < min_dist: min_dist = d
            
            if min_dist < 2.5: complex_type = f"Strong Interaction Complex ({num_frags} Fragments, min dist {min_dist:.2f} Å)"
            else: complex_type = f"Weak Interaction Complex ({num_frags} Fragments, min dist {min_dist:.2f} Å)"
                
        mace_applicable = True
        mace_reason = "Fully Compatible"
        
        valid_mace_elements = {'H', 'C', 'N', 'O', 'P', 'S', 'F', 'Cl', 'Br', 'I'}
        symbols = set(mol.get_chemical_symbols())
        invalid_elements = symbols - valid_mace_elements
        
        if invalid_elements:
            mace_applicable = False
            mace_reason = f"Unsupported elements ({', '.join(invalid_elements)})"
        elif config.get("charge", 0) != 0 or config.get("multiplicity", 1) != 1:
            mace_applicable = False
            mace_reason = f"Charged/Open-Shell System (Charge={config.get('charge', 0)}, Mult={config.get('multiplicity', 1)})"
        elif num_frags > 1 and min_dist > 12.0:
            mace_applicable = False
            mace_reason = f"Fragments too distant ({min_dist:.2f} Å) for message passing"

        est_basis = estimate_basis_functions(mol)
        routing_flag = evaluate_quantum_hardware_routing(num_atoms, est_basis, calculation_type="frequency")

        return {
            "file": os.path.basename(xyz_file),
            "atoms": num_atoms,
            "size": size_class,
            "fragments_count": num_frags,
            "type": complex_type,
            "mace_applicable": mace_applicable,
            "mace_reason": mace_reason,
            "pyscf_v_orca": routing_flag
        }
        
    except Exception as e:
        print_status(f"Failed to analyze topology for {xyz_file}: {e}", "error")
        return None

def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}   ChemComp_OPI-ORCA_v3-1 - STAGE 1.1: INGESTION & CONFIG   {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    config = load_or_query_config()
    xyz_files = check_ingestion_folder()
    
    print(f"\n{Colors.BOLD}--- Topological Analysis ---{Colors.ENDC}")
    topology_data = []
    
    for f in xyz_files:
        print(f"\n{Colors.BOLD}Scanning: {os.path.basename(f)}{Colors.ENDC}")
        data = analyze_topology(f, config)
        if data:
            topology_data.append(data)
            print_status(f"Topology Summary for {data['file']}:", "info")
            print(f"    ↳ Size: {data['atoms']} atoms [{data['size']}]")
            print(f"    ↳ Interaction: {data['type']}")
            if data['mace_applicable']:
                print(f"    ↳ MACE Constraints: {Colors.OKGREEN}Applicable ({data['mace_reason']}){Colors.ENDC}")
            else:
                print(f"    ↳ MACE Constraints: {Colors.WARNING}Not Applicable - Bypassing to xTB ({data['mace_reason']}){Colors.ENDC}")
            
            route_color = Colors.OKGREEN if data["pyscf_v_orca"] == "PySCF_GPU" else Colors.WARNING
            print(f"    ↳ Final Tensor Route: {route_color}{data['pyscf_v_orca']}{Colors.ENDC}")

    config["ingested_files"] = topology_data
    with open(CONFIG_FILE, 'w') as f:
        json.dump(config, f, indent=4)
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 1.1 COMPLETE: Topology, Dependencies, and Routings logged. Proceed to Stage 1.2.{Colors.ENDC}")

if __name__ == "__main__":
    main()



   ChemComp_OPI-ORCA_v3-1 - STAGE 1.1: INGESTION & CONFIG   
➡️ Project Name automatically set to: CyBuCl
➡️ System Charge automatically set to: 0
➡️ Spin Multiplicity automatically set to: 1
✅ Detected 2 initial isomer file(s) for processing.

--- Topological Analysis ---

Scanning: CycBuCl_Ax.xyz

--- Hardware Tensor Routing Matrix ---
 Atoms: 12 | Estimated Basis Functions (def2-TZVP): 109
 Estimated VRAM Needed: 0.00 GB (frequency)
 Available VRAM: 24.00 GB | System RAM: 62.55 GB | CPU Cores: 16
✅ ROUTE TO PySCF (GPU): System fits comfortably inside VRAM. Brute force speed is optimal.
➡️ Topology Summary for CycBuCl_Ax.xyz:
    ↳ Size: 12 atoms [Small]
    ↳ Interaction: Single Covalently Bound Molecule
    ↳ MACE Constraints: Applicable (Fully Compatible)
    ↳ Final Tensor Route: PySCF_GPU

Scanning: CycBuCl_Eq.xyz

--- Hardware Tensor Routing Matrix ---
 Atoms: 12 | Estimated Basis Functions (def2-TZVP): 109
 Estimated VRAM Needed: 0.00 GB (frequency)
 Available VRAM: 24.00 GB 

🧠 Stage 1.2: Script Engine Core

Purpose: Mounts universal math protocols into memory (OpenMPI subprocess wrappers, MACE-OFF23 loaders, Kabsch alignment, and Escape Room logic).

In [9]:
#!/usr/bin/env python3

# ==============================================================================
# 01-SCRIPT-ENG (Stage 1.2)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage acts as the central "Script Engine" for the entire OPI-ORCA pipeline. 
# It aggregates universally required functions, including the OpenMPI subprocess 
# wrapper, MACE checker, Topographic Crusher, Escape Room, Memory Estimators, 
# and the Macroscopic Inventory Export Protocol.
# 
# 2. Use instructions:
# Execute this script directly in your active Jupyter environment. It will load 
# all functions into active memory (globals) and run a diagnostic self-test.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: The Python kernel will register all protocols. It will run lightning-fast 
# unit tests verifying MACE logic, Kabsch algorithms, Enantiomer detection, and 
# Deduplication.
# FAILURE: Failed tests halt execution, indicating environment math corruption.
# ==============================================================================

import os
import sys
import csv
import json
import glob
import shutil
import subprocess
import urllib.request
import numpy as np
import warnings
warnings.filterwarnings("ignore")

try:
    from ase import Atoms
    from ase.io import read, write
    from ase.neighborlist import natural_cutoffs, NeighborList
    import networkx as nx
except ImportError:
    print("❌ ERROR: Required libraries (ASE, NetworkX) not found. Please run Stage 0.")
    sys.exit(1)

# ==============================================================================
# I. TERMINAL & STATUS FORMATTING
# ==============================================================================

class Colors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'

def print_status(msg, status="info"):
    if status == "success": print(f"{Colors.OKGREEN}✅ {msg}{Colors.ENDC}")
    elif status == "error": print(f"{Colors.FAIL}❌ {msg}{Colors.ENDC}")
    elif status == "warning": print(f"{Colors.WARNING}⚠️ {msg}{Colors.ENDC}")
    else: print(f"{Colors.OKCYAN}➡️ {msg}{Colors.ENDC}")

# ==============================================================================
# II. STATE & CONFIGURATION MANAGEMENT
# ==============================================================================

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
WEBHOOK_FILE = "discord_webhook.txt"

def load_system_config():
    if not os.path.exists(SYSTEM_CONFIG):
        print_status(f"System config missing. Please run Stage 0.", "error")
        return None
    with open(SYSTEM_CONFIG, 'r') as f: return json.load(f)

def load_user_config():
    if not os.path.exists(USER_CONFIG):
        print_status(f"User config missing. Please run Stage 1.1.", "error")
        return None
    with open(USER_CONFIG, 'r') as f: return json.load(f)

def send_ping(message):
    webhook_url = None
    if os.path.exists(WEBHOOK_FILE):
        with open(WEBHOOK_FILE, "r") as f: webhook_url = f.read().strip()
    if not webhook_url or webhook_url.lower() == "none": return
    try:
        data = {"content": message}
        req = urllib.request.Request(webhook_url, method="POST", data=json.dumps(data).encode('utf-8'))
        req.add_header('Content-Type', 'application/json')
        req.add_header('User-Agent', 'ChemComp_OPI-ORCA_v3-1/1.0') 
        urllib.request.urlopen(req, timeout=5)
    except: pass

# ==============================================================================
# III. ENGINE PROTOCOLS: ORCA SUBPROCESS & RESOURCE ESTIMATION
# ==============================================================================

def estimate_orca_memory(atoms, method="DLPNO-CCSD(T)", basis_set="def2-TZVP"):
    """
    Evaluates topological basis functions and method scaling architecture to output 
    the minimum required %maxcore per CPU thread to prevent OOM/Disk thrashing crashes.
    """
    bf_count = 0
    basis_lower = basis_set.lower()
    
    # Heuristic BF counting for main required bases
    for atom in atoms:
        sym = atom.symbol
        if "aug" in basis_lower: 
            if sym in ['H', 'He']: bf_count += 9
            elif sym in ['Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne']: bf_count += 23
            elif sym in ['Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar']: bf_count += 32
            else: bf_count += 50
        else: 
            if sym in ['H', 'He']: bf_count += 5
            elif sym in ['Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne']: bf_count += 14
            elif sym in ['Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar']: bf_count += 18
            else: bf_count += 35

    method_upper = method.upper()
    if "DLPNO-CCSD(T)" in method_upper:
        mem_per_core = 3000 + (bf_count * 2.0)
    elif "CCSD(T)" in method_upper:
        mem_per_core = 4000 + (bf_count * 4.5)
    elif "REVDSD" in method_upper or "DH" in method_upper:
        mem_per_core = 2500 + (bf_count * 2.5)
    else:
        mem_per_core = 2000 + (bf_count * 0.8)

    mem_per_core = max(2000, min(int(mem_per_core), 128000))
    return mem_per_core

def estimate_job_time(atoms, method, basis_set, iterations, sys_config):
    """Estimates the absolute time needed to complete a computational chemistry job."""
    bf_count = 0
    basis_lower = basis_set.lower()
    for atom in atoms:
        sym = atom.symbol
        if "aug" in basis_lower:
            if sym in ['H', 'He']: bf_count += 9
            elif sym in ['Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne']: bf_count += 23
            elif sym in ['Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar']: bf_count += 32
            else: bf_count += 50
        else:
            if sym in ['H', 'He']: bf_count += 5
            elif sym in ['Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne']: bf_count += 14
            elif sym in ['Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar']: bf_count += 18
            else: bf_count += 35

    cpu_cores = sys_config.get("hardware", {}).get("cpu_threads", 4)
    N = len(atoms)

    method_upper = method.upper()
    if "XTB" in method_upper:
        time_per_iter = (N * 0.05) / max(1, cpu_cores // 2)
    elif "MACE" in method_upper:
        time_per_iter = (N * 0.02)
    elif "R2SCAN" in method_upper or "SCAN" in method_upper:
        time_per_iter = (bf_count ** 2.2) / (50000 * cpu_cores)
    elif "CCSD(T)" in method_upper or "DLPNO" in method_upper:
        time_per_iter = (bf_count ** 4.5) / (1e8 * cpu_cores)
    else:
        time_per_iter = (bf_count ** 2.0) / (20000 * cpu_cores)

    total_seconds = max(1.0, time_per_iter * iterations)
    
    if total_seconds < 60:
        return f"{int(total_seconds)}s"
    elif total_seconds < 3600:
        return f"{int(total_seconds // 60)}m"
    else:
        return f"{total_seconds / 3600:.1f}h"


def run_orca_subprocess_safely(job_name, input_file, work_dir, sys_config, exit_on_fail=True):
    """Executes ORCA natively while injecting OpenMPI paths and suppressing TCP network bugs."""
    orca_bin = sys_config.get("paths", {}).get("orca_binary", "orca")
    mpi_path = sys_config.get("paths", {}).get("openmpi_binary")
    
    print_status(f"Igniting ORCA Subprocess: {job_name} ...", "info")
    custom_env = os.environ.copy()
    
    if mpi_path and os.path.exists(mpi_path):
        mpi_bin_dir = os.path.dirname(mpi_path)
        mpi_lib_dir = os.path.abspath(os.path.join(mpi_bin_dir, '..', 'lib'))
        custom_env["PATH"] = f"{mpi_bin_dir}:{custom_env.get('PATH', '')}"
        custom_env["LD_LIBRARY_PATH"] = f"{mpi_lib_dir}:{custom_env.get('LD_LIBRARY_PATH', '')}"

    custom_env["OMPI_MCA_btl"] = "^tcp"

    try:
        result = subprocess.run([orca_bin, input_file], cwd=work_dir, env=custom_env, capture_output=True, text=True)
    except Exception as e:
        print_status(f"Subprocess failed to launch: {e}", "error")
        if exit_on_fail: sys.exit(1)
        return None

    out_file = input_file.replace('.inp', '.out')
    out_path = os.path.join(work_dir, out_file)
    with open(out_path, 'w') as f:
        f.write(result.stdout)
        f.write("\n--- STDERR ---\n")
        f.write(result.stderr)

    if "ORCA TERMINATED NORMALLY" not in result.stdout:
        print_status(f"ORCA crashed during {job_name}.", "error")
        if not exit_on_fail: return None
        print(f"\n{Colors.WARNING}--- Last 20 lines of output ---{Colors.ENDC}")
        lines = result.stdout.splitlines()[-20:]
        for line in lines: print(line)
        send_ping(f"🚨 **ChemComp_OPI-ORCA_v3-1 Alert:** Job `{job_name}` crashed abnormally!")
        sys.exit(1)

    print_status(f"Execution complete: {job_name}", "success")
    return out_path

# ==============================================================================
# IV. ENGINE PROTOCOLS: DATA PARSING & EVALUATION
# ==============================================================================

def parse_orca_energy(out_file):
    energy = 0.0
    with open(out_file, 'r') as f:
        for line in f:
            if "FINAL SINGLE POINT ENERGY" in line:
                energy = float(line.split()[4])
    return energy

def parse_first_vibrational_freq(out_file):
    freqs = []
    capture = False
    with open(out_file, 'r') as f:
        for line in f:
            if "VIBRATIONAL FREQUENCIES" in line:
                capture = True; continue
            if capture and "cm**-1" in line:
                try: freqs.append(float(line.split()[1]))
                except: pass
            if capture and "NORMAL MODES" in line: break
    real_freqs = [f for f in freqs if f > 5.0]
    return real_freqs[0] if real_freqs else 0.0

def calculate_kabsch_rmsd(P, Q):
    """Calculates minimal RMSD alignment. Requires identical stereocenters to align."""
    Pc = P - np.mean(P, axis=0)
    Qc = Q - np.mean(Q, axis=0)
    C = np.dot(Pc.T, Qc)
    U, S, Vt = np.linalg.svd(C)
    
    if (np.linalg.det(U) * np.linalg.det(Vt)) < 0.0:
        S[-1] = -S[-1]
        U[:, -1] = -U[:, -1]
        
    R = np.dot(U, Vt)
    P_rot = np.dot(Pc, R)
    return np.sqrt(np.mean(np.sum((P_rot - Qc)**2, axis=1)))

def is_mace_applicable(atoms):
    valid_elements = {'H', 'C', 'N', 'O', 'P', 'S', 'F', 'Cl', 'Br', 'I'}
    symbols = set(atoms.get_chemical_symbols())
    return symbols.issubset(valid_elements)

def find_oet_binary(binary_name, sys_config):
    """Robustly searches for ORCA External Tools binaries across typical venv paths."""
    path_which = shutil.which(binary_name)
    if path_which: return path_which
    
    oet_venv = sys_config.get("paths", {}).get("oet_venv", "")
    if not oet_venv: return binary_name
    
    search_dirs = [
        os.path.join(oet_venv, "bin"),
        os.path.dirname(oet_venv),
        os.path.join(os.path.dirname(oet_venv), "bin")
    ]
    
    for d in search_dirs:
        test_path = os.path.join(d, binary_name)
        if os.path.exists(test_path):
            return test_path
            
    return binary_name

def get_goat_header(sys_config, atoms, iterations=300):
    """
    Provides an interactive method selector for the GOAT loops incorporating 
    estimated completion time and method evaluation profiles.
    """
    print(f"\n{Colors.OKCYAN}--- GOAT Exploration Method Selector ---{Colors.ENDC}")
    mace_app = is_mace_applicable(atoms)
    
    oet_mace_path = find_oet_binary("oet_mace", sys_config)
    oet_gxtb_path = find_oet_binary("oet_gxtb", sys_config)

    t_xtb = estimate_job_time(atoms, "xTB2", "", iterations, sys_config)
    print(f" [1] External g-xTB (ExtOpt) | Est. Time: ~{t_xtb: <6} | Benefit: Robust semi-empirical baseline.")

    if mace_app:
        t_mace = estimate_job_time(atoms, "MACE", "", iterations, sys_config)
        print(f" [2] MACE-OFF23 (ExtOpt)     | Est. Time: ~{t_mace: <6} | Benefit: Near-DFT accuracy.")
    else:
        print(f" [2] MACE-OFF23 (ExtOpt)     | {Colors.FAIL}UNAVAILABLE{Colors.ENDC} (System failed MACE-OFF23 applicability)")

    while True:
        choice = input(f"Select GOAT Method [1/2] for {iterations} iterations: ").strip()
        
        if choice == '1':
            print_status("Selected External g-xTB via ExtOpt for GOAT Exploration.", "success")
            return f"! ExtOpt GOAT\n%method\n  extcmd \"{oet_gxtb_path}\"\nend\n"
            
        elif choice == '2' and mace_app:
            mace_exists = shutil.which(oet_mace_path) is not None or os.path.exists(oet_mace_path)
            if mace_exists:
                print_status("Selected MACE-OFF23 via ExtOpt for GOAT.", "success")
                return f"! ExtOpt GOAT\n%method\n  extcmd \"{oet_mace_path}\"\nend\n"
            else:
                print_status(f"MACE executable missing at '{oet_mace_path}'. Falling back to External g-xTB.", "warning")
                return f"! ExtOpt GOAT\n%method\n  extcmd \"{oet_gxtb_path}\"\nend\n"
                
        else:
            print_status("Invalid selection. Please choose an available option.", "warning")

def extract_goat_ensemble(work_dir, job_prefix):
    ensemble_candidates = glob.glob(os.path.join(work_dir, f"{job_prefix}*_ensemble.xyz")) + glob.glob(os.path.join(work_dir, f"{job_prefix}*all.xyz"))
    extracted = []
    if ensemble_candidates:
        goat_ensemble = read(ensemble_candidates[0], index=':')
        for iso in goat_ensemble:
            comment = iso.info.get('comment', '')
            for t in comment.replace(':', ' ').split():
                try:
                    e = float(t)
                    if e < 0: iso.info['energy'] = e; break
                except ValueError: pass
            extracted.append(iso)
    return extracted

# ==============================================================================
# V. ENGINE PROTOCOLS: CRUSHER & ESCAPE ROOM
# ==============================================================================

def run_escape_room_unconditional(base_name, seed_atoms, work_dir, sys_config, cores):
    """Generates structurally diverse seeds using in-memory high-speed MLFFs/Tight-Binding."""
    print_status(f"Triggering FAST ESCAPE ROOM protocol to violently map topological basins...", "info")
    escaped_isomers = []
    calc = None
    calc_name = "None"
    
    # 1. Attempt MACE-OFF23 (GPU Accelerated)
    if is_mace_applicable(seed_atoms):
        try:
            from mace.calculators import mace_mp
            import logging
            logging.getLogger("mace").setLevel(logging.ERROR)
            device = "cuda" if sys_config.get("hardware", {}).get("gpu_type") == "NVIDIA" else "cpu"
            calc = mace_mp(model="small", device=device, default_dtype="float32")
            calc_name = f"MACE-OFF23 ({device.upper()})"
        except Exception as e:
            print_status(f"MACE setup bypassed: {e}", "warning")
            
    # 2. Attempt tblite (GFN2-xTB) as Fallback
    if calc is None:
        try:
            from tblite.ase import TBLite
            calc = TBLite(method="GFN2-xTB")
            calc_name = "tblite (GFN2-xTB)"
        except Exception as e:
            print_status(f"tblite setup bypassed: {e}", "warning")

    if calc is not None:
        print_status(f"Escape Room Engine Engaged: {calc_name} (In-Memory ASE)", "success")
        from ase.optimize import LBFGS
        from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
        from ase.md.langevin import Langevin
        from ase import units
        
        eV_to_Hartree = 0.03674930814
        
        # Phase 1: Thermal Shock & Quench
        try:
            print_status("Phase 1: Thermal Shock (1000K MD) & Rapid Quench...", "info")
            shock_atoms = seed_atoms.copy()
            shock_atoms.calc = calc
            
            MaxwellBoltzmannDistribution(shock_atoms, temperature_K=1000)
            dyn = Langevin(shock_atoms, 1.0 * units.fs, temperature_K=1000, friction=0.01, logfile=None)
            dyn.run(250)
            
            opt1 = LBFGS(shock_atoms, logfile=None)
            opt1.run(fmax=0.05, steps=250)
            
            shock_atoms.info['energy'] = shock_atoms.get_potential_energy() * eV_to_Hartree
            escaped_isomers.append(shock_atoms.copy())
        except Exception as e:
            print_status(f"Thermal Shock failed: {e}", "error")

        # Phase 2: Stochastic Rattle & Quench
        try:
            print_status("Phase 2: Stochastic Rattle (0.25 Å) & Rapid Quench...", "info")
            rattle_atoms = seed_atoms.copy()
            rattle_atoms.rattle(stdev=0.25, seed=42)
            rattle_atoms.calc = calc
            
            opt2 = LBFGS(rattle_atoms, logfile=None)
            opt2.run(fmax=0.05, steps=250)
            
            rattle_atoms.info['energy'] = rattle_atoms.get_potential_energy() * eV_to_Hartree
            escaped_isomers.append(rattle_atoms.copy())
        except Exception as e:
            print_status(f"Rattle failed: {e}", "error")
            
        for idx, iso in enumerate(escaped_isomers):
            write(os.path.join(work_dir, f"{base_name}_Escape_Out_{idx}.xyz"), iso)
            
        return escaped_isomers
        
    else:
        print_status("No fast Python calculators available. Falling back to Subprocess...", "warning")
        oet_gxtb_path = find_oet_binary("oet_gxtb", sys_config)
        
        # ExtOpt Subprocess Fallback (Strict g-xTB override)
        shocked_atoms = Atoms(symbols=seed_atoms.get_chemical_symbols(), positions=seed_atoms.get_positions())
        shocked_atoms.rattle(stdev=0.15, seed=123)

        shock_opt_name = f"{base_name}_Escape_MD_OPT"
        write(os.path.join(work_dir, f"{shock_opt_name}.xyz"), shocked_atoms)
        with open(os.path.join(work_dir, f"{shock_opt_name}.inp"), 'w') as f:
            f.write(f"! ExtOpt OPT\n%method\n  extcmd \"{oet_gxtb_path}\"\nend\n%pal nprocs {cores} end\n* xyzfile 0 1 {shock_opt_name}.xyz\n")
        
        opt_out = run_orca_subprocess_safely("Escape Room: Quenching Thermal Shock", f"{shock_opt_name}.inp", work_dir, sys_config, exit_on_fail=False)
        if opt_out:
            opt_md = read(os.path.join(work_dir, f"{shock_opt_name}.xyz"))
            opt_md.info['energy'] = parse_orca_energy(opt_out)
            escaped_isomers.append(opt_md)

        rattle_atoms = Atoms(symbols=seed_atoms.get_chemical_symbols(), positions=seed_atoms.get_positions())
        rattle_atoms.rattle(stdev=0.25, seed=42) 
        rattle_name = f"{base_name}_Escape_Rattle"
        write(os.path.join(work_dir, f"{rattle_name}.xyz"), rattle_atoms)
        
        with open(os.path.join(work_dir, f"{rattle_name}.inp"), 'w') as f:
            f.write(f"! ExtOpt OPT\n%method\n  extcmd \"{oet_gxtb_path}\"\nend\n%pal nprocs {cores} end\n* xyzfile 0 1 {rattle_name}.xyz\n")
        
        rattle_out = run_orca_subprocess_safely("Escape Room: Quenching Rattle Distortion", f"{rattle_name}.inp", work_dir, sys_config, exit_on_fail=False)
        if rattle_out:
            try:
                opt_rattle = read(os.path.join(work_dir, f"{rattle_name}.xyz"))
                opt_rattle.info['energy'] = parse_orca_energy(rattle_out)
                escaped_isomers.append(opt_rattle)
            except: pass
        
        return escaped_isomers

# ==============================================================================
# VII. ENGINE DIAGNOSTICS & INITIALIZATION
# ==============================================================================

def run_engine_diagnostics():
    """Validates the mathematical protocols inside the engine before execution."""
    print(f"\n{Colors.BOLD}--- Engine Diagnostics & Self-Test ---{Colors.ENDC}")
    passed = 0
    total = 3

    try:
        h2o = Atoms('H2O', positions=[[0,0,0], [0.75,0,0], [0,0.75,0]])
        fe = Atoms('Fe', positions=[[0,0,0]])
        if is_mace_applicable(h2o) and not is_mace_applicable(fe):
            print_status("MACE Applicability Matrix: PASS", "success")
            passed += 1
        else:
            print_status("MACE Applicability Matrix: FAIL", "error")
    except Exception as e: print_status(f"MACE Check: ERROR ({e})", "error")

    try:
        P = np.array([[0,0,0], [1,0,0], [0,1,0]], dtype=float)
        Q = np.array([[1,1,0], [1,2,0], [0,1,0]], dtype=float) 
        rmsd = calculate_kabsch_rmsd(P, Q)
        if rmsd < 1e-6:
            print_status("Kabsch SVD Rotational Alignment: PASS", "success")
            passed += 1
        else:
            print_status(f"Kabsch Alignment: FAIL (RMSD={rmsd})", "error")
    except Exception as e: print_status(f"Kabsch Alignment: ERROR ({e})", "error")

    try:
        mem = estimate_orca_memory(h2o, method="CCSD(T)", basis_set="def2-TZVP")
        if 4000 < mem < 10000: 
            print_status(f"Memory Allocator Engine: PASS (Est: {mem} MB per core)", "success")
            passed += 1
        else:
            print_status(f"Memory Allocator Engine: FAIL (Est: {mem} MB)", "error")
    except Exception as e: print_status(f"Memory Allocator Engine: ERROR ({e})", "error")

    print(f"\nDiagnostics completed: {Colors.OKGREEN if passed == total else Colors.FAIL}{passed}/{total} tests passed.{Colors.ENDC}")
    return passed == total

def initialize_engine():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 1.2: SCRIPT ENGINE CORE{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    
    sys_cfg = load_system_config()
    usr_cfg = load_user_config()
    
    if not sys_cfg or not usr_cfg:
        print(f"\n{Colors.FAIL}{Colors.BOLD}🚨 ENGINE FAILURE: Missing configurations. Ensure Stages 0 and 1.1 are complete. 🚨{Colors.ENDC}")
        sys.exit(1)

    print_status("System & User Configurations Mounted.", "success")
    print_status("Notification Engine Armed.", "success")
    
    if run_engine_diagnostics():
        print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 1.2 COMPLETE: Engine is fully validated and active in memory.{Colors.ENDC}")
        print(f"You may now execute Stage 1.3 to load the Visual Triage Engine.")
    else:
        print(f"\n{Colors.FAIL}{Colors.BOLD}🚨 ENGINE FAILURE: Diagnostic tests failed. Check for environment corruption. 🚨{Colors.ENDC}")

if __name__ == "__main__":
    initialize_engine()



 ChemComp_OPI-ORCA_v3-1 - STAGE 1.2: SCRIPT ENGINE CORE
✅ System & User Configurations Mounted.
✅ Notification Engine Armed.

--- Engine Diagnostics & Self-Test ---
✅ MACE Applicability Matrix: PASS
✅ Kabsch SVD Rotational Alignment: PASS
✅ Memory Allocator Engine: PASS (Est: 4108 MB per core)

Diagnostics completed: 3/3 tests passed.

🏁 STAGE 1.2 COMPLETE: Engine is fully validated and active in memory.
You may now execute Stage 1.3 to load the Visual Triage Engine.


🖥️ Stage 1.3: Script Engine Part 2

Purpose: Mounts high-accuracy optimization loops, the Topographic Grouping Crusher, Inventory Export logic, and interactive 3D UI Triage into memory.

In [10]:
#!/usr/bin/env python3

# ==============================================================================
# 01-VISUAL-IS (Stage 1.3)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage acts as Part 2 of the Script Engine. It houses the macroscopic 
# Thermodynamic Export protocols, the advanced Visual Isomer Selection UI, 
# and the Universal High-Accuracy Parallel Optimization Loops. 
# 
# 2. Use instructions:
# Execute this script directly in your active Jupyter environment after Stage 1.2. 
# It will load these advanced UI and Optimization protocols into active memory.
# ==============================================================================

import os
import io
import csv
import sys
import json
import numpy as np
import subprocess
import concurrent.futures

try:
    from ase import Atoms
    from ase.io import read, write
    from ase.neighborlist import natural_cutoffs, NeighborList
    import networkx as nx
except ImportError:
    print("❌ ERROR: Required libraries (ASE, NetworkX) not found. Please run Stage 0.")
    sys.exit(1)

# Ensure UI Libraries are present
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import py3Dmol
except ImportError:
    print("⚠️ UI Libraries missing. Installing ipywidgets and py3Dmol...")
    subprocess.run([sys.executable, "-m", "pip", "install", "ipywidgets", "py3Dmol"], capture_output=True)
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import py3Dmol

# Inherit Formatting from memory if available
if 'Colors' in globals():
    Colors = globals()['Colors']
    print_status = globals()['print_status']
    calculate_kabsch_rmsd = globals()['calculate_kabsch_rmsd']
else:
    class Colors:
        OKCYAN = '\033[96m'; OKGREEN = '\033[92m'; WARNING = '\033[93m'; FAIL = '\033[91m'; ENDC = '\033[0m'; BOLD = '\033[1m'
    def print_status(msg, status="info"):
        print(f"[{status.upper()}] {msg}")

def get_fragment_count(atoms):
    cutoffs = [c * 1.2 for c in natural_cutoffs(atoms)]
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)
    matrix = nl.get_connectivity_matrix()
    graph = nx.from_scipy_sparse_array(matrix)
    return len(list(nx.connected_components(graph)))

def align_structures(ref_atoms, mobile_atoms):
    """Physically rotates mobile_atoms to perfectly overlap ref_atoms using Kabsch SVD."""
    P = mobile_atoms.get_positions()
    Q = ref_atoms.get_positions()
    Pc = P - np.mean(P, axis=0)
    Qc = Q - np.mean(Q, axis=0)
    C = np.dot(Pc.T, Qc)
    U, S, Vt = np.linalg.svd(C)
    
    if (np.linalg.det(U) * np.linalg.det(Vt)) < 0.0:
        S[-1] = -S[-1]
        U[:, -1] = -U[:, -1]
        
    R = np.dot(U, Vt)
    mobile_atoms.set_positions(np.dot(Pc, R) + np.mean(Q, axis=0))

# ==============================================================================
# UNIVERSAL OPTIMIZATION LOOPS & MOLSYM PROTOCOLS
# ==============================================================================
SYSTEM_CONFIG = "opi_system_config.json"
MOLSYM_AVAILABLE = False
if os.path.exists(SYSTEM_CONFIG):
    with open(SYSTEM_CONFIG, 'r') as f:
        _sys_config_tmp = json.load(f)
        _molsym_path = _sys_config_tmp.get("paths", {}).get("molsym_path", "")
        if _molsym_path and _molsym_path not in sys.path:
            sys.path.append(_molsym_path)
        try:
            import molsym
            MOLSYM_AVAILABLE = True
        except ImportError:
            pass

def apply_molsym_preopt(atoms, base_name, idx, work_dir, prefix="molsym"):
    """Aligns and symmetrizes Cartesian coordinates using MolSym."""
    if not MOLSYM_AVAILABLE: return atoms
    temp_xyz = os.path.join(work_dir, f"{prefix}_temp_{base_name}_{idx}.xyz")
    write(temp_xyz, atoms)
    try:
        mol = molsym.Molecule.from_file(temp_xyz)
        mol.symmetrize()
        sym_xyz = os.path.join(work_dir, f"{prefix}_sym_{base_name}_{idx}.xyz")
        if os.path.exists(sym_xyz):
            sym_atoms = read(sym_xyz)
            os.remove(sym_xyz); os.remove(temp_xyz)
            return sym_atoms
    except Exception: pass
    if os.path.exists(temp_xyz): os.remove(temp_xyz)
    return atoms

def execute_pyscf_gpu_subprocess(xyz_file, out_xyz, charge, spin_mult):
    """Writes and executes an isolated PySCF GPU script."""
    spin_val = spin_mult - 1
    script_content = f"""
import sys
from pyscf import gto, dft
try:
    from pyscf.geomopt.geometric_solver import optimize
except ImportError:
    print("geomeTRIC optimizer not installed. PySCF geometry optimization aborted.")
    sys.exit(1)

mol = gto.M(atom='{xyz_file}', basis='def2-tzvp', charge={charge}, spin={spin_val})
mol.build()

mf = dft.RKS(mol) if {spin_val} == 0 else dft.UKS(mol)
mf.xc = 'wB97X-D4'

try:
    import gpu4pyscf
    mf = mf.to_gpu()
except ImportError: pass

try:
    mol_eq = optimize(mf, maxsteps=150)
    mol_eq.tofile('{out_xyz}')
except Exception as e:
    print(f"PySCF Optimization Failed: {{e}}")
    sys.exit(1)
"""
    script_name = f"pyscf_runner_{os.path.basename(xyz_file)}.py"
    with open(script_name, "w") as f: f.write(script_content)
    res = subprocess.run([sys.executable, script_name], capture_output=True, text=True)
    if os.path.exists(script_name): os.remove(script_name)
    return res.returncode == 0 and os.path.exists(out_xyz)

def engine_high_acc_opt_loop(preopt_isomers, base_name, work_dir, routing_flag, charge, mult, concurrent_jobs, cores_per_job, sys_config, stage_suffix, job_prefix_mod):
    """Handles High-Accuracy optimizations (PySCF GPU fallback to ORCA r2SCAN-3c)."""
    run_orca_safely = globals().get('run_orca_subprocess_safely')
    parse_energy = globals().get('parse_orca_energy')
    parse_freq = globals().get('parse_first_vibrational_freq')
    
    high_acc_isomers = []

    if routing_flag == "PySCF_GPU":
        print_status(f"Executing ωB97X-D4/def2-TZVP on PySCF (GPU Accelerated)...", "info")
        for idx, iso in enumerate(preopt_isomers):
            iso_path = os.path.join(work_dir, f"pyscf_{job_prefix_mod}_in_{idx}.xyz")
            out_path = os.path.join(work_dir, f"pyscf_{job_prefix_mod}_out_{idx}.xyz")
            write(iso_path, iso)

            if execute_pyscf_gpu_subprocess(iso_path, out_path, charge, mult):
                high_acc_isomers.append(read(out_path))
            else:
                print_status(f"PySCF Failed for Iso{idx}. Enforcing ORCA Fallback...", "warning")
                routing_flag = "ORCA_CPU"
                break

    if routing_flag == "ORCA_CPU":
        print_status(f"Executing r²SCAN-3c on ORCA (CPU Bound)...", "info")
        def run_orca_opt(idx, iso):
            iso_path = os.path.join(work_dir, f"orca_{job_prefix_mod}_in_{idx}.xyz")
            inp_name = f"{base_name}_{job_prefix_mod}_{idx}.inp"
            pickett_name = f"{base_name}_Iso{idx}_{stage_suffix}.pickett"
            write(iso_path, iso)
            with open(os.path.join(work_dir, inp_name), "w") as f:
                f.write(f"! r2SCAN-3c OPT FREQ\n%pal nprocs {cores_per_job} end\n%maxcore 500\n")
                f.write(f"%output\n  Pickettname \"{pickett_name}\"\nend\n")
                f.write(f"* xyzfile {charge} {mult} {os.path.basename(iso_path)}\n")
            
            out = run_orca_safely(f"r²SCAN Iso{idx}", inp_name, work_dir, sys_config, exit_on_fail=False)
            if out:
                try:
                    final_iso = read(os.path.join(work_dir, f"{base_name}_{job_prefix_mod}_{idx}_OPT_FREQ.xyz"))
                    final_iso.info['energy'] = parse_energy(out)
                    final_iso.info['v1_freq'] = parse_freq(out)
                    return final_iso
                except: pass
            return None

        # Switched to ThreadPoolExecutor to bypass pickle restrictions on lambda functions
        with concurrent.futures.ThreadPoolExecutor(max_workers=concurrent_jobs) as executor:
            for res in executor.map(lambda arg: run_orca_opt(*arg), enumerate(preopt_isomers)):
                if res: high_acc_isomers.append(res)

    return high_acc_isomers

def engine_ccsdt_opt_loop(isomers, base_name, work_dir, block_header, maxcore, charge, mult, concurrent_jobs, cores_per_job, sys_config, stage_suffix, job_prefix_mod, requires_freq=True):
    """Handles rigorous CCSD(T)/DLPNO-CCSD(T) extrapolations with AutoStart checkpoints."""
    run_orca_safely = globals().get('run_orca_subprocess_safely')
    parse_energy = globals().get('parse_orca_energy')
    parse_freq = globals().get('parse_first_vibrational_freq')
    
    high_level_isomers = []

    def run_high_level_opt(idx, iso):
        sym_iso = apply_molsym_preopt(iso, base_name, idx, work_dir)
        job_prefix = f"{base_name}_{job_prefix_mod}_{idx}"
        inp_name = f"{job_prefix}.inp"
        out_name = f"{job_prefix}.out"
        iso_path = os.path.join(work_dir, f"{job_prefix}.xyz")
        gbw_file = os.path.join(work_dir, f"{job_prefix}.gbw")
        trj_file = os.path.join(work_dir, f"{job_prefix}_trj.xyz")
        pickett_name = f"{base_name}_Iso{idx}_{stage_suffix}.pickett"

        if os.path.exists(os.path.join(work_dir, out_name)):
            with open(os.path.join(work_dir, out_name), 'r') as f:
                content = f.read()
                success = ("ORCA TERMINATED NORMALLY" in content)
                if requires_freq: success = success and ("VIBRATIONAL FREQUENCIES" in content)
                if success:
                    print_status(f"Iso{idx}: Calculation already complete. Recovering from checkpoint.", "success")
                    try:
                        chk_iso = read(os.path.join(work_dir, f"{job_prefix}_OPT_FREQ.xyz" if requires_freq else f"{job_prefix}_OPT.xyz"))
                        chk_iso.info['energy'] = parse_energy(os.path.join(work_dir, out_name))
                        if requires_freq: chk_iso.info['v1_freq'] = parse_freq(os.path.join(work_dir, out_name))
                        elif 'v1_freq' in iso.info: chk_iso.info['v1_freq'] = iso.info['v1_freq']
                        return chk_iso
                    except: pass

        if os.path.exists(trj_file):
            print_status(f"Iso{idx}: Incomplete run detected. Recovering last known geometry from trajectory.", "warning")
            try: sym_iso = read(trj_file, index=':')[-1]
            except: pass

        write(iso_path, sym_iso)

        with open(os.path.join(work_dir, inp_name), "w") as f:
            f.write(block_header)
            if os.path.exists(gbw_file):
                print_status(f"Iso{idx}: Injecting .gbw wavefunction for AutoStart.", "info")
                f.write(f"! MORead\n%moinp \"{os.path.basename(gbw_file)}\"\n")
            f.write(f"%pal nprocs {cores_per_job} end\n%maxcore {maxcore}\n")
            f.write(f"%output\n  Pickettname \"{pickett_name}\"\nend\n")
            f.write(f"* xyzfile {charge} {mult} {os.path.basename(iso_path)}\n")

        out = run_orca_safely(f"CC-Extrap Iso{idx}", inp_name, work_dir, sys_config, exit_on_fail=False)
        if out:
            try:
                final_iso = read(os.path.join(work_dir, f"{job_prefix}_OPT_FREQ.xyz" if requires_freq else f"{job_prefix}_OPT.xyz"))
                final_iso.info['energy'] = parse_energy(out)
                if requires_freq: final_iso.info['v1_freq'] = parse_freq(out)
                elif 'v1_freq' in iso.info: final_iso.info['v1_freq'] = iso.info['v1_freq']
                return final_iso
            except: pass
        return None

    # Switched to ThreadPoolExecutor to bypass pickle restrictions on lambda functions
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrent_jobs) as executor:
        for res in executor.map(lambda arg: run_high_level_opt(*arg), enumerate(isomers)):
            if res: high_level_isomers.append(res)
    return high_level_isomers

def engine_revdsd_opt_loop(isomers, base_name, work_dir, block_header, maxcore, charge, mult, concurrent_jobs, cores_per_job, sys_config, stage_suffix, job_prefix_mod):
    """Handles rigorous Double-Hybrid (revDSD-PBEP86-D4) extrapolations with AutoStart checkpoints."""
    run_orca_safely = globals().get('run_orca_subprocess_safely')
    parse_energy = globals().get('parse_orca_energy')
    
    high_level_isomers = []

    def run_high_level_opt(idx, iso):
        sym_iso = apply_molsym_preopt(iso, base_name, idx, work_dir)
        job_prefix = f"{base_name}_{job_prefix_mod}_{idx}"
        inp_name = f"{job_prefix}.inp"
        out_name = f"{job_prefix}.out"
        iso_path = os.path.join(work_dir, f"{job_prefix}.xyz")
        gbw_file = os.path.join(work_dir, f"{job_prefix}.gbw")
        trj_file = os.path.join(work_dir, f"{job_prefix}_trj.xyz")
        pickett_name = f"{base_name}_Iso{idx}_{stage_suffix}.pickett"

        if os.path.exists(os.path.join(work_dir, out_name)):
            with open(os.path.join(work_dir, out_name), 'r') as f:
                if "ORCA TERMINATED NORMALLY" in f.read():
                    print_status(f"Iso{idx}: Calculation already complete. Recovering from checkpoint.", "success")
                    try:
                        chk_iso = read(os.path.join(work_dir, f"{job_prefix}_OPT.xyz"))
                        chk_iso.info['energy'] = parse_energy(os.path.join(work_dir, out_name))
                        if 'v1_freq' in iso.info: chk_iso.info['v1_freq'] = iso.info['v1_freq']
                        return chk_iso
                    except: pass

        if os.path.exists(trj_file):
            print_status(f"Iso{idx}: Incomplete run detected. Recovering last known geometry from trajectory.", "warning")
            try: sym_iso = read(trj_file, index=':')[-1]
            except: pass

        write(iso_path, sym_iso)

        with open(os.path.join(work_dir, inp_name), "w") as f:
            f.write(block_header)
            if os.path.exists(gbw_file):
                print_status(f"Iso{idx}: Injecting .gbw wavefunction for AutoStart.", "info")
                f.write(f"! MORead\n%moinp \"{os.path.basename(gbw_file)}\"\n")
            f.write(f"%pal nprocs {cores_per_job} end\n%maxcore {maxcore}\n")
            f.write(f"%output\n  Pickettname \"{pickett_name}\"\nend\n")
            f.write(f"* xyzfile {charge} {mult} {os.path.basename(iso_path)}\n")

        out = run_orca_safely(f"DH-Extrap Iso{idx}", inp_name, work_dir, sys_config, exit_on_fail=False)
        if out:
            try:
                final_iso = read(os.path.join(work_dir, f"{job_prefix}_OPT.xyz"))
                final_iso.info['energy'] = parse_energy(out)
                if 'v1_freq' in iso.info: final_iso.info['v1_freq'] = iso.info['v1_freq']
                return final_iso
            except: pass
        return None

    # Switched to ThreadPoolExecutor to bypass pickle restrictions on lambda functions
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrent_jobs) as executor:
        for res in executor.map(lambda arg: run_high_level_opt(*arg), enumerate(isomers)):
            if res: high_level_isomers.append(res)
    return high_level_isomers

# ==============================================================================
# GROUPING CRUSHER & EXPORT PROTOCOLS
# ==============================================================================

_CRUSHER_STATE = {"exec_count": -1, "aggression": 1.0}

def get_crusher_aggression():
    global _CRUSHER_STATE
    try:
        from IPython import get_ipython
        current_exec = get_ipython().execution_count
    except Exception:
        current_exec = -1

    if current_exec != _CRUSHER_STATE["exec_count"] or current_exec == -1:
        print(f"\n{Colors.OKCYAN}--- Topological Crusher Protocol ---{Colors.ENDC}")
        print("The Crusher deduplicates isomers based on spatial and energetic tolerances.")
        print(f" {Colors.WARNING}[0.5]{Colors.ENDC} : Lenient (Retains more isomers, highly sensitive to subtle differences)")
        print(f" {Colors.WARNING}[1.0]{Colors.ENDC} : Standard (Default baseline algorithmic tolerance)")
        print(f" {Colors.WARNING}[2.0]{Colors.ENDC} : Strict (Merges slightly distinct conformers)")
        print(f" {Colors.WARNING}[5.0]{Colors.ENDC} : Aggressive (Heavily reduces the ensemble)")
        print(f" {Colors.WARNING}[10.0]{Colors.ENDC}: Extreme (Merges broadly similar structural basins)")
        
        while True:
            ans = input("Select Aggression Multiplier [0.5, 1, 2, 5, 10] (Default 1.0): ").strip()
            if not ans:
                agg = 1.0
                break
            try:
                agg = float(ans)
                if agg in [0.5, 1.0, 2.0, 5.0, 10.0]: break
                else: print("Please enter a valid multiplier from the list.")
            except ValueError:
                print("Invalid input.")
                
        print_status(f"Crusher Aggression set to {agg}x for this Stage execution.", "success")
        _CRUSHER_STATE["aggression"] = agg
        _CRUSHER_STATE["exec_count"] = current_exec

    return _CRUSHER_STATE["aggression"]

def the_grouping_crusher(isomer_list, temperature, silent=False, aggression=None):
    if aggression is None: aggression = get_crusher_aggression()
    if not silent: print_status(f"Deploying The Grouping Crusher (Aggression: {aggression}x) on {len(isomer_list)} structures...", "info")
    kb_hartree_per_K = 3.1668e-6
    energy_tol = temperature * kb_hartree_per_K * aggression

    parsed_isomers = []
    for iso in isomer_list:
        positions = iso.get_positions()
        moments = np.sort(iso.get_moments_of_inertia())
        diff = positions[:, np.newaxis, :] - positions[np.newaxis, :, :]
        dist_matrix = np.linalg.norm(diff, axis=-1)
        parsed_isomers.append({
            "atoms": iso, "energy": iso.info.get('energy', 0.0), 
            "positions": positions, "moments": moments, "dist_matrix": dist_matrix
        })

    parsed_isomers.sort(key=lambda x: x["energy"])
    unique_pool = []
    for current in parsed_isomers:
        is_duplicate = False
        for group in unique_pool:
            unique = group[0] 
            if abs(current["energy"] - unique["energy"]) < energy_tol:
                I1, I2 = current["moments"], unique["moments"]
                with np.errstate(divide='ignore', invalid='ignore'):
                    I_diff = np.abs(I1 - I2) / np.maximum(I1, 1e-6)
                if np.all(I_diff < (0.02 * aggression)):
                    is_duplicate = True
                elif calculate_kabsch_rmsd(current["positions"], unique["positions"]) < (0.15 * aggression):
                    is_duplicate = True
                else:
                    dist_diff = np.abs(current["dist_matrix"] - unique["dist_matrix"])
                    dyn_tol = (0.05 * unique["dist_matrix"] + 0.05) * aggression
                    if np.all(dist_diff < dyn_tol): is_duplicate = True
                        
            if is_duplicate:
                align_structures(unique["atoms"], current["atoms"])
                current["atoms"].info['energy'] = current["energy"]
                group.append(current)
                break
                
        if not is_duplicate: 
            current["atoms"].info['energy'] = current["energy"]
            unique_pool.append([current])

    if not silent: print_status(f"Grouping Crusher Complete. Formed {len(unique_pool)} unique structural groups.", "success")
    return [[u["atoms"] for u in g] for g in unique_pool]

def the_crusher(isomer_list, temperature, silent=False, aggression=None):
    groups = the_grouping_crusher(isomer_list, temperature, silent, aggression)
    return [g[0] for g in groups]

def export_thermodynamic_inventory(final_unique, base_name, temperature, output_dir, stage_prefix="Stage2"):
    print_status("Compiling Thermodynamic and Spectroscopic Data...", "info")
    final_unique.sort(key=lambda x: x.info.get('energy', 0.0))
    min_E = final_unique[0].info.get('energy', 0.0)
    kb_hartree = 3.1668e-6

    processed_data = []
    total_w = 0.0
    for idx, iso in enumerate(final_unique):
        delta_e = iso.info.get('energy', 0.0) - min_E
        weight = np.exp(-delta_e / (temperature * kb_hartree))
        total_w += weight
        stereo_label = "Unique"
        for prev_data in processed_data:
            if abs(iso.info.get('energy', 0.0) - prev_data['atoms'].info.get('energy', 0.0)) < 1e-5:
                pos1 = iso.get_positions()
                pos2 = -1.0 * prev_data['atoms'].get_positions() 
                if calculate_kabsch_rmsd(pos1, pos2) < 0.15:
                    stereo_label = f"Enantiomer of Iso{prev_data['idx']}"
                    break
        processed_data.append({'idx': idx + 1, 'atoms': iso, 'weight': weight, 'stereo': stereo_label})

    csv_rows = []
    headers = ["Fragment-Isomer name in Script", "Enantiomer or Unique", "Energy in cm-1", 
               f"Boltzman Distribution population in % at {temperature} K", 
               "Rotational Constant A", "Rotational Constant B", "Rotational Constant C", 
               "Point Group", "First Vibrational Energy Level"]

    for pd in processed_data:
        iso = pd['atoms']
        pop = (pd['weight'] / total_w) * 100.0
        energy_cm1 = iso.info.get('energy', 0.0) * 219474.63
        I = np.sort(iso.get_moments_of_inertia())
        A = 505.379 / I[0] if I[0] > 1e-3 else 0.0
        B = 505.379 / I[1] if I[1] > 1e-3 else 0.0
        C = 505.379 / I[2] if I[2] > 1e-3 else 0.0
        pg = iso.info.get('point_group', "C1")
        v1 = iso.info.get('v1_freq', 0.0)
        
        # Pull custom isotopic name from variants if available (Stage 5 fix)
        custom_name = iso.info.get('export_custom_name')
        name_str = custom_name if custom_name else f"{base_name}_Iso{pd['idx']}"
        
        iso.info['comment'] = f"[{name_str}] [{pd['stereo']}] [{energy_cm1:.2f}] [{pop:.2f}] [{A:.2f}] [{B:.2f}] [{C:.2f}] [{pg}] [{v1:.2f}]"
        csv_rows.append([name_str, pd['stereo'], f"{energy_cm1:.2f}", f"{pop:.2f}", f"{A:.2f}", f"{B:.2f}", f"{C:.2f}", pg, f"{v1:.2f}"])

    if not os.path.exists(output_dir): os.makedirs(output_dir)
    xyz_path = os.path.join(output_dir, f"{stage_prefix}_{base_name}_Final_Ensemble.xyz")
    write(xyz_path, [p['atoms'] for p in processed_data])

    csv_path = os.path.join(output_dir, f"{stage_prefix}_{base_name}_Inventory.csv")
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(csv_rows)

    print_status(f"Inventory Exported: {xyz_path} and {csv_path}", "success")
    return xyz_path, csv_path

# ==============================================================================
# VISUAL ISOMER SELECTION UI (INTERACTIVE TRIAGE)
# ==============================================================================

def get_xyz_string(atoms):
    f = io.StringIO()
    write(f, atoms, format='xyz')
    return f.getvalue()

class VisualIsomerTriage:
    def __init__(self, grouped_isomers, base_name="Molecule"):
        if grouped_isomers and not isinstance(grouped_isomers[0], list):
            self.groups = [[iso] for iso in grouped_isomers]
        else:
            self.groups = grouped_isomers
            
        self.base_name = base_name
        self.page = 0
        self.per_page = 25
        self.join_pending_idx = None
        self.num_start = sum(len(g) for g in self.groups)
        
        self.out = widgets.Output()
        display(self.out)
        self.render()

    def render(self):
        with self.out:
            clear_output(wait=True)
            self.groups.sort(key=lambda g: g[0].info.get('energy', 0.0))
            
            start = self.page * self.per_page
            end = min(start + self.per_page, len(self.groups))
            current_groups = self.groups[start:end]
            
            grid_items = []
            for i, group in enumerate(current_groups):
                global_idx = start + i
                rep = group[0]
                energy_cm1 = rep.info.get('energy', 0.0) * 219474.63
                
                title_html = f"<div style='text-align:center; font-family:monospace; margin-bottom:5px;'>" \
                             f"<b>Group {global_idx+1}</b><br>{len(group)} Isomers<br>ΔE: {energy_cm1:.1f} cm<sup>-1</sup></div>"
                title = widgets.HTML(title_html)
                
                view = py3Dmol.view(width=400, height=400)
                colors = ['cyan', 'magenta', 'yellow', 'pink', 'orange']
                for j, iso in enumerate(group):
                    xyz = get_xyz_string(iso)
                    view.addModel(xyz, 'xyz')
                    view.setStyle({'model': -1}, {'stick': {'color': colors[j % len(colors)]}})
                view.zoomTo()
                
                view_out = widgets.Output(layout=widgets.Layout(width='400px', height='400px', border='1px solid #ccc'))
                with view_out: view.show()
                    
                btn_join = widgets.Button(description="Join", tooltip="Merge this into another group", button_style='info')
                btn_disjoin = widgets.Button(description="Disjoin", tooltip="Split all isomers in this group", button_style='warning')
                btn_del = widgets.Button(description="Delete", tooltip="Delete this entire group", button_style='danger')
                
                btn_join.global_idx = global_idx
                btn_disjoin.global_idx = global_idx
                btn_del.global_idx = global_idx
                
                btn_join.on_click(self.on_join)
                btn_disjoin.on_click(self.on_disjoin)
                btn_del.on_click(self.on_delete)
                
                if len(group) == 1: btn_disjoin.disabled = True
                    
                if self.join_pending_idx == global_idx:
                    btn_join.button_style = 'success'
                    btn_join.description = "Merging..."
                    
                controls = widgets.HBox([btn_join, btn_disjoin, btn_del], layout=widgets.Layout(justify_content='center'))
                cell = widgets.VBox([title, view_out, controls], layout=widgets.Layout(border='2px solid #444', padding='10px', margin='5px'))
                grid_items.append(cell)
                
            grid = widgets.GridBox(grid_items, layout=widgets.Layout(
                grid_template_columns="repeat(5, 430px)", grid_gap="15px"
            ))
            
            btn_prev = widgets.Button(description="< Previous Page", disabled=(self.page == 0), button_style='primary')
            btn_next = widgets.Button(description="Next Page >", disabled=(end >= len(self.groups)), button_style='primary')
            btn_accept = widgets.Button(description="Accept Selections", button_style='success', layout=widgets.Layout(width='200px'))
            
            btn_prev.on_click(self.on_prev)
            btn_next.on_click(self.on_next)
            btn_accept.on_click(self.on_accept)
            
            bottom_bar = widgets.HBox([btn_prev, btn_accept, btn_next], layout=widgets.Layout(justify_content='space-between', margin='20px 0px'))
            display(widgets.VBox([grid, bottom_bar]))

    def on_join(self, btn):
        idx = btn.global_idx
        if self.join_pending_idx is None:
            self.join_pending_idx = idx
            self.render()
        else:
            if self.join_pending_idx != idx:
                target, source = self.join_pending_idx, idx
                e_target = self.groups[target][0].info.get('energy', 0)
                e_source = self.groups[source][0].info.get('energy', 0)
                if e_source < e_target:
                    target, source = source, target
                
                for iso in self.groups[source]:
                    align_structures(self.groups[target][0], iso)
                    
                self.groups[target].extend(self.groups[source])
                del self.groups[source]
            self.join_pending_idx = None
            self.render()
            
    def on_disjoin(self, btn):
        idx = btn.global_idx
        group = self.groups[idx]
        self.groups[idx] = [group[0]]
        for iso in group[1:]:
            self.groups.append([iso])
        self.render()

    def on_delete(self, btn):
        idx = btn.global_idx
        del self.groups[idx]
        if self.join_pending_idx == idx: self.join_pending_idx = None
        self.render()
        
    def on_prev(self, btn):
        self.page -= 1; self.render()
        
    def on_next(self, btn):
        self.page += 1; self.render()
        
    def on_accept(self, btn):
        with self.out:
            clear_output(wait=True)
            print(f"\n{Colors.OKGREEN}{Colors.BOLD}✅ Selections Accepted & Finalized!{Colors.ENDC}")
            num_fragments = get_fragment_count(self.groups[0][0]) if self.groups else 0
            num_end = sum(len(g) for g in self.groups)
            print(f" • Number of Fragments: {Colors.OKCYAN}{num_fragments}{Colors.ENDC}")
            print(f" • Number Isomers Start: {Colors.WARNING}{self.num_start}{Colors.ENDC}")
            print(f" • Number Isomers End: {Colors.OKGREEN}{num_end}{Colors.ENDC} ({len(self.groups)} Unique Groups)")
            
            global absolute_final_isomers
            absolute_final_isomers = [g[0] for g in self.groups]

def initialize_engine_part_2():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 1.3: SCRIPT ENGINE PART 2 {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print_status("High-Accuracy Parallel Optimization Loops Mounted (ThreadPool Patched).", "success")
    if MOLSYM_AVAILABLE:
        print_status("MolSym Auto-Symmetrization Integration Active.", "success")
    else:
        print_status("MolSym library not found. Symmetrization will be bypassed.", "warning")
    print_status("Grouping Crusher Protocol Mounted.", "success")
    print_status("Inventory Export Protocol Mounted.", "success")
    print_status("Visual Isomer Triage Protocol Mounted.", "success")
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 1.3 COMPLETE: Engine Part 2 is fully active in memory.{Colors.ENDC}")

if __name__ == "__main__":
    initialize_engine_part_2()



 ChemComp_OPI-ORCA_v3-1 - STAGE 1.3: SCRIPT ENGINE PART 2 
✅ High-Accuracy Parallel Optimization Loops Mounted (ThreadPool Patched).
✅ MolSym Auto-Symmetrization Integration Active.
✅ Grouping Crusher Protocol Mounted.
✅ Inventory Export Protocol Mounted.
✅ Visual Isomer Triage Protocol Mounted.

🏁 STAGE 1.3 COMPLETE: Engine Part 2 is fully active in memory.


🐐 Stage 2: FRGOAT Protocol

Purpose: Executes Batch-Aware Quenched Cascade and Topographic Escape Room (GOAT) to map the conformational basins of the monomer.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 02-FRGOAT-AA (Stage 2.0)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes the Batch-Aware Quenched Cascade and Topographic Escape Room.
# Utilizing the verified Stage 1.2 Script Engine (loaded directly from memory), 
# it routes the molecule through a rigorous 9-step iterative exploration loop. 
# It inherently respects the topological constraints (MACE limits) discovered in Stage 1.1.
# 
# 2. Use instructions:
# Execute this script after completing Stages 0, 1.1, and 1.2 in your Jupyter notebook. 
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Bypasses small molecules (<4 atoms). For applicable molecules, it 
# triggers unconditional thermal/rattle shocks, performs a two-phase deep GOAT 
# search utilizing the optimized `MAXGLOBALITER 300` constraints, deduplicates 
# via the Crusher, and delegates the final CSV/XYZ export back to the Engine.
# ==============================================================================

import os
import sys
import json
import glob
import shutil
import importlib.util
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
# Actively read the Stage 1.2 material directly from memory if available
if 'the_crusher' in globals() and 'run_orca_subprocess_safely' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    parse_freq = globals()['parse_first_vibrational_freq']
    the_crusher = globals()['the_crusher']
    run_escape_room = globals()['run_escape_room_unconditional']
    extract_ensemble = globals()['extract_goat_ensemble']
    get_goat_header = globals()['get_goat_header']
    export_thermo = globals()['export_thermodynamic_inventory']
    send_ping = globals()['send_ping']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    # Fallback to physical file import for terminal execution safety
    ENGINE_PATH = "01_SCRIPT_ENG.py"
    if not os.path.exists(ENGINE_PATH):
        print(f"❌ ERROR: Engine missing from memory AND disk.")
        print("ACTION REQUIRED: Ensure you have executed the Stage 1.2 cell first.")
        sys.exit(1)

    spec = importlib.util.spec_from_file_location("engine", ENGINE_PATH)
    engine = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(engine)

    Colors = engine.Colors
    print_status = engine.print_status
    run_orca_safely = engine.run_orca_subprocess_safely
    parse_energy = engine.parse_orca_energy
    parse_freq = engine.parse_first_vibrational_freq
    the_crusher = engine.the_crusher
    run_escape_room = engine.run_escape_room_unconditional
    extract_ensemble = engine.extract_goat_ensemble
    get_goat_header = engine.get_goat_header
    export_thermo = engine.export_thermodynamic_inventory
    send_ping = engine.send_ping

try:
    from ase.io import read, write
    from ase.neighborlist import natural_cutoffs, NeighborList
    import networkx as nx
except ImportError:
    print_status("Required libraries (ASE, NetworkX) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
OUTPUT_DIR = "Isomer_xyz_GOAT_Optimized"

def get_fragment_count(atoms):
    """Uses NetworkX to definitively calculate how many disjoint fragments exist."""
    cutoffs = [c * 1.2 for c in natural_cutoffs(atoms)]
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)
    matrix = nl.get_connectivity_matrix()
    graph = nx.from_scipy_sparse_array(matrix)
    return len(list(nx.connected_components(graph)))

def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}    ChemComp_OPI-ORCA_v3-1 - STAGE 2: 02-FRGOAT-AA    {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(SYSTEM_CONFIG) or not os.path.exists(USER_CONFIG):
        print_status("Configuration files missing. Ensure Stages 0 and 1.1 are complete.", "error")
        sys.exit(1)

    with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    
    system_temperature = user_config.get("temperature", 298.15)
    ingested_files = user_config.get("ingested_files", [])
    if not ingested_files:
        print_status("No ingested files found in configuration. Run Stage 1.1.", "error")
        sys.exit(1)

    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

    cores = sys_config.get("hardware", {}).get("cpu_threads", 4)
    if cores > 16: cores = 16

    all_unique_files = []

    for item in ingested_files:
        filename = item.get("file")
        mace_applicable = item.get("mace_applicable", True)
        mace_reason = item.get("mace_reason", "Unknown")
        
        file_path = os.path.join("Isomer_xyz_Initial_Files", filename)
        base_name = filename.replace('.xyz', '')
        
        print(f"\n{Colors.BOLD}--- Iterative Discovery Loop: {base_name} ---{Colors.ENDC}")
        
        # Bypass check
        final_ensemble_path = os.path.join(OUTPUT_DIR, f"Stage2_FRGOAT_{base_name}_Final_Ensemble.xyz")
        if os.path.exists(final_ensemble_path):
            print_status(f"Stage 2 already completed for {base_name}. Bypassing...", "success")
            all_unique_files.append(final_ensemble_path)
            continue
        
        work_dir = f"Stage2_GOAT_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        initial_atoms = read(file_path)
        N_atoms = len(initial_atoms)
        
        # ---------------------------------------------------------
        # 2. THE 4-ATOM BYPASS
        # ---------------------------------------------------------
        if N_atoms < 4:
            print_status(f"{base_name} has {N_atoms} atoms. Mathematically impossible to have conformational isomers. Bypassing GOAT.", "warning")
            
            inp_filename = f"{base_name}_OPT_FREQ.inp"
            shutil.copy(file_path, os.path.join(work_dir, f"{base_name}_initial.xyz"))
            with open(os.path.join(work_dir, inp_filename), 'w') as f:
                f.write(f"! xTB2 OPT FREQ\n%pal nprocs {cores} end\n* xyzfile 0 1 {base_name}_initial.xyz\n")
                
            out_path = run_orca_safely(f"Direct OPT+FREQ ({base_name})", inp_filename, work_dir, sys_config)
            
            final_atoms = read(os.path.join(work_dir, f"{base_name}_OPT_FREQ.xyz"))
            final_atoms.info['energy'] = parse_energy(out_path)
            final_unique = [final_atoms]
            freq_outputs = [out_path]
            
        # ---------------------------------------------------------
        # THE 9-STEP ITERATIVE LOOP
        # ---------------------------------------------------------
        else:
            all_raw_isomers = []
            
            # 1. Preserve original isomer (OPT)
            inp_opt = f"{base_name}_Init_OPT.inp"
            shutil.copy(file_path, os.path.join(work_dir, f"{base_name}_Init.xyz"))
            with open(os.path.join(work_dir, inp_opt), 'w') as f:
                f.write(f"! xTB2 OPT\n%pal nprocs {cores} end\n* xyzfile 0 1 {base_name}_Init.xyz\n")
            opt_out = run_orca_safely(f"Initial Optimization ({base_name})", inp_opt, work_dir, sys_config)
            
            opt_atoms = read(os.path.join(work_dir, f"{base_name}_Init_OPT.xyz"))
            opt_atoms.info['energy'] = parse_energy(opt_out)
            all_raw_isomers.append(opt_atoms)
            
            # 3. The Escape Room Protocol (Unconditional Diversity Seeding)
            escaped = run_escape_room(base_name, opt_atoms, work_dir, sys_config, cores)
            all_raw_isomers.extend(escaped)
            
            # 4. Crusher Protocol
            unique_seeds = the_crusher(all_raw_isomers, system_temperature)
            
            # 5. GOAT-diversity (Phase 1)
            target_seed_1 = unique_seeds[-1] 
            seed_name_1 = f"{base_name}_Seed1.xyz"
            write(os.path.join(work_dir, seed_name_1), target_seed_1)
            
            # Stage 1.1 Dependency Verification
            if not mace_applicable:
                print_status(f"Stage 1.1 Flag: {mace_reason}. Forcing xTB2 Fallback.", "warning")
                goat_header = "! GOAT XTB2\n"
            else:
                goat_header = get_goat_header(sys_config, target_seed_1)
            
            inp_goat_1 = f"{base_name}_GOAT_Phase1.inp"
            with open(os.path.join(work_dir, inp_goat_1), 'w') as f:
                f.write(goat_header)
                f.write(f"%pal nprocs {cores} end\n")
                f.write("%goat\n  MAXITERMULT 6\n  MINGLOBALITER 30\n  MAXGLOBALITER 300\n  NWorkers auto\n  MAXEN 12.0\n  RMSD 1.5\n  ROTCONSTDIFF 0.1\nend\n")
                f.write(f"* xyzfile 0 1 {seed_name_1}\n")
            run_orca_safely(f"GOAT Conformer Search Phase 1 ({base_name})", inp_goat_1, work_dir, sys_config)
            all_raw_isomers.extend(extract_ensemble(work_dir, f"{base_name}_GOAT_Phase1"))
            
            # 6. Crusher Protocol
            unique_seeds = the_crusher(all_raw_isomers, system_temperature)
            
            # 7. GOAT (Phase 2)
            target_seed_2 = unique_seeds[0] 
            seed_name_2 = f"{base_name}_Seed2.xyz"
            write(os.path.join(work_dir, seed_name_2), target_seed_2)
            
            inp_goat_2 = f"{base_name}_GOAT_Phase2.inp"
            with open(os.path.join(work_dir, inp_goat_2), 'w') as f:
                f.write(goat_header) 
                f.write(f"%pal nprocs {cores} end\n")
                f.write("%goat\n  MAXITERMULT 6\n  MINGLOBALITER 30\n  MAXGLOBALITER 300\n  NWorkers auto\n  MAXEN 12.0\n  RMSD 1.5\n  ROTCONSTDIFF 0.1\nend\n")
                f.write(f"* xyzfile 0 1 {seed_name_2}\n")
            run_orca_safely(f"GOAT Conformer Search Phase 2 ({base_name})", inp_goat_2, work_dir, sys_config)
            all_raw_isomers.extend(extract_ensemble(work_dir, f"{base_name}_GOAT_Phase2"))
            
            # 8. Crusher Protocol
            final_unique = the_crusher(all_raw_isomers, system_temperature)
                
            # 9. Final Output Procedure (FREQs)
            freq_outputs = []
            print_status(f"Calculating Final Frequencies for {len(final_unique)} unique isomers...", "info")
            for idx, iso in enumerate(final_unique):
                freq_name = f"{base_name}_Iso{idx+1}_FREQ"
                write(os.path.join(work_dir, f"{freq_name}.xyz"), iso)
                with open(os.path.join(work_dir, f"{freq_name}.inp"), 'w') as f:
                    f.write(f"! xTB2 FREQ\n%pal nprocs {cores} end\n* xyzfile 0 1 {freq_name}.xyz\n")
                out_path = run_orca_safely(f"FREQ Calculation (Iso {idx+1})", f"{freq_name}.inp", work_dir, sys_config)
                freq_outputs.append(out_path)

        # ---------------------------------------------------------
        # FINAL ENSEMBLE OUTPUT GENERATION (Delegated to Stage 1.2 Engine)
        # ---------------------------------------------------------
        # Inject v1 frequencies into atoms before passing to the engine's exporter
        for iso, out_path in zip(final_unique, freq_outputs):
            iso.info['v1_freq'] = parse_freq(out_path)
            
        xyz_path, csv_path = export_thermo(
            final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage2_FRGOAT"
        )
        all_unique_files.append(xyz_path)
        
        # 10. Print Block & Visualization
        n_frag = get_fragment_count(final_unique[0])
        print(f"\n{Colors.BOLD}--- Current Configuration & Stage Summary ---{Colors.ENDC}")
        print(f" • Project Name        : {Colors.OKCYAN}{user_config.get('molecule_name', 'Unknown')}{Colors.ENDC}")
        print(f" • Total Isomers       : {Colors.OKCYAN}{len(final_unique)}{Colors.ENDC}")
        print(f" • Total Fragments     : {Colors.OKCYAN}{n_frag}{Colors.ENDC}")
        
        print(f"\n{Colors.BOLD}--- Visualizing Topology: {base_name}.xyz ---{Colors.ENDC}")
        try:
            import matplotlib.pyplot as plt
            from ase.visualize.plot import plot_atoms
            
            min_E = final_unique[0].info['energy']
            if len(final_unique) <= 5:
                vis_list = final_unique
            else:
                indices = np.linspace(0, len(final_unique)-1, 5, dtype=int)
                vis_list = [final_unique[i] for i in indices]
                
            fig, axes = plt.subplots(1, len(vis_list), figsize=(4 * len(vis_list), 4))
            if len(vis_list) == 1: axes = [axes]
            
            for ax, iso in zip(axes, vis_list):
                plot_atoms(iso, ax, rotation=('15x,15y,0z'))
                rel_e = (iso.info['energy'] - min_E) * 219474.63
                ax.set_title(f"ΔE: {rel_e:.1f} cm-1", fontweight='bold')
                ax.axis('off')
                
            plt.suptitle(f"Topological Extent: {base_name}", fontsize=14, fontweight='bold', y=1.05)
            plt.tight_layout()
            plt.show()
        except ImportError:
            print_status("Matplotlib not found. Skipping visualization.", "warning")
        except Exception as e:
            print_status(f"Visualization failed: {e}", "warning")
    
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 2 COMPLETE: Protocol Finished.{Colors.ENDC}")
    send_ping(f"✅ **ChemComp_OPI-ORCA_v3-1 Pipeline Update:** Stage 2 (Escape Room & Loop) complete! Generated final thermodynamic ensembles.")

if __name__ == "__main__":
    main()


👁️ Stage 2.1: Visual Isomer Selection

Purpose: Provides an interactive 3D UI to verify, merge, or disjoin the automatically generated monomer isomers.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 02_1-VISUAL-TRIAGE-AA (Stage 2.1)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage loads the automatically generated thermodynamic ensembles from Stage 2 
# (GOAT Optimization) and routes them into the Visual Isomer Selection Protocol 
# established in Stage 1.3. It allows the user to visually inspect the topological 
# deduplication, split false duplicates (Disjoin), merge missed duplicates (Join), 
# and delete impossible geometries.
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment. Select your molecule from 
# the generated dropdown menu and click "Launch Triage". Use the 3D viewers to 
# verify the structures. Once complete, click "Accept Selections" to instantly 
# generate a Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI. Upon acceptance, successfully writes 
# to the 'Isomer_xyz_Stage2_1_Verified' folder using the Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_GOAT_Optimized"
OUTPUT_DIR = "Isomer_xyz_Stage2_1_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class VerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine lifecycle.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage2_1_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Macroscopic Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_interactive_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 2.1: VISUAL ISOMER SELECTION{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found. Run Stage 2.", "error")
        return

    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage2_FRGOAT_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 2 Ensembles found in {INPUT_DIR}.", "error")
        return

    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage2_FRGOAT_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Molecule:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 2 Ensemble to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            print_status(f"Loading {base_name} into memory...", "info")
            
            try:
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                print_status("Executing Grouping Crusher alignment...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                VerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_interactive_triage()


🎯 Stage 2.2: High-Accuracy Refine

Purpose: Refines the visually verified monomer isomers using high-accuracy DFT (r²SCAN-3c or PySCF GPU).

In [11]:
#!/usr/bin/env python3

# ==============================================================================
# 02_2-HIGH-ACC-OPT-AA (Stage 2.2)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage bridges the gap between topological discovery and high-accuracy DFT.
# It symmetrizes coordinates using MolSym, performs a MACE pre-optimization, and 
# then routes the isomers to either GPU-accelerated PySCF (ωB97X-D4/def2-TZVP) 
# or CPU-parallelized ORCA (r²SCAN-3c) by calling the Engine Loop from memory.
# It concludes with a strict Topographic xTB Funnel filter.
#
# *NEW BYPASS FEATURE*: If the user already knows the target isomers and wishes 
# to skip the GOAT exploration (Stages 2/2.1), this stage will automatically scan 
# the 'Isomer_xyz_Initial_Files' directory and optimize those initial geometries 
# directly if no Stage 2.1 output is found.
# 
# 2. Use instructions:
# Execute this script after completing Stage 2.1 in VS Code.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates highly refined ab initio geometries, outputs Pickett files 
# for SPCAT, and exports a fully authenticated Thermodynamic Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_high_acc_opt_loop' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    parse_freq = globals()['parse_first_vibrational_freq']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    get_goat_header = globals()['get_goat_header']
    apply_molsym_preopt = globals()['apply_molsym_preopt']
    engine_high_acc_opt_loop = globals()['engine_high_acc_opt_loop']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage2_1_Verified"
OUTPUT_DIR = "Isomer_xyz_Stage2_2_HighAcc"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

def calculate_dynamic_cores(num_isomers):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_mb = mem_gb * 1024
    
    max_jobs_by_mem = max(1, int(mem_mb / 500))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 2.2: HIGH-ACCURACY REFINE {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(INPUT_DIR) and not os.path.exists("Isomer_xyz_Initial_Files"):
        print_status(f"Neither Stage 2.1 nor Initial Input directories found. Run Stage 1.1 or 2.1.", "error")
        sys.exit(1)
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        routing_flag = item.get("pyscf_v_orca", "ORCA_CPU")
        mace_applicable = item.get("mace_applicable", True)
        
        ensemble_file_verified = os.path.join(INPUT_DIR, f"Stage2_1_Verified_{base_name}_Final_Ensemble.xyz")
        ensemble_file_initial = os.path.join("Isomer_xyz_Initial_Files", f"{base_name}.xyz")
        
        if os.path.exists(ensemble_file_verified):
            ensemble_file = ensemble_file_verified
        elif os.path.exists(ensemble_file_initial):
            ensemble_file = ensemble_file_initial
            print_status(f"Stage 2.1 ensemble missing. Bypassing Isomer Search & utilizing Initial File for {base_name}.", "info")
        else:
            print_status(f"Missing both Stage 2.1 ensemble and initial file for {base_name}. Skipping.", "error")
            continue
            
        print(f"\n{Colors.BOLD}--- Processing Ensemble: {base_name} ---{Colors.ENDC}")
        isomers = read(ensemble_file, index=':')
        num_isomers = len(isomers)
        
        work_dir = f"Stage2_2_HighAcc_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers)
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job", "info")
        
        # ---------------------------------------------------------
        # PHASE 1: MolSym & MACE Pre-Optimization
        # ---------------------------------------------------------
        print_status("Phase 1: MolSym Symmetrization & MACE-OFF23 Pre-Optimization...", "info")
        preopt_isomers = []
        for idx, iso in enumerate(isomers):
            sym_iso = apply_molsym_preopt(iso, base_name, idx, work_dir)
            
            if mace_applicable:
                iso_path = os.path.join(work_dir, f"{base_name}_mace_{idx}.xyz")
                write(iso_path, sym_iso)
                
                mace_inp = f"{base_name}_mace_{idx}.inp"
                mace_hdr = get_goat_header(sys_config, sym_iso)
                with open(os.path.join(work_dir, mace_inp), "w") as f:
                    f.write(mace_hdr.replace("! ExtOpt GOAT", "! ExtOpt OPT")) 
                    f.write(f"%pal nprocs {cores_per_job} end\n")
                    f.write(f"* xyzfile {charge} {mult} {os.path.basename(iso_path)}\n")
                    
                out = run_orca_safely(f"MACE PreOpt Iso{idx}", mace_inp, work_dir, sys_config, exit_on_fail=False)
                if out:
                    try:
                        sym_iso = read(os.path.join(work_dir, f"{base_name}_mace_{idx}_OPT.xyz"))
                    except: pass
                    
            preopt_isomers.append(sym_iso)

        # ---------------------------------------------------------
        # PHASE 2: Hardware Routed High-Accuracy DFT (Delegated to Stage 1.3 Engine)
        # ---------------------------------------------------------
        print(f"\n{Colors.BOLD}--- Phase 2: Quantum Hardware Routing ({routing_flag}) ---{Colors.ENDC}")
        
        high_acc_isomers = engine_high_acc_opt_loop(
            preopt_isomers=preopt_isomers,
            base_name=base_name,
            work_dir=work_dir,
            routing_flag=routing_flag,
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage2_2",
            job_prefix_mod="r2scan"
        )

        # ---------------------------------------------------------
        # PHASE 3: Strict Topographic Funnel & Filter
        # ---------------------------------------------------------
        print(f"\n{Colors.BOLD}--- Phase 3: The Strict Topographic Funnel ---{Colors.ENDC}")
        unique_pre_filter = the_crusher(high_acc_isomers, system_temperature)
        
        strict_isomers = []
        for idx, iso in enumerate(unique_pre_filter):
            iso_path = os.path.join(work_dir, f"strict_{idx}.xyz")
            inp_name = f"{base_name}_strict_{idx}.inp"
            write(iso_path, iso)
            
            with open(os.path.join(work_dir, inp_name), "w") as f:
                f.write(f"! xTB2 ExtremeOpt\n")
                f.write(f"%pal nprocs {cores_per_job} end\n")
                f.write(f"* xyzfile {charge} {mult} {os.path.basename(iso_path)}\n")
                
            out = run_orca_safely(f"Strict Filter Iso{idx}", inp_name, work_dir, sys_config, exit_on_fail=False)
            if out:
                try:
                    strict_iso = read(os.path.join(work_dir, f"{base_name}_strict_{idx}_ExtremeOpt.xyz"))
                    strict_iso.info['energy'] = parse_energy(out)
                    if 'v1_freq' in iso.info: strict_iso.info['v1_freq'] = iso.info['v1_freq']
                    strict_isomers.append(strict_iso)
                except: pass

        final_unique = the_crusher(strict_isomers, system_temperature)

        # ---------------------------------------------------------
        # FINAL EXPORT PROTOCOL
        # ---------------------------------------------------------
        xyz_path, csv_path = export_thermo(
            final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage2_2_HighAcc"
        )
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 2.2 COMPLETE: High-Accuracy Ab Initio Refinement Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.

 ChemComp_OPI-ORCA_v3-1 - STAGE 2.2: HIGH-ACCURACY REFINE 
➡️ Stage 2.1 ensemble missing. Bypassing Isomer Search & utilizing Initial File for CycBuCl_Ax.

--- Processing Ensemble: CycBuCl_Ax ---
➡️ Resource Matrix: 1 concurrent jobs @ 12 cores/job
➡️ Phase 1: MolSym Symmetrization & MACE-OFF23 Pre-Optimization...

--- GOAT Exploration Method Selector ---
 [1] External g-xTB (ExtOpt) | Est. Time: ~15s    | Benefit: Robust semi-empirical baseline.
 [2] MACE-OFF23 (ExtOpt)     | Est. Time: ~1m     | Benefit: Near-DFT accuracy.
✅ Selected MACE-OFF23 via ExtOpt for GOAT.
➡️ Igniting ORCA Subprocess: MACE PreOpt Iso0 ...
❌ ORCA crashed during MACE PreOpt Iso0.

--- Phase 2: Quantum Hardware Routing (PySCF_GPU) ---
➡️ Executing ωB97X-D4/def2-TZVP on PySCF (GPU Accelerated)...

--- Phase 3: The Strict Topographic Funnel ---

--- Topological Crusher Protocol ---
The Crusher deduplicates isomers based on spatial and energe

IndexError: list index out of range

👁️ Stage 2.3: Final Visual Triage

Purpose: Final 3D UI verification to ensure high-accuracy DFT did not artificially collapse distinct monomer geometries.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 02_3-VISUAL-TRIAGE-HIGHACC-AA (Stage 2.3)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# High-accuracy DFT and symmetry enforcement (from Stage 2.2) can cause previously 
# distinct topological isomers to collapse into the same energetic minimum. 
# This stage acts as the final macroscopic gatekeeper. It loads the high-accuracy 
# ensembles, runs them through the Grouping Crusher, and launches the 3D Visual 
# Triage Board so the user can verify the final deduplication before moving to 
# advanced structural fitting or VPT2.
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment after Stage 2.2 has finished.
# Select your molecule from the dropdown menu and click "Launch Triage". Use the 
# 3D viewers to verify the structures. Click "Accept Selections" to instantly 
# generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI. Upon acceptance, successfully writes 
# to the 'Isomer_xyz_Stage2_3_Verified' folder using the Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage2_2_HighAcc"
OUTPUT_DIR = "Isomer_xyz_Stage2_3_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class HighAccVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine lifecycle for High-Acc data.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage2_3_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Final High-Accuracy Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_high_acc_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 2.3: FINAL VISUAL TRIAGE {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found. Run Stage 2.2.", "error")
        return

    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage2_2_HighAcc_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 2.2 High-Accuracy Ensembles found in {INPUT_DIR}.", "error")
        return

    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage2_2_HighAcc_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Molecule:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 2.2 High-Accuracy Ensemble to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            print_status(f"Loading high-accuracy coordinates for {base_name}...", "info")
            
            try:
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                print_status("Executing Grouping Crusher alignment on refined coordinates...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                HighAccVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_high_acc_triage()


🚀 Stage 2.6: High-Level Extrapolation

Purpose: Executes High-Level Coupled-Cluster (CCSD(T)) extrapolations on the refined monomer isomers.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 02_6-HIGH-LEVEL-EXTRAP-AA (Stage 2.6)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes the ultimate High-Level DFT/Coupled-Cluster extrapolations 
# on the verified isomers from Stage 2.5 by passing the execution directly 
# to the generalized Stage 1.3 CCSD(T) Engine loop.
# 
# 2. Use instructions:
# Execute this script after completing Stage 2.5. It utilizes Checkpoint/Restart 
# logic seamlessly via the engine.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates ultra-high accuracy geometries and VPT2 frequencies. Outputs 
# specialized Pickett files for SPCAT, and generates the final Stage 2.6 Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_ccsdt_opt_loop' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    parse_freq = globals()['parse_first_vibrational_freq']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    estimate_memory = globals()['estimate_orca_memory']
    engine_ccsdt_opt_loop = globals()['engine_ccsdt_opt_loop']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage2_5_Verified" 
OUTPUT_DIR = "Isomer_xyz_Stage2_6_HighLevel"

if not os.path.exists(INPUT_DIR):
    if os.path.exists("Isomer_xyz_Stage2_3_Verified"):
        INPUT_DIR = "Isomer_xyz_Stage2_3_Verified"
        print_status(f"Stage 2.5 input not found. Falling back to {INPUT_DIR}.", "warning")
    else:
        print_status(f"Input directory '{INPUT_DIR}' not found.", "error")
        sys.exit(1)

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# DYNAMIC EXECUTION PROTOCOLS
# ==============================================================================

def calculate_dynamic_cores(num_isomers, is_ccsd=False):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_per_job = 8000 if is_ccsd else 4000 
    
    max_jobs_by_mem = max(1, int((mem_gb * 1024) / mem_per_job))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

def determine_extrapolation_level(base_size, num_isomers):
    size_map = {"Small": 0, "Medium": 1, "Large": 2}
    inv_map = {0: "Small", 1: "Medium", 2: "Large"}
    
    base_idx = size_map.get(base_size, 0)
    bump = num_isomers // 5
    final_idx = min(2, base_idx + bump)
    final_size = inv_map[final_idx]
    
    if final_size == "Small":
        method = "CCSD(T)"
        basis = "aug-cc-pwCVQZ"
        header = f"! {method} {basis} TightOPT AutoAux ExtremeSCF DEFGRID3 UseSym NumFreq VPT2\n"
        header += "%method\n  Z_Tol 1e-14;\n  FrozenCore FC_NONE;\nend\n"
        header += "%geom\n  TolE 1e-9;\n  TolRMSG 3e-6;\n  TolMaxG 1e-5;\nend\n"
    elif final_size == "Medium":
        method = "DLPNO-CCSD(T)"
        basis = "aug-cc-pVTZ"
        header = f"! {method} TightPNO {basis} TightOPT AutoAux ExtremeSCF DEFGRID3 UseSym NumFreq VPT2\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
        header += "%geom\n  TolE 1e-9;\n  TolRMSG 3e-6;\n  TolMaxG 1e-5;\nend\n"
    else:
        method = "DLPNO-CCSD(T)"
        basis = "cc-pVTZ"
        header = f"! {method} NormalPNO {basis} OPT AutoAux TightSCF DEFGRID2 UseSym NumFreq VPT2\n"
        
    return final_size, method, basis, header

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 2.6: HIGH-LEVEL EXTRAP{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        original_size = item.get("size", "Small")
        
        ensemble_file = os.path.join(INPUT_DIR, f"*_{base_name}_Final_Ensemble.xyz")
        found_files = glob.glob(ensemble_file)
        if not found_files: continue
        ensemble_file = found_files[0]
            
        print(f"\n{Colors.BOLD}--- Extrapolating Ensemble: {base_name} ---{Colors.ENDC}")
        isomers = read(ensemble_file, index=':')
        num_isomers = len(isomers)
        
        work_dir = f"Stage2_6_HighLevel_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        final_size, method, basis, block_header = determine_extrapolation_level(original_size, num_isomers)
        print_status(f"Isomer Count Scaling: {num_isomers} Isomers -> Shifted size from {original_size} to {Colors.WARNING}{final_size}{Colors.ENDC}", "info")
        print_status(f"Assigned Method: {Colors.OKGREEN}{method} / {basis}{Colors.ENDC}", "info")
        
        is_ccsd = "DLPNO" not in method
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers, is_ccsd)
        
        try:
            maxcore = estimate_memory(isomers[0], method=method, basis_set=basis)
        except Exception:
            maxcore = 3000 
            
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job ({maxcore} MB maxcore)", "info")
        
        # Execute Engine Protocol loop
        high_level_isomers = engine_ccsdt_opt_loop(
            isomers=isomers,
            base_name=base_name,
            work_dir=work_dir,
            block_header=block_header,
            maxcore=maxcore,
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage2_6",
            job_prefix_mod="extrap",
            requires_freq=True
        )

        if high_level_isomers:
            final_unique = the_crusher(high_level_isomers, system_temperature, silent=True)
            xyz_path, csv_path = export_thermo(
                final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage2_6_HighLevel"
            )
        else:
            print_status(f"All calculations failed for {base_name}. No inventory exported.", "error")
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 2.6 COMPLETE: High-Level Extrapolation Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 2.7: Extrapolated Visual Triage

Purpose: Final visual verification of the ultimate CCSD(T) monomer global minima.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 02_7-VISUAL-TRIAGE-EXTRAP-AA (Stage 2.7)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# Following the ultimate High-Level Extrapolations (CCSD(T) / DLPNO) in Stage 2.6,
# flat potential energy surfaces might collapse previously distinct geometries.
# This stage loads the extrapolated ensembles, groups them via the grouping 
# crusher, and launches the final 3D Visual Triage Board. 
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment after Stage 2.6 has finished.
# Select your molecule from the dropdown menu and click "Launch Triage". Use the 
# 3D viewers to verify the structures. Click "Accept Selections" to instantly 
# generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI. Upon acceptance, successfully writes 
# to the 'Isomer_xyz_Stage2_7_Verified' folder using the Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage2_6_HighLevel"
OUTPUT_DIR = "Isomer_xyz_Stage2_7_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class ExtrapVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for Extrapolated data.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage2_7_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Ultimate Extrapolated Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_extrap_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 2.7: EXTRAPOLATED TRIAGE {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found. Run Stage 2.6.", "error")
        return

    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage2_6_HighLevel_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 2.6 Extrapolated Ensembles found in {INPUT_DIR}.", "error")
        return

    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage2_6_HighLevel_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Molecule:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 2.6 Extrapolated Ensemble to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            print_status(f"Loading extrapolated coordinates for {base_name}...", "info")
            
            try:
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                print_status("Executing Grouping Crusher alignment on extrapolated coordinates...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                ExtrapVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_extrap_triage()


🔗 Stage 3.1: Strong Interaction Complex

Purpose: Assembles distinct fragments and maps Strong Interaction conformational space via Frozen Escape Room and GOAT.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 03_1-STRONG-INT-AA (Stage 3.0)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage specifically explores the vast intermolecular conformational space 
# of Strong Interaction Complexes. It isolates the high-accuracy fragments from 
# Stage 2.7, cross-combines every unique monomer conformation into an assembly, 
# and runs them through a specialized 9-step GOAT exploration loop.
# 
# 2. Use instructions:
# Execute this script after completing Stage 2.7. It relies on the Stage 1.3 
# Script Engine loaded in your active Jupyter memory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Automatically bypasses single molecules. Extracts fragments, builds 
# combinatorial assemblies, runs the Frozen Escape Room (rigid-body perturbations), 
# explores via GOAT, and exports a unified Strong Interaction Thermodynamic Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import shutil
import numpy as np
import itertools
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'run_orca_subprocess_safely' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    parse_freq = globals()['parse_first_vibrational_freq']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    get_goat_header = globals()['get_goat_header']
    extract_ensemble = globals()['extract_goat_ensemble']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
    from ase import Atoms
    from ase.neighborlist import natural_cutoffs, NeighborList
    import networkx as nx
except ImportError:
    print_status("Required libraries (ASE, NetworkX) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage2_7_Verified"
OUTPUT_DIR = "Isomer_xyz_Stage3_1_StrongInt"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# FRAGMENTATION & FROZEN ESCAPE ROOM PROTOCOLS
# ==============================================================================

def get_fragments_from_atoms(atoms):
    """Splits an ASE Atoms object into a list of isolated fragment Atoms objects."""
    cutoffs = [c * 1.2 for c in natural_cutoffs(atoms)]
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)
    matrix = nl.get_connectivity_matrix()
    graph = nx.from_scipy_sparse_array(matrix)
    
    fragments = []
    for component in nx.connected_components(graph):
        indices = list(component)
        fragments.append(atoms[indices])
    return fragments

def rigid_rattle_complex(atoms, translation_stdev=2.0, rotation_stdev=180.0):
    """
    Applies rigid-body perturbations. Freezes the internal geometries of the 
    fragments, only altering the intermolecular distance, angle, and dihedral.
    """
    frags = get_fragments_from_atoms(atoms)
    if len(frags) < 2: return atoms.copy()
    
    new_atoms = Atoms()
    new_atoms += frags[0] # Keep the heaviest/first fragment fixed
    
    for frag in frags[1:]:
        f = frag.copy()
        
        # Random Translation Vector
        vec = np.random.randn(3)
        vec /= np.linalg.norm(vec)
        vec *= np.random.uniform(0.5, translation_stdev)
        
        # Random Rotation Vector
        axis = np.random.randn(3)
        axis /= np.linalg.norm(axis)
        angle = np.random.uniform(-rotation_stdev, rotation_stdev)
        
        f.rotate(angle, axis, center='COM')
        f.translate(vec)
        new_atoms += f
        
    return new_atoms

def run_frozen_escape_room(base_name, seed_atoms, work_dir, sys_config, cores, charge, mult):
    """Executes the Escape Room exclusively using rigid-body frozen fragment perturbations."""
    print_status(f"Triggering FROZEN ESCAPE ROOM for {base_name}...", "info")
    escaped_isomers = []
    
    # 1. "Thermal Shock" Equivalent (Violent Rigid Perturbation)
    shocked_atoms = rigid_rattle_complex(seed_atoms, translation_stdev=3.0, rotation_stdev=180.0)
    shock_opt_name = f"{base_name}_Escape_Shock_OPT"
    write(os.path.join(work_dir, f"{shock_opt_name}.xyz"), shocked_atoms)
    
    with open(os.path.join(work_dir, f"{shock_opt_name}.inp"), 'w') as f:
        f.write(f"! xTB2 OPT\n%pal nprocs {cores} end\n")
        f.write(f"* xyzfile {charge} {mult} {shock_opt_name}.xyz\n")
    
    out_shock = run_orca_safely("Frozen Escape: Quench Shock", f"{shock_opt_name}.inp", work_dir, sys_config, exit_on_fail=False)
    if out_shock:
        try:
            opt_shock = read(os.path.join(work_dir, f"{shock_opt_name}.xyz"))
            opt_shock.info['energy'] = parse_energy(out_shock)
            escaped_isomers.append(opt_shock)
        except: pass

    # 2. "Stochastic Rattle" Equivalent (Mild Rigid Perturbation)
    rattle_atoms = rigid_rattle_complex(seed_atoms, translation_stdev=1.0, rotation_stdev=45.0)
    rattle_name = f"{base_name}_Escape_Rattle_OPT"
    write(os.path.join(work_dir, f"{rattle_name}.xyz"), rattle_atoms)
    
    with open(os.path.join(work_dir, f"{rattle_name}.inp"), 'w') as f:
        f.write(f"! xTB2 OPT\n%pal nprocs {cores} end\n")
        f.write(f"* xyzfile {charge} {mult} {rattle_name}.xyz\n")
    
    out_rattle = run_orca_safely("Frozen Escape: Quench Rattle", f"{rattle_name}.inp", work_dir, sys_config, exit_on_fail=False)
    if out_rattle:
        try:
            opt_rattle = read(os.path.join(work_dir, f"{rattle_name}.xyz"))
            opt_rattle.info['energy'] = parse_energy(out_rattle)
            escaped_isomers.append(opt_rattle)
        except: pass
        
    return escaped_isomers

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================

def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 3.1: STRONG INTERACTIONS {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found. Run Stage 2.7.", "error")
        sys.exit(1)
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)
    cores = min(16, sys_config.get("hardware", {}).get("cpu_threads", 4))

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        num_frags = item.get("fragments_count", 1)
        topology_type = item.get("type", "")
        mace_applicable = item.get("mace_applicable", True)
        
        print(f"\n{Colors.BOLD}--- Evaluating Topology: {base_name} ---{Colors.ENDC}")
        
        # 1. Intelligent Stage Bypass
        if num_frags == 1:
            print_status(f"Bypass Triggered: {base_name} is a single molecule. Skipping Stage 3.1.", "warning")
            continue
        if "Weak Interaction" in topology_type:
            print_status(f"Bypass Triggered: {base_name} is classified as purely Weak Interaction. Skipping Stage 3.1.", "warning")
            continue
            
        print_status(f"Strong Interaction Complex detected ({num_frags} fragments). Executing Stage 3.1 Assembly...", "success")
        
        ensemble_file = os.path.join(INPUT_DIR, f"Stage2_7_Verified_{base_name}_Final_Ensemble.xyz")
        if not os.path.exists(ensemble_file):
            print_status(f"Missing Stage 2.7 ensemble for {base_name}. Skipping.", "error")
            continue
            
        work_dir = f"Stage3_1_StrongInt_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
        
        # 2. Extract and Deduplicate Monomer Fragments
        isomers = read(ensemble_file, index=':')
        frag_pools = [[] for _ in range(num_frags)]
        
        ref_frags = get_fragments_from_atoms(isomers[0])
        ref_coms = [f.get_center_of_mass() for f in ref_frags]
        
        for iso in isomers:
            frags = get_fragments_from_atoms(iso)
            if len(frags) == num_frags:
                for i, f in enumerate(frags):
                    frag_pools[i].append(f)
                    
        unique_frags = []
        for i, pool in enumerate(frag_pools):
            print_status(f"Sifting unique conformers for Fragment {i+1}...", "info")
            unique_frags.append(the_crusher(pool, system_temperature, silent=True))
            
        # 3. Combinatorial Assembly
        combinations = list(itertools.product(*unique_frags))
        print_status(f"Assembled {len(combinations)} total cross-combinations from unique fragments.", "success")
        
        combined_seeds = []
        combo_names = []
        for idx, combo in enumerate(combinations):
            new_mol = Atoms()
            for i, frag in enumerate(combo):
                f_copy = frag.copy()
                f_copy.translate(ref_coms[i] - f_copy.get_center_of_mass()) 
                new_mol += f_copy
            combined_seeds.append(new_mol)
            combo_names.append(f"F1I{idx}_F2I{idx}") 
            
        # 4. Pre-Optimization & Dihedral Bypass Check
        if len(combined_seeds[0]) < 4:
            print_status("Bypass Triggered: Combined complex has < 4 atoms (No intermolecular dihedrals). Skipping deep GOAT exploration.", "warning")
            continue
            
        # 5. The Frozen Escape Room & 9-Step GOAT Loop
        all_raw_complexes = []
        for c_idx, c_seed in enumerate(combined_seeds):
            c_name = f"{base_name}_Comb{c_idx}"
            
            # Initial Opt
            inp_opt = f"{c_name}_Init.inp"
            write(os.path.join(work_dir, f"{c_name}_Init.xyz"), c_seed)
            with open(os.path.join(work_dir, inp_opt), "w") as f:
                if mace_applicable:
                    f.write(get_goat_header(sys_config, c_seed).replace("GOAT", "OPT"))
                else:
                    f.write(f"! xTB2 OPT\n")
                f.write(f"%pal nprocs {cores} end\n")
                f.write(f"* xyzfile {charge} {mult} {c_name}_Init.xyz\n")
                
            out = run_orca_safely(f"Combine Opt {c_idx}", inp_opt, work_dir, sys_config, exit_on_fail=False)
            if out:
                try:
                    opt_c = read(os.path.join(work_dir, f"{c_name}_Init_OPT.xyz"))
                    opt_c.info['energy'] = parse_energy(out)
                    all_raw_complexes.append(opt_c)
                    
                    # Run Frozen Escape Room 
                    escaped = run_frozen_escape_room(c_name, opt_c, work_dir, sys_config, cores, charge, mult)
                    all_raw_complexes.extend(escaped)
                except: pass

        unique_seeds = the_crusher(all_raw_complexes, system_temperature)
        
        # Deep GOAT Explorations
        target_seed = unique_seeds[0] if unique_seeds else combined_seeds[0]
        write(os.path.join(work_dir, f"{base_name}_GOAT_Seed.xyz"), target_seed)
        
        goat_header = get_goat_header(sys_config, target_seed) if mace_applicable else "! GOAT XTB2\n"
        inp_goat = f"{base_name}_GOAT.inp"
        
        with open(os.path.join(work_dir, inp_goat), "w") as f:
            f.write(goat_header)
            f.write(f"%pal nprocs {cores} end\n")
            f.write("%goat\n  MAXITERMULT 6\n  MINGLOBALITER 30\n  MAXGLOBALITER 300\n  NWorkers auto\n  MAXEN 12.0\n  RMSD 1.5\n  ROTCONSTDIFF 0.1\nend\n")
            f.write(f"* xyzfile {charge} {mult} {base_name}_GOAT_Seed.xyz\n")
            
        run_orca_safely(f"GOAT Complex Exploration", inp_goat, work_dir, sys_config, exit_on_fail=False)
        all_raw_complexes.extend(extract_ensemble(work_dir, f"{base_name}_GOAT"))
        
        final_unique = the_crusher(all_raw_complexes, system_temperature)
        
        # 6. Final Frequencies & Pickett Generation
        freq_outputs = []
        print_status(f"Calculating Final Frequencies for {len(final_unique)} interacting complexes...", "info")
        for idx, iso in enumerate(final_unique):
            freq_name = f"{base_name}_Iso{idx+1}_FREQ"
            pickett_name = f"{base_name}_Iso{idx+1}_Stage3_1.pickett" 
            
            write(os.path.join(work_dir, f"{freq_name}.xyz"), iso)
            with open(os.path.join(work_dir, f"{freq_name}.inp"), 'w') as f:
                f.write(f"! xTB2 FREQ\n%pal nprocs {cores} end\n")
                f.write(f"%output\n  Pickettname \"{pickett_name}\"\nend\n")
                f.write(f"* xyzfile {charge} {mult} {freq_name}.xyz\n")
                
            out_path = run_orca_safely(f"FREQ Calculation (Iso {idx+1})", f"{freq_name}.inp", work_dir, sys_config, exit_on_fail=False)
            freq_outputs.append(out_path)

        for iso, out_path in zip(final_unique, freq_outputs):
            if out_path: iso.info['v1_freq'] = parse_freq(out_path)

        xyz_path, csv_path = export_thermo(
            final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage3_1_StrongInt"
        )

    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 3.1 COMPLETE: Strong Interaction Conformational Mapping Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 3.2: Strong Interaction Triage

Purpose: Interactive 3D UI to verify the topological deduplication of assembled Strong Complexes.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 03_2-VISUAL-TRIAGE-STRONGINT-AA (Stage 3.1)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage provides the visual gatekeeper for Strong Interaction Complexes 
# generated in Stage 3.1. Since combinatorial fragment assembly and frozen-body 
# perturbations can generate massive geometric diversity, the user must visually 
# verify the topological groups.
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment after Stage 3.1 has finished.
# If your system was bypassed (single molecule, weak-interaction only, or < 4 atoms), 
# this stage will politely notify you and do nothing. Otherwise, select the complex 
# and click "Launch Triage". Click "Accept Selections" to export the inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI for complexes. Upon acceptance, 
# successfully writes to the 'Isomer_xyz_Stage3_2_Verified' folder.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage3_1_StrongInt"
OUTPUT_DIR = "Isomer_xyz_Stage3_2_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class StrongIntVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for Assembled Complexes.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage3_2_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Strong Interaction Complex Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_strong_int_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 3.2: STRONG INT. TRIAGE {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 3.1 (Single fragment, weak interaction, or < 4 atoms).", "info")
        return

    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage3_1_StrongInt_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 3.1 Assembled Ensembles found in {INPUT_DIR}.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 3.1.", "info")
        return

    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage3_1_StrongInt_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Complex:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 3.1 Assembled Complex to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            print_status(f"Loading assembled coordinates for {base_name}...", "info")
            
            try:
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                print_status("Executing Grouping Crusher alignment on complex geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                StrongIntVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_strong_int_triage()


🎯 Stage 3.4: High-Accuracy Complex Refinement

Purpose: Executes high-accuracy DFT refinement on Strong Complexes while strictly enforcing rigid-body fragment constraints.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 03_4-HIGH-ACC-COMPLEX-AA (Stage 3.2)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage bridges the gap between topological discovery and high-accuracy DFT 
# specifically for Strong Interaction Complexes. It delegats loop execution directly
# to the Stage 1.3 engine while passing overriding parameters specific to complexes.
# 
# 2. Use instructions:
# Execute this script after completing Stage 3.2 in VS Code. 
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates highly refined ab initio complex geometries, outputs Pickett 
# files for SPCAT, and exports a fully authenticated Thermodynamic Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_high_acc_opt_loop' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    parse_freq = globals()['parse_first_vibrational_freq']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    get_goat_header = globals()['get_goat_header']
    apply_molsym_preopt = globals()['apply_molsym_preopt']
    engine_high_acc_opt_loop = globals()['engine_high_acc_opt_loop']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage3_2_Verified"
OUTPUT_DIR = "Isomer_xyz_Stage3_4_HighAccComplex"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

def calculate_dynamic_cores(num_isomers):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_mb = mem_gb * 1024
    
    max_jobs_by_mem = max(1, int(mem_mb / 500))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 3.4: HIGH-ACC COMPLEX REF {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 3.2.", "info")
        sys.exit(0)
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        routing_flag = item.get("pyscf_v_orca", "ORCA_CPU")
        mace_applicable = item.get("mace_applicable", True)
        num_frags = item.get("fragments_count", 1)
        
        # 1. Complex Bypass Check
        if num_frags == 1:
            print_status(f"Bypass Triggered: {base_name} is a single fragment. Skipping Stage 3.4.", "warning")
            continue
            
        ensemble_file = os.path.join(INPUT_DIR, f"Stage3_2_Verified_{base_name}_Final_Ensemble.xyz")
        if not os.path.exists(ensemble_file): 
            print_status(f"Missing Stage 3.2 ensemble for {base_name}. Skipping.", "error")
            continue
            
        isomers = read(ensemble_file, index=':')
        num_isomers = len(isomers)
        
        if len(isomers[0]) < 4:
            print_status(f"Bypass Triggered: {base_name} complex lacks enough atoms (<4) for an intermolecular dihedral. Skipping.", "warning")
            continue
            
        print(f"\n{Colors.BOLD}--- Processing Complex Ensemble: {base_name} ---{Colors.ENDC}")
        work_dir = f"Stage3_4_HighAccComplex_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers)
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job", "info")
        
        # ---------------------------------------------------------
        # PHASE 1: MolSym -> MACE Pre-Optimization -> MolSym
        # ---------------------------------------------------------
        print_status("Phase 1: Double-Symmetrization & MACE-OFF23 Pre-Optimization...", "info")
        preopt_isomers = []
        for idx, iso in enumerate(isomers):
            sym_iso_1 = apply_molsym_preopt(iso, base_name, idx, work_dir, prefix="molsym1")
            
            if mace_applicable:
                iso_path = os.path.join(work_dir, f"{base_name}_mace_comp_{idx}.xyz")
                write(iso_path, sym_iso_1)
                
                mace_inp = f"{base_name}_mace_comp_{idx}.inp"
                mace_hdr = get_goat_header(sys_config, sym_iso_1)
                with open(os.path.join(work_dir, mace_inp), "w") as f:
                    f.write(mace_hdr.replace("! ExtOpt GOAT", "! ExtOpt OPT")) 
                    f.write(f"%pal nprocs {cores_per_job} end\n")
                    f.write(f"* xyzfile {charge} {mult} {os.path.basename(iso_path)}\n")
                    
                out = run_orca_safely(f"MACE PreOpt Complex {idx}", mace_inp, work_dir, sys_config, exit_on_fail=False)
                if out:
                    try:
                        sym_iso_1 = read(os.path.join(work_dir, f"{base_name}_mace_comp_{idx}_OPT.xyz"))
                    except: pass
            
            # Step 3: Final MolSym cleanup before DFT
            sym_iso_final = apply_molsym_preopt(sym_iso_1, base_name, idx, work_dir, prefix="molsym2")
            preopt_isomers.append(sym_iso_final)

        # ---------------------------------------------------------
        # PHASE 2: Hardware Routed High-Accuracy DFT (Engine execution)
        # ---------------------------------------------------------
        print(f"\n{Colors.BOLD}--- Phase 2: Quantum Hardware Routing ({routing_flag}) ---{Colors.ENDC}")
        
        high_acc_isomers = engine_high_acc_opt_loop(
            preopt_isomers=preopt_isomers,
            base_name=base_name,
            work_dir=work_dir,
            routing_flag=routing_flag,
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage3_4",
            job_prefix_mod="comp_r2scan"
        )

        # ---------------------------------------------------------
        # PHASE 3: Strict Topographic Funnel & Filter
        # ---------------------------------------------------------
        print(f"\n{Colors.BOLD}--- Phase 3: The Strict Topographic Funnel ---{Colors.ENDC}")
        unique_pre_filter = the_crusher(high_acc_isomers, system_temperature)
        
        strict_isomers = []
        for idx, iso in enumerate(unique_pre_filter):
            iso_path = os.path.join(work_dir, f"strict_comp_{idx}.xyz")
            inp_name = f"{base_name}_strict_comp_{idx}.inp"
            write(iso_path, iso)
            
            with open(os.path.join(work_dir, inp_name), "w") as f:
                f.write(f"! xTB2 ExtremeOpt\n") # Blast out shallow transition states
                f.write(f"%pal nprocs {cores_per_job} end\n")
                f.write(f"* xyzfile {charge} {mult} {os.path.basename(iso_path)}\n")
                
            out = run_orca_safely(f"Strict Filter Complex {idx}", inp_name, work_dir, sys_config, exit_on_fail=False)
            if out:
                try:
                    strict_iso = read(os.path.join(work_dir, f"{base_name}_strict_comp_{idx}_ExtremeOpt.xyz"))
                    strict_iso.info['energy'] = parse_energy(out)
                    if 'v1_freq' in iso.info: strict_iso.info['v1_freq'] = iso.info['v1_freq']
                    strict_isomers.append(strict_iso)
                except: pass

        final_unique = the_crusher(strict_isomers, system_temperature)

        # ---------------------------------------------------------
        # FINAL EXPORT PROTOCOL
        # ---------------------------------------------------------
        xyz_path, csv_path = export_thermo(
            final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage3_4_HighAccComplex"
        )
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 3.4 COMPLETE: High-Accuracy Complex Refinement Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


🚀 Stage 3.5: High-Level Complex Extrap

Purpose: Executes Double-Hybrid (revDSD-PBEP86-D4) extrapolations on Strong Complexes.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 03_5-HIGH-LEVEL-COMPLEX-EXTRAP-AA (Stage 3.5)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes the ultimate High-Level Double-Hybrid Extrapolations 
# (revDSD-PBEP86-D4) on the verified strong interaction complexes from Stage 3.4. 
# It delegates looping natively to the Stage 1.3 Double Hybrid optimization loop.
# 
# 2. Use instructions:
# Execute this script after completing Stage 3.4. It utilizes Checkpoint/Restart 
# logic, so if the kernel dies, re-run this cell to resume optimizations.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates ultra-high accuracy intermolecular geometries. Outputs 
# specialized Pickett files and generates the final Stage 3.5 Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_revdsd_opt_loop' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    estimate_memory = globals()['estimate_orca_memory']
    engine_revdsd_opt_loop = globals()['engine_revdsd_opt_loop']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage3_4_HighAccComplex" 
OUTPUT_DIR = "Isomer_xyz_Stage3_5_HighLevelComplex"

if not os.path.exists(INPUT_DIR):
    print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
    print_status("Bypass Logic: Your system likely did not qualify for Stage 3.4.", "info")
    sys.exit(0)

if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
    
with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# DYNAMIC EXECUTION PROTOCOLS
# ==============================================================================

def calculate_dynamic_cores(num_isomers):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_per_job = 6000 
    
    max_jobs_by_mem = max(1, int((mem_gb * 1024) / mem_per_job))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

def determine_complex_extrapolation_level(base_size, num_isomers):
    size_map = {"Small": 0, "Medium": 1, "Large": 2}
    inv_map = {0: "Small", 1: "Medium", 2: "Large"}
    
    base_idx = size_map.get(base_size, 0)
    bump = num_isomers // 5
    final_idx = min(2, base_idx + bump)
    final_size = inv_map[final_idx]
    
    if final_size == "Small":
        method = "revDSD-PBEP86-D4"
        basis = "aug-cc-pVTZ"
        header = f"! {method} {basis} aug-cc-pVTZ/JK aug-cc-pVTZ/C RIJCOSX ExtremeSCF TightOPT DEFGRID3 OPT\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
    elif final_size == "Medium":
        method = "revDSD-PBEP86-D4"
        basis = "cc-pVTZ"
        header = f"! {method} {basis} cc-pVTZ/JK cc-pVTZ/C RIJCOSX TightSCF OPT DEFGRID2\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
    else:
        method = "revDSD-PBEP86-D4"
        basis = "def2-TZVP"
        header = f"! {method} {basis} def2/J def2-TZVP/C RIJCOSX NormalSCF OPT DEFGRID2\n"
        
    return final_size, method, basis, header

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 3.5: COMPLEX EXTRAPOLATION{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        original_size = item.get("size", "Small")
        num_frags = item.get("fragments_count", 1)
        
        if num_frags == 1:
            print_status(f"Bypass Triggered: {base_name} is a single fragment. Skipping Stage 3.5.", "warning")
            continue
        
        ensemble_file = os.path.join(INPUT_DIR, f"*_{base_name}_Final_Ensemble.xyz")
        found_files = glob.glob(ensemble_file)
        if not found_files: continue
        ensemble_file = found_files[0]
            
        isomers = read(ensemble_file, index=':')
        num_isomers = len(isomers)
        
        if len(isomers[0]) < 4:
            print_status(f"Bypass Triggered: {base_name} complex lacks enough atoms (<4) for an intermolecular dihedral. Skipping.", "warning")
            continue
            
        print(f"\n{Colors.BOLD}--- Extrapolating Complex Ensemble: {base_name} ---{Colors.ENDC}")
        work_dir = f"Stage3_5_HighLevelComplex_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        final_size, method, basis, block_header = determine_complex_extrapolation_level(original_size, num_isomers)
        print_status(f"Isomer Count Scaling: {num_isomers} Isomers -> Shifted size from {original_size} to {Colors.WARNING}{final_size}{Colors.ENDC}", "info")
        print_status(f"Assigned Double Hybrid Method: {Colors.OKGREEN}{method} / {basis}{Colors.ENDC}", "info")
        
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers)
        
        try:
            maxcore = estimate_memory(isomers[0], method=method, basis_set=basis)
        except Exception:
            maxcore = 3000
            
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job ({maxcore} MB maxcore)", "info")
        
        # Execute Engine Protocol Loop
        high_level_isomers = engine_revdsd_opt_loop(
            isomers=isomers,
            base_name=base_name,
            work_dir=work_dir,
            block_header=block_header,
            maxcore=maxcore,
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage3_5",
            job_prefix_mod="comp_extrap"
        )

        if high_level_isomers:
            final_unique = the_crusher(high_level_isomers, system_temperature, silent=True)
            xyz_path, csv_path = export_thermo(
                final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage3_5_HighLevelComplex"
            )
        else:
            print_status(f"All calculations failed for {base_name}. No inventory exported.", "error")
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 3.5 COMPLETE: High-Level Complex Extrapolation Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 3.6: Complex Extrap Triage

Purpose: Visual UI verification of the Double-Hybrid extrapolated Strong Complexes.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 03_6-VISUAL-TRIAGE-COMPLEX-EXTRAP-AA (Stage 3.6)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# Following the ultimate High-Level Extrapolations (revDSD-PBEP86-D4) for Strong 
# Interaction Complexes in Stage 3.5, flat intermolecular potential energy surfaces 
# might collapse previously distinct geometries. This stage loads the extrapolated 
# complex ensembles, groups them via the grouping crusher, and launches the final 
# 3D Visual Triage Board. 
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment after Stage 3.5 has finished.
# If your system was bypassed (single molecule or < 4 atoms), this stage will 
# politely notify you and do nothing. Select your complex from the dropdown menu 
# and click "Launch Triage". Verify the 3D structures and click "Accept Selections" 
# to instantly generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI. Upon acceptance, successfully writes 
# to the 'Isomer_xyz_Stage3_6_Verified' folder using the Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
# Actively read the Stage 1.3 UI and Stage 1.2 Math components directly from memory
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage3_5_HighLevelComplex"
OUTPUT_DIR = "Isomer_xyz_Stage3_6_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class ComplexExtrapVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for High-Level 
    Extrapolated Complexes.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        # Initialize the parent UI
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        # Trigger the parent class acceptance (clears screen, prints fragment counts)
        super().on_accept(btn)
        
        # Extract the finalized lowest-energy representatives from the user's groups
        final_reps = [g[0] for g in self.groups]
        
        # Automatically trigger the Inventory Export Protocol
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage3_6_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Ultimate Complex Extrapolation Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_complex_extrap_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 3.6: COMPLEX EXTRAP TRIAGE{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 3.5.", "info")
        return

    # Specifically search for the output generated by Stage 3.5 High-Level Complex Extrapolation
    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage3_5_HighLevelComplex_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 3.5 Extrapolated Ensembles found in {INPUT_DIR}.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 3.5 (Single fragment or < 4 atoms).", "info")
        return

    # Extract base names for the dropdown menu
    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage3_5_HighLevelComplex_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Complex:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(
        description='Launch Triage',
        button_style='info',
        icon='eye'
    )
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 3.5 Extrapolated Complex to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            
            # Additional safety check against user config flags
            ingested_files = user_config.get("ingested_files", [])
            for item in ingested_files:
                if item.get("file", "").replace('.xyz', '') == base_name:
                    if item.get("fragments_count", 1) == 1:
                        print_status(f"Bypass Enforced: {base_name} is a single fragment.", "warning")
                        return

            print_status(f"Loading extrapolated complex coordinates for {base_name}...", "info")
            
            try:
                # Read all isomers from the Stage 3.5 ensemble
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                if len(isomers[0]) < 4:
                    print_status(f"Bypass Enforced: {base_name} lacks enough atoms (<4) for intermolecular dihedral evaluation.", "warning")
                    return
                
                # Pass them through the grouping crusher to array them for the UI
                print_status("Executing Grouping Crusher alignment on highly refined geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                # Instantiate the dynamically subclassed UI
                ComplexExtrapVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_complex_extrap_triage()


👑 Stage 3.7: Ultimate Complex Extrap

Purpose: Executes strictly optimized CCSD(T) extrapolations on Strong Complexes to secure the global minimum.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 03_7-ULTIMATE-COMPLEX-EXTRAP-AA (Stage 3.7)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes the ultimate High-Level Coupled-Cluster Extrapolations 
# (CCSD(T) / DLPNO-CCSD(T)) on the verified strong interaction complexes from 
# Stage 3.6 by passing execution to the generalized Stage 1.3 CCSD(T) Engine loop.
# It strictly runs Optimizations (No frequencies) to secure the absolute global minimum.
# 
# 2. Use instructions:
# Execute this script after completing Stage 3.6. It utilizes Checkpoint/Restart 
# logic natively via the engine.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates absolute "Gold Standard" theoretical complex geometries. 
# Outputs specialized Pickett files, and generates the final Stage 3.7 Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_ccsdt_opt_loop' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    estimate_memory = globals()['estimate_orca_memory']
    engine_ccsdt_opt_loop = globals()['engine_ccsdt_opt_loop']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage3_6_Verified" 
OUTPUT_DIR = "Isomer_xyz_Stage3_7_UltimateComplex"

if not os.path.exists(INPUT_DIR):
    print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
    print_status("Bypass Logic: Your system likely did not qualify for Stage 3.6.", "info")
    sys.exit(0)

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# DYNAMIC EXECUTION PROTOCOLS
# ==============================================================================

def calculate_dynamic_cores(num_isomers, is_ccsd=False):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_per_job = 8000 if is_ccsd else 4000 
    
    max_jobs_by_mem = max(1, int((mem_gb * 1024) / mem_per_job))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

def determine_ultimate_extrapolation_level(base_size, num_isomers):
    size_map = {"Small": 0, "Medium": 1, "Large": 2}
    inv_map = {0: "Small", 1: "Medium", 2: "Large"}
    
    base_idx = size_map.get(base_size, 0)
    bump = num_isomers // 5
    final_idx = min(2, base_idx + bump)
    final_size = inv_map[final_idx]
    
    if final_size == "Small":
        method = "CCSD(T)"
        basis = "aug-cc-pwCVQZ"
        header = f"! {method} {basis} TightOPT AutoAux ExtremeSCF DEFGRID3 UseSym\n"
        header += "%method\n  Z_Tol 1e-14;\n  FrozenCore FC_NONE;\nend\n"
        header += "%geom\n  TolE 1e-9;\n  TolRMSG 3e-6;\n  TolMaxG 1e-5;\nend\n"
    elif final_size == "Medium":
        method = "DLPNO-CCSD(T)"
        basis = "aug-cc-pVTZ"
        header = f"! {method} TightPNO {basis} TightOPT AutoAux ExtremeSCF DEFGRID3 UseSym\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
        header += "%geom\n  TolE 1e-9;\n  TolRMSG 3e-6;\n  TolMaxG 1e-5;\nend\n"
    else:
        method = "DLPNO-CCSD(T)"
        basis = "cc-pVTZ"
        header = f"! {method} NormalPNO {basis} OPT AutoAux TightSCF DEFGRID2 UseSym\n"
        
    return final_size, method, basis, header

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 3.7: ULTIMATE CC EXTRAP{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        original_size = item.get("size", "Small")
        num_frags = item.get("fragments_count", 1)
        
        if num_frags == 1:
            print_status(f"Bypass Triggered: {base_name} is a single fragment. Skipping Stage 3.7.", "warning")
            continue
        
        ensemble_file = os.path.join(INPUT_DIR, f"*_{base_name}_Final_Ensemble.xyz")
        found_files = glob.glob(ensemble_file)
        if not found_files: continue
        ensemble_file = found_files[0]
            
        isomers = read(ensemble_file, index=':')
        num_isomers = len(isomers)
        
        if len(isomers[0]) < 4:
            print_status(f"Bypass Triggered: {base_name} complex lacks enough atoms (<4) for an intermolecular dihedral. Skipping.", "warning")
            continue
            
        print(f"\n{Colors.BOLD}--- Extrapolating Complex Ensemble: {base_name} ---{Colors.ENDC}")
        work_dir = f"Stage3_7_UltimateComplex_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        final_size, method, basis, block_header = determine_ultimate_extrapolation_level(original_size, num_isomers)
        print_status(f"Isomer Count Scaling: {num_isomers} Isomers -> Shifted size from {original_size} to {Colors.WARNING}{final_size}{Colors.ENDC}", "info")
        print_status(f"Assigned Coupled-Cluster Method: {Colors.OKGREEN}{method} / {basis}{Colors.ENDC}", "info")
        
        is_ccsd = "DLPNO" not in method
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers, is_ccsd)
        
        try:
            maxcore = estimate_memory(isomers[0], method=method, basis_set=basis)
        except Exception:
            maxcore = 3000
            
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job ({maxcore} MB maxcore)", "info")
        
        # Execute Engine Protocol Loop (Passing requires_freq=False for strict Opt structure)
        high_level_isomers = engine_ccsdt_opt_loop(
            isomers=isomers,
            base_name=base_name,
            work_dir=work_dir,
            block_header=block_header,
            maxcore=maxcore,
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage3_7",
            job_prefix_mod="ccsd_extrap",
            requires_freq=False
        )

        if high_level_isomers:
            final_unique = the_crusher(high_level_isomers, system_temperature, silent=True)
            xyz_path, csv_path = export_thermo(
                final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage3_7_UltimateComplex"
            )
        else:
            print_status(f"All calculations failed for {base_name}. No inventory exported.", "error")
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 3.7 COMPLETE: Ultimate Coupled-Cluster Extrapolation Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 3.8: Ultimate Complex Triage

Purpose: Absolute final visual triage for "Gold Standard" CCSD(T) Strong Complexes.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 03_8-VISUAL-TRIAGE-ULTIMATE-COMPLEX-AA (Stage 3.8)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# Following the ultimate High-Level Coupled-Cluster Extrapolations (CCSD(T) / 
# DLPNO-CCSD(T)) for Strong Interaction Complexes in Stage 3.7, this stage 
# provides the absolute final visual verification. It loads the Coupled-Cluster 
# complex ensembles, groups them via the grouping crusher, and launches the final 
# 3D Visual Triage Board to lock in the "Gold Standard" global minima.
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment after Stage 3.7 has finished.
# If your system was bypassed (single molecule or < 4 atoms), this stage will 
# politely notify you and do nothing. Select your complex from the dropdown menu 
# and click "Launch Triage". Verify the 3D structures and click "Accept Selections" 
# to instantly generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI. Upon acceptance, successfully writes 
# to the 'Isomer_xyz_Stage3_8_Verified' folder using the Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
# Actively read the Stage 1.3 UI and Stage 1.2 Math components directly from memory
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage3_7_UltimateComplex"
OUTPUT_DIR = "Isomer_xyz_Stage3_8_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class UltimateComplexVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for Ultimate 
    Coupled-Cluster Complexes.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        # Initialize the parent UI
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        # Trigger the parent class acceptance (clears screen, prints fragment counts)
        super().on_accept(btn)
        
        # Extract the finalized lowest-energy representatives from the user's groups
        final_reps = [g[0] for g in self.groups]
        
        # Automatically trigger the Inventory Export Protocol
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage3_8_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Gold Standard Complex Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_ultimate_complex_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 3.8: ULTIMATE CC TRIAGE{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 3.7.", "info")
        return

    # Specifically search for the output generated by Stage 3.7
    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage3_7_UltimateComplex_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 3.7 Extrapolated Ensembles found in {INPUT_DIR}.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 3.7 (Single fragment or < 4 atoms).", "info")
        return

    # Extract base names for the dropdown menu
    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage3_7_UltimateComplex_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Complex:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(
        description='Launch Triage',
        button_style='info',
        icon='eye'
    )
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 3.7 Coupled-Cluster Complex to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            
            # Additional safety check against user config flags
            ingested_files = user_config.get("ingested_files", [])
            for item in ingested_files:
                if item.get("file", "").replace('.xyz', '') == base_name:
                    if item.get("fragments_count", 1) == 1:
                        print_status(f"Bypass Enforced: {base_name} is a single fragment.", "warning")
                        return

            print_status(f"Loading Gold Standard complex coordinates for {base_name}...", "info")
            
            try:
                # Read all isomers from the Stage 3.7 ensemble
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                if len(isomers[0]) < 4:
                    print_status(f"Bypass Enforced: {base_name} lacks enough atoms (<4) for intermolecular dihedral evaluation.", "warning")
                    return
                
                # Pass them through the grouping crusher to array them for the UI
                print_status("Executing Grouping Crusher alignment on highly refined geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                # Instantiate the dynamically subclassed UI
                UltimateComplexVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_ultimate_complex_triage()


☁️ Stage 4.1: Weak Interaction Complex

Purpose: Maps Weak Interaction Complex surfaces via rigid-body fragment assembly and GOAT-COURSE.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 04_1-WEAK-INT-AA (Stage 4.0)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage maps the vast, shallow potential energy surfaces of Weak Interaction 
# Complexes. It extracts the finalized, Gold-Standard CCSD(T) internal geometries 
# from Stage 2.7 (Monomers) and Stage 3.8 (Strong Complexes), combines them, and 
# executes a specialized RIGIDBODYOPT 9-step iterative loop using %goat-course. 
# 
# 2. Use instructions:
# Execute this script after completing Stage 3.8. It relies on the Stage 1.3 
# Script Engine loaded in your active Jupyter memory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Automatically bypasses purely strong complexes. Extracts high-accuracy 
# sub-units, runs the Frozen Escape Room, explores via GOAT-COURSE, and exports 
# a unified Weak Interaction Thermodynamic Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import shutil
import numpy as np
import itertools
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'run_orca_subprocess_safely' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    parse_freq = globals()['parse_first_vibrational_freq']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    get_goat_header = globals()['get_goat_header']
    extract_ensemble = globals()['extract_goat_ensemble']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
    from ase import Atoms
    from ase.neighborlist import natural_cutoffs, NeighborList
    import networkx as nx
except ImportError:
    print_status("Required libraries (ASE, NetworkX) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
OUTPUT_DIR = "Isomer_xyz_Stage4_1_WeakInt"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# TOPOLOGY SPLICING & RIGID-BODY PROTOCOLS
# ==============================================================================

def get_fragment_indices(atoms):
    cutoffs = [c * 1.2 for c in natural_cutoffs(atoms)]
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)
    matrix = nl.get_connectivity_matrix()
    graph = nx.from_scipy_sparse_array(matrix)
    return [list(c) for c in nx.connected_components(graph)]

def get_interaction_matrix(atoms, frag_indices):
    n = len(frag_indices)
    inter_dist = np.full((n, n), np.inf)
    pos = atoms.get_positions()
    for i in range(n):
        for j in range(i+1, n):
            d = np.min(np.linalg.norm(pos[frag_indices[i]][:, None, :] - pos[frag_indices[j]][None, :, :], axis=-1))
            inter_dist[i, j] = d
            inter_dist[j, i] = d
    return inter_dist

def find_best_ensemble(base_name, stage_prefix):
    """Searches backwards through completion tiers to find the most accurate coordinates."""
    tiers = [8, 7, 6, 5, 4, 3, 2, 1] if stage_prefix == "Stage3" else [7, 6, 5, 4, 3, 2, 1]
    for t in tiers:
        pattern = os.path.join(f"Isomer_xyz_{stage_prefix}_{t}_*", f"*{base_name}_Final_Ensemble.xyz")
        matches = glob.glob(pattern)
        if matches: return matches[0]
    return None

def rigid_rattle_complex(atoms, translation_stdev=2.0, rotation_stdev=180.0):
    frags = [atoms[idx] for idx in get_fragment_indices(atoms)]
    if len(frags) < 2: return atoms.copy()
    
    new_atoms = Atoms()
    new_atoms += frags[0] 
    
    for frag in frags[1:]:
        f = frag.copy()
        vec = np.random.randn(3)
        vec /= np.linalg.norm(vec)
        vec *= np.random.uniform(0.5, translation_stdev)
        
        axis = np.random.randn(3)
        axis /= np.linalg.norm(axis)
        angle = np.random.uniform(-rotation_stdev, rotation_stdev)
        
        f.rotate(angle, axis, center='COM')
        f.translate(vec)
        new_atoms += f
        
    return new_atoms

def run_frozen_escape_room(base_name, seed_atoms, work_dir, sys_config, cores, charge, mult):
    print_status(f"Triggering FROZEN ESCAPE ROOM for {base_name}...", "info")
    escaped_isomers = []
    
    # 1. Thermal Shock (Rigid Body Translation)
    shocked = rigid_rattle_complex(seed_atoms, translation_stdev=3.5, rotation_stdev=180.0)
    s_name = f"{base_name}_WEscape_Shock"
    write(os.path.join(work_dir, f"{s_name}.xyz"), shocked)
    
    with open(os.path.join(work_dir, f"{s_name}.inp"), 'w') as f:
        f.write(f"! xTB2 OPT RIGIDBODYOPT\n%pal nprocs {cores} end\n")
        f.write(f"* xyzfile {charge} {mult} {s_name}.xyz\n")
    
    out_s = run_orca_safely("Frozen Escape: Shock", f"{s_name}.inp", work_dir, sys_config, exit_on_fail=False)
    if out_s:
        try:
            opt_s = read(os.path.join(work_dir, f"{s_name}_OPT.xyz"))
            opt_s.info['energy'] = parse_energy(out_s)
            escaped_isomers.append(opt_s)
        except: pass

    # 2. Stochastic Rattle
    rattle = rigid_rattle_complex(seed_atoms, translation_stdev=1.0, rotation_stdev=45.0)
    r_name = f"{base_name}_WEscape_Rattle"
    write(os.path.join(work_dir, f"{r_name}.xyz"), rattle)
    
    with open(os.path.join(work_dir, f"{r_name}.inp"), 'w') as f:
        f.write(f"! xTB2 OPT RIGIDBODYOPT\n%pal nprocs {cores} end\n")
        f.write(f"* xyzfile {charge} {mult} {r_name}.xyz\n")
    
    out_r = run_orca_safely("Frozen Escape: Rattle", f"{r_name}.inp", work_dir, sys_config, exit_on_fail=False)
    if out_r:
        try:
            opt_r = read(os.path.join(work_dir, f"{r_name}_OPT.xyz"))
            opt_r.info['energy'] = parse_energy(out_r)
            escaped_isomers.append(opt_r)
        except: pass
        
    return escaped_isomers

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================

def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 4.1: WEAK INTERACTIONS{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)
    cores = min(16, sys_config.get("hardware", {}).get("cpu_threads", 4))

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        mace_applicable = item.get("mace_applicable", True)
        
        print(f"\n{Colors.BOLD}--- Evaluating Topology: {base_name} ---{Colors.ENDC}")
        
        initial_xyz = os.path.join("Isomer_xyz_Initial_Files", f"{base_name}.xyz")
        if not os.path.exists(initial_xyz): continue
        init_atoms = read(initial_xyz)
        
        frag_indices = get_fragment_indices(init_atoms)
        if len(frag_indices) == 1:
            print_status(f"Bypass Triggered: {base_name} is a single fragment. Skipping Stage 4.1.", "warning")
            continue
            
        inter_dist = get_interaction_matrix(init_atoms, frag_indices)
        strong_frags = set()
        weak_frags = set()
        
        for i in range(len(frag_indices)):
            is_strong = False
            for j in range(len(frag_indices)):
                if i != j and inter_dist[i, j] < 2.5: is_strong = True
            if is_strong: strong_frags.add(i)
            else: weak_frags.add(i)
            
        if len(weak_frags) == 0:
            print_status(f"Bypass Triggered: {base_name} consists ONLY of strong interactions. Skipping Stage 4.1.", "warning")
            continue
            
        print_status(f"Weak Interaction boundaries detected. Assembling high-accuracy subunits...", "success")
        
        work_dir = f"Stage4_1_WeakInt_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
        
        # 1. Splice and Load Subunits
        pools_to_combine = []
        ref_coms = []
        
        if strong_frags:
            strong_file = find_best_ensemble(base_name, "Stage3")
            if not strong_file:
                print_status(f"Missing Stage 3 output for strong cluster of {base_name}. Skipping.", "error")
                continue
            strong_isomers = read(strong_file, index=':')
            s_idx = []
            for f in strong_frags: s_idx.extend(frag_indices[f])
            
            s_pool = [iso[sorted(s_idx)] for iso in strong_isomers]
            s_pool = the_crusher(s_pool, system_temperature, silent=True)
            pools_to_combine.append(s_pool)
            ref_coms.append(init_atoms[sorted(s_idx)].get_center_of_mass())
            
        weak_file = find_best_ensemble(base_name, "Stage2")
        if not weak_file:
            print_status(f"Missing Stage 2 output for weak fragments of {base_name}. Skipping.", "error")
            continue
        weak_isomers = read(weak_file, index=':')
        
        for w_f in weak_frags:
            w_idx = sorted(frag_indices[w_f])
            w_pool = [iso[w_idx] for iso in weak_isomers]
            w_pool = the_crusher(w_pool, system_temperature, silent=True)
            pools_to_combine.append(w_pool)
            ref_coms.append(init_atoms[w_idx].get_center_of_mass())
            
        # 2. Combinatorial Assembly
        combinations = list(itertools.product(*pools_to_combine))
        print_status(f"Assembled {len(combinations)} total cross-combinations from unique rigid subunits.", "success")
        
        combined_seeds = []
        for combo in combinations:
            new_mol = Atoms()
            for i, frag in enumerate(combo):
                f_copy = frag.copy()
                f_copy.translate(ref_coms[i] - f_copy.get_center_of_mass())
                new_mol += f_copy
            combined_seeds.append(new_mol)
            
        if len(combined_seeds[0]) < 4:
            print_status("Bypass Triggered: Combined complex has < 4 atoms (No dihedrals).", "warning")
            continue
            
        # 3. The Frozen GOAT-COURSE Loop
        all_raw_complexes = []
        for c_idx, c_seed in enumerate(combined_seeds):
            c_name = f"{base_name}_WComb{c_idx}"
            
            # Initial Rigid Opt
            inp_opt = f"{c_name}_Init.inp"
            write(os.path.join(work_dir, f"{c_name}_Init.xyz"), c_seed)
            with open(os.path.join(work_dir, inp_opt), "w") as f:
                if mace_applicable:
                    hdr = get_goat_header(sys_config, c_seed).replace("GOAT", "OPT")
                    f.write(f"! RIGIDBODYOPT\n{hdr}")
                else:
                    f.write(f"! xTB2 OPT RIGIDBODYOPT\n")
                f.write(f"%pal nprocs {cores} end\n")
                f.write(f"* xyzfile {charge} {mult} {c_name}_Init.xyz\n")
                
            out = run_orca_safely(f"Rigid Combine Opt {c_idx}", inp_opt, work_dir, sys_config, exit_on_fail=False)
            if out:
                try:
                    opt_c = read(os.path.join(work_dir, f"{c_name}_Init_OPT.xyz"))
                    opt_c.info['energy'] = parse_energy(out)
                    all_raw_complexes.append(opt_c)
                    
                    # Run Frozen Escape Room 
                    escaped = run_frozen_escape_room(c_name, opt_c, work_dir, sys_config, cores, charge, mult)
                    all_raw_complexes.extend(escaped)
                except: pass

        unique_seeds = the_crusher(all_raw_complexes, system_temperature)
        target_seed = unique_seeds[0] if unique_seeds else combined_seeds[0]
        write(os.path.join(work_dir, f"{base_name}_GOAT_Seed.xyz"), target_seed)
        
        # Deep GOAT-COURSE Exploration
        if mace_applicable:
            goat_header = f"! RIGIDBODYOPT\n{get_goat_header(sys_config, target_seed).replace('GOAT', 'GOAT-COURSE')}"
        else:
            goat_header = "! GOAT-COURSE XTB2 RIGIDBODYOPT\n"
            
        inp_goat = f"{base_name}_GOAT_COURSE.inp"
        with open(os.path.join(work_dir, inp_goat), "w") as f:
            f.write(goat_header)
            f.write(f"%pal nprocs {cores} end\n")
            f.write("%goat-course\n  MAXITERMULT 6\n  MINGLOBALITER 30\n  MAXGLOBALITER 300\n  NWorkers auto\n  MAXEN 12.0\n  RMSD 1.5\n  ROTCONSTDIFF 0.1\nend\n")
            f.write(f"* xyzfile {charge} {mult} {base_name}_GOAT_Seed.xyz\n")
            
        run_orca_safely(f"GOAT-COURSE Exploration", inp_goat, work_dir, sys_config, exit_on_fail=False)
        all_raw_complexes.extend(extract_ensemble(work_dir, f"{base_name}_GOAT_COURSE"))
        
        final_unique = the_crusher(all_raw_complexes, system_temperature)
        
        # 4. Final Frequencies & Pickett Output
        freq_outputs = []
        print_status(f"Calculating Final Frequencies for {len(final_unique)} weak complexes...", "info")
        for idx, iso in enumerate(final_unique):
            freq_name = f"{base_name}_Iso{idx+1}_FREQ"
            pickett_name = f"{base_name}_Iso{idx+1}_Stage4_1.pickett"
            
            write(os.path.join(work_dir, f"{freq_name}.xyz"), iso)
            with open(os.path.join(work_dir, f"{freq_name}.inp"), 'w') as f:
                f.write(f"! xTB2 FREQ RIGIDBODYOPT\n%pal nprocs {cores} end\n")
                f.write(f"%output\n  Pickettname \"{pickett_name}\"\nend\n")
                f.write(f"* xyzfile {charge} {mult} {freq_name}.xyz\n")
                
            out_path = run_orca_safely(f"Rigid FREQ (Iso {idx+1})", f"{freq_name}.inp", work_dir, sys_config, exit_on_fail=False)
            freq_outputs.append(out_path)

        for iso, out_path in zip(final_unique, freq_outputs):
            if out_path: iso.info['v1_freq'] = parse_freq(out_path)

        xyz_path, csv_path = export_thermo(
            final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage4_1_WeakInt"
        )

    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 4.1 COMPLETE: Weak Interaction Conformational Mapping Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 4.2: Weak Interaction Triage

Purpose: Interactive 3D UI to verify the topological deduplication of Weak Complexes.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 04_2-VISUAL-TRIAGE-WEAKINT-AA (Stage 4.1)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage provides the visual gatekeeper for Weak Interaction Complexes 
# generated in Stage 4.1. Because rigid-body perturbations and combinatorial 
# fragment assemblies can generate immense and subtle geometric diversity, the 
# spectroscopist must visually verify the topological groups before moving to 
# high-accuracy refinements.
# 
# 2. Use instructions:
# Execute this script in your active Jupyter environment/VS Code after Stage 4.1.
# If your system was bypassed (single molecule, strong-interaction only, or < 4 atoms), 
# this stage will politely notify you and do nothing. Otherwise, select the complex 
# and click "Launch Triage". Click "Accept Selections" to export the verified inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI for weak complexes. Upon acceptance, 
# successfully writes to the 'Isomer_xyz_Stage4_2_Verified' folder using the 
# Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage4_1_WeakInt"
OUTPUT_DIR = "Isomer_xyz_Stage4_2_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class WeakIntVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for Weak Complexes.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage4_2_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Weak Interaction Complex Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_weak_int_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 4.2: WEAK INT. TRIAGE {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.1 (Single fragment, strong interaction only, or < 4 atoms).", "info")
        return

    # Specifically search for the output generated by Stage 4.1 Weak Interaction Assembly
    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage4_1_WeakInt_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 4.1 Assembled Ensembles found in {INPUT_DIR}.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.1.", "info")
        return

    # Extract base names for the dropdown menu
    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage4_1_WeakInt_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Complex:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 4.1 Weak Interaction Complex to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            
            # Additional safety check against user config flags
            ingested_files = user_config.get("ingested_files", [])
            for item in ingested_files:
                if item.get("file", "").replace('.xyz', '') == base_name:
                    if item.get("fragments_count", 1) == 1:
                        print_status(f"Bypass Enforced: {base_name} is a single fragment.", "warning")
                        return

            print_status(f"Loading assembled weak coordinates for {base_name}...", "info")
            
            try:
                # Read all assembled isomers from the Stage 4.1 ensemble
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                if len(isomers[0]) < 4:
                    print_status(f"Bypass Enforced: {base_name} lacks enough atoms (<4) for intermolecular dihedral evaluation.", "warning")
                    return
                
                # Pass them through the grouping crusher to array them for the UI
                print_status("Executing Grouping Crusher alignment on weak complex geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                # Instantiate the dynamically subclassed UI
                WeakIntVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_weak_int_triage()


🎯 Stage 4.4: High-Accuracy Weak Complex Refinement

Purpose: Executes high-accuracy DFT refinement on Weak Complexes while strictly freezing internal fragment geometries.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 04_4-HIGH-ACC-WEAK-COMPLEX-AA (Stage 4.2)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes High-Accuracy DFT (r²SCAN-3c) refinement specifically 
# for Weak Interaction Complexes. Crucially, the internal high-accuracy (CCSD(T)) 
# geometries of the constituent fragments are STRICTLY FROZEN using RIGIDBODYOPT. 
# Only the weak intermolecular van der Waals (vdW) interactions are optimized.
# It concludes with a rigid-body ExtremeOpt Topographic Funnel filter.
# 
# 2. Use instructions:
# Execute this script after completing Stage 4.2 in VS Code. 
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates highly refined rigid-body complex geometries, outputs Pickett 
# files for SPCAT, and exports a fully authenticated Thermodynamic Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_high_acc_opt_loop' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    apply_molsym_preopt = globals()['apply_molsym_preopt']
    engine_high_acc_opt_loop = globals()['engine_high_acc_opt_loop']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage4_2_Verified"
OUTPUT_DIR = "Isomer_xyz_Stage4_4_HighAccWeak"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

def calculate_dynamic_cores(num_isomers):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_mb = mem_gb * 1024
    
    max_jobs_by_mem = max(1, int(mem_mb / 500))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 4.4: WEAK COMPLEX REF {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.2.", "info")
        sys.exit(0)
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        num_frags = item.get("fragments_count", 1)
        
        # 1. Complex Bypass Check
        if num_frags == 1:
            print_status(f"Bypass Triggered: {base_name} is a single fragment. Skipping Stage 4.4.", "warning")
            continue
            
        ensemble_file = os.path.join(INPUT_DIR, f"Stage4_2_Verified_{base_name}_Final_Ensemble.xyz")
        if not os.path.exists(ensemble_file): 
            print_status(f"Missing Stage 4.2 ensemble for {base_name}. Skipping.", "error")
            continue
            
        isomers = read(ensemble_file, index=':')
        num_isomers = len(isomers)
        
        if len(isomers[0]) < 4:
            print_status(f"Bypass Triggered: {base_name} complex lacks enough atoms (<4) for an intermolecular dihedral. Skipping.", "warning")
            continue
            
        print(f"\n{Colors.BOLD}--- Processing Weak Complex Ensemble: {base_name} ---{Colors.ENDC}")
        work_dir = f"Stage4_4_HighAccWeak_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers)
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job", "info")
        
        # ---------------------------------------------------------
        # PHASE 1: MolSym Auto-Symmetrization
        # ---------------------------------------------------------
        print_status("Phase 1: MolSym Symmetrization (vdW coordinates only)...", "info")
        preopt_isomers = []
        for idx, iso in enumerate(isomers):
            sym_iso = apply_molsym_preopt(iso, base_name, idx, work_dir, prefix="molsym_weak")
            preopt_isomers.append(sym_iso)

        # ---------------------------------------------------------
        # PHASE 2: Frozen High-Accuracy DFT (Delegated to Stage 1.3 Engine)
        # ---------------------------------------------------------
        print(f"\n{Colors.BOLD}--- Phase 2: Frozen High-Accuracy Optimization ---{Colors.ENDC}")
        print_status("Forcing ORCA_CPU Routing to satisfy rigid-body mathematical constraints.", "info")
        
        # We explicitly enforce ORCA_CPU and inject RIGIDBODYOPT so the internal 
        # CCSDT fragment geometries remain perfectly locked.
        high_acc_isomers = engine_high_acc_opt_loop(
            preopt_isomers=preopt_isomers,
            base_name=base_name,
            work_dir=work_dir,
            routing_flag="ORCA_CPU",
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage4_4",
            job_prefix_mod="weak_r2scan",
            command_override="! r2SCAN-3c RIGIDBODYOPT FREQ"
        )

        # ---------------------------------------------------------
        # PHASE 3: Strict Topographic Funnel & Filter (Frozen)
        # ---------------------------------------------------------
        print(f"\n{Colors.BOLD}--- Phase 3: The Strict Topographic Funnel (Rigid Body) ---{Colors.ENDC}")
        unique_pre_filter = the_crusher(high_acc_isomers, system_temperature)
        
        strict_isomers = []
        for idx, iso in enumerate(unique_pre_filter):
            iso_path = os.path.join(work_dir, f"strict_weak_comp_{idx}.xyz")
            inp_name = f"{base_name}_strict_weak_comp_{idx}.inp"
            write(iso_path, iso)
            
            with open(os.path.join(work_dir, inp_name), "w") as f:
                # Blast out shallow transition states while keeping fragments strictly frozen
                f.write(f"! xTB2 ExtremeOpt RIGIDBODYOPT\n") 
                f.write(f"%pal nprocs {cores_per_job} end\n")
                f.write(f"* xyzfile {charge} {mult} {os.path.basename(iso_path)}\n")
                
            out = run_orca_safely(f"Strict Rigid Filter Complex {idx}", inp_name, work_dir, sys_config, exit_on_fail=False)
            if out:
                try:
                    strict_iso = read(os.path.join(work_dir, f"{base_name}_strict_weak_comp_{idx}_ExtremeOpt.xyz"))
                    strict_iso.info['energy'] = parse_energy(out)
                    if 'v1_freq' in iso.info: strict_iso.info['v1_freq'] = iso.info['v1_freq']
                    strict_isomers.append(strict_iso)
                except: pass

        final_unique = the_crusher(strict_isomers, system_temperature)

        # ---------------------------------------------------------
        # FINAL EXPORT PROTOCOL
        # ---------------------------------------------------------
        xyz_path, csv_path = export_thermo(
            final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage4_4_HighAccWeak"
        )
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 4.4 COMPLETE: High-Accuracy Weak Complex Refinement Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


🚀 Stage 4.5: High-Level Weak Complex Extrap

Purpose: Executes Double-Hybrid (revDSD) extrapolations on Weak Complexes with frozen internal coordinates.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 04_5-HIGH-LEVEL-WEAK-COMPLEX-EXTRAP-AA (Stage 4.5)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes the ultimate High-Level Double-Hybrid Extrapolations 
# (revDSD-PBEP86-D4) specifically for Weak Interaction Complexes from Stage 4.4. 
# Crucially, the internal high-accuracy (CCSD(T)) geometries of the constituent 
# fragments are STRICTLY FROZEN using RIGIDBODYOPT. Only the weak intermolecular 
# van der Waals (vdW) interactions are optimized. It delegates execution to the 
# Stage 1.3 engine and scales basis sets dynamically.
# 
# 2. Use instructions:
# Execute this script after completing Stage 4.4 in VS Code. It utilizes 
# Checkpoint/Restart logic natively via the engine.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates ultra-high accuracy intermolecular rigid-body geometries. 
# Outputs specialized Pickett files and exports the final Stage 4.5 Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_revdsd_opt_loop' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    apply_molsym_preopt = globals()['apply_molsym_preopt']
    engine_revdsd_opt_loop = globals()['engine_revdsd_opt_loop']
    estimate_memory = globals()['estimate_orca_memory']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage4_4_HighAccWeak"
OUTPUT_DIR = "Isomer_xyz_Stage4_5_HighLevelWeak"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# DYNAMIC EXECUTION PROTOCOLS
# ==============================================================================

def calculate_dynamic_cores(num_isomers):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_per_job = 6000 
    
    max_jobs_by_mem = max(1, int((mem_gb * 1024) / mem_per_job))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

def determine_complex_extrapolation_level(base_size, num_isomers):
    """
    Bumps the size category by 1 for every 5 isomers to prevent timeframe explosions,
    then assigns the rigorous Double-Hybrid keywords requested. 
    Overrides standard OPT with RIGIDBODYOPT.
    """
    size_map = {"Small": 0, "Medium": 1, "Large": 2}
    inv_map = {0: "Small", 1: "Medium", 2: "Large"}
    
    base_idx = size_map.get(base_size, 0)
    bump = num_isomers // 5
    final_idx = min(2, base_idx + bump)
    final_size = inv_map[final_idx]
    
    if final_size == "Small":
        method = "revDSD-PBEP86-D4"
        basis = "aug-cc-pVTZ"
        header = f"! {method} {basis} aug-cc-pVTZ/JK aug-cc-pVTZ/C RIJCOSX ExtremeSCF RIGIDBODYOPT DEFGRID3\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
    elif final_size == "Medium":
        method = "revDSD-PBEP86-D4"
        basis = "cc-pVTZ"
        header = f"! {method} {basis} cc-pVTZ/JK cc-pVTZ/C RIJCOSX TightSCF RIGIDBODYOPT DEFGRID2\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
    else:
        method = "revDSD-PBEP86-D4"
        basis = "def2-TZVP"
        header = f"! {method} {basis} def2/J def2-TZVP/C RIJCOSX NormalSCF RIGIDBODYOPT DEFGRID2\n"
        
    return final_size, method, basis, header

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 4.5: WEAK COMPLEX EXTRAP{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.4.", "info")
        sys.exit(0)
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        original_size = item.get("size", "Small")
        num_frags = item.get("fragments_count", 1)
        
        # 1. Complex Bypass Check
        if num_frags == 1:
            print_status(f"Bypass Triggered: {base_name} is a single fragment. Skipping Stage 4.5.", "warning")
            continue
            
        ensemble_file = os.path.join(INPUT_DIR, f"Stage4_4_HighAccWeak_{base_name}_Final_Ensemble.xyz")
        if not os.path.exists(ensemble_file): 
            print_status(f"Missing Stage 4.4 ensemble for {base_name}. Skipping.", "error")
            continue
            
        isomers = read(ensemble_file, index=':')
        num_isomers = len(isomers)
        
        if len(isomers[0]) < 4:
            print_status(f"Bypass Triggered: {base_name} complex lacks enough atoms (<4) for an intermolecular dihedral. Skipping.", "warning")
            continue
            
        print(f"\n{Colors.BOLD}--- Extrapolating Weak Complex Ensemble: {base_name} ---{Colors.ENDC}")
        work_dir = f"Stage4_5_HighLevelWeak_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        # ---------------------------------------------------------
        # PHASE 1: Pre-Optimize with MolSym (vdW coordinates)
        # ---------------------------------------------------------
        print_status("Phase 1: MolSym Symmetrization (vdW coordinates only)...", "info")
        preopt_isomers = []
        for idx, iso in enumerate(isomers):
            sym_iso = apply_molsym_preopt(iso, base_name, idx, work_dir, prefix="molsym_extrap_weak")
            preopt_isomers.append(sym_iso)

        # ---------------------------------------------------------
        # PHASE 2: Config & Execution
        # ---------------------------------------------------------
        final_size, method, basis, block_header = determine_complex_extrapolation_level(original_size, num_isomers)
        print_status(f"Isomer Count Scaling: {num_isomers} Isomers -> Shifted size from {original_size} to {Colors.WARNING}{final_size}{Colors.ENDC}", "info")
        print_status(f"Assigned Double Hybrid Method: {Colors.OKGREEN}{method} / {basis} (RIGIDBODYOPT){Colors.ENDC}", "info")
        
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers)
        
        try:
            maxcore = estimate_memory(isomers[0], method=method, basis_set=basis)
        except Exception:
            maxcore = 3000
            
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job ({maxcore} MB maxcore)", "info")
        
        # Execute Engine Protocol Loop natively from Stage 1.3
        high_level_isomers = engine_revdsd_opt_loop(
            isomers=preopt_isomers,
            base_name=base_name,
            work_dir=work_dir,
            block_header=block_header,
            maxcore=maxcore,
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage4_5",
            job_prefix_mod="weak_extrap"
        )

        # ---------------------------------------------------------
        # PHASE 3: Crusher & Final Export
        # ---------------------------------------------------------
        if high_level_isomers:
            final_unique = the_crusher(high_level_isomers, system_temperature, silent=True)
            xyz_path, csv_path = export_thermo(
                final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage4_5_HighLevelWeak"
            )
        else:
            print_status(f"All calculations failed for {base_name}. No inventory exported.", "error")
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 4.5 COMPLETE: High-Level Weak Complex Extrapolation Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 4.6: Weak Extrap Triage

Purpose: Visual verification of the frozen Double-Hybrid Weak Complexes.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 04_6-VISUAL-TRIAGE-WEAK-COMPLEX-EXTRAP-AA (Stage 4.6)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# Following the ultimate High-Level Extrapolations (revDSD-PBEP86-D4) for Weak 
# Interaction Complexes in Stage 4.5, flat intermolecular potential energy surfaces 
# might collapse previously distinct geometries. This stage loads the extrapolated 
# complex ensembles, groups them via the grouping crusher, and launches the final 
# 3D Visual Triage Board. 
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment after Stage 4.5 has finished.
# If your system was bypassed (single molecule or < 4 atoms), this stage will 
# politely notify you and do nothing. Select your complex from the dropdown menu 
# and click "Launch Triage". Verify the 3D structures and click "Accept Selections" 
# to instantly generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI. Upon acceptance, successfully writes 
# to the 'Isomer_xyz_Stage4_6_Verified' folder using the Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage4_5_HighLevelWeak"
OUTPUT_DIR = "Isomer_xyz_Stage4_6_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class WeakComplexExtrapVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for High-Level 
    Extrapolated Weak Complexes.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage4_6_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Ultimate Weak Complex Extrapolation Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_weak_complex_extrap_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 4.6: WEAK EXTRAP TRIAGE{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.5.", "info")
        return

    # Specifically search for the output generated by Stage 4.5 High-Level Weak Complex Extrapolation
    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage4_5_HighLevelWeak_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 4.5 Extrapolated Ensembles found in {INPUT_DIR}.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.5 (Single fragment or < 4 atoms).", "info")
        return

    # Extract base names for the dropdown menu
    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage4_5_HighLevelWeak_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Complex:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 4.5 Extrapolated Weak Complex to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            
            # Additional safety check against user config flags
            ingested_files = user_config.get("ingested_files", [])
            for item in ingested_files:
                if item.get("file", "").replace('.xyz', '') == base_name:
                    if item.get("fragments_count", 1) == 1:
                        print_status(f"Bypass Enforced: {base_name} is a single fragment.", "warning")
                        return

            print_status(f"Loading extrapolated weak complex coordinates for {base_name}...", "info")
            
            try:
                # Read all isomers from the Stage 4.5 ensemble
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                if len(isomers[0]) < 4:
                    print_status(f"Bypass Enforced: {base_name} lacks enough atoms (<4) for intermolecular dihedral evaluation.", "warning")
                    return
                
                # Pass them through the grouping crusher to array them for the UI
                print_status("Executing Grouping Crusher alignment on highly refined geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                # Instantiate the dynamically subclassed UI
                WeakComplexExtrapVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_weak_complex_extrap_triage()


🔓 Stage 4.7: Full Opt Weak Complex Extrap

Purpose: Executes fully relaxed Double-Hybrid optimizations on Weak Complexes without rigid constraints.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 04_7-HIGH-LEVEL-WEAK-COMPLEX-FULL-OPT-AA (Stage 4.7)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes a FULL (relaxed) High-Level Double-Hybrid Extrapolation 
# (revDSD-PBEP86-D4) for Weak Interaction Complexes. Unlike Stage 4.5, this 
# optimization drops the RIGIDBODYOPT constraint, allowing the CCSD(T) internal 
# fragment geometries to gently relax in response to the optimized intermolecular 
# van der Waals forces. 
# 
# 2. Use instructions:
# Execute this script after completing Stage 4.6 (or Stage 4.4 if triage was skipped).
# It utilizes Checkpoint/Restart logic natively via the engine and dynamically 
# scales basis sets based on isomer count.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates ultra-high accuracy relaxed complex geometries. Outputs 
# specialized Pickett files and exports the final Stage 4.7 Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_revdsd_opt_loop' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    apply_molsym_preopt = globals()['apply_molsym_preopt']
    engine_revdsd_opt_loop = globals()['engine_revdsd_opt_loop']
    estimate_memory = globals()['estimate_orca_memory']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
OUTPUT_DIR = "Isomer_xyz_Stage4_7_FullOptWeak"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# DYNAMIC EXECUTION PROTOCOLS
# ==============================================================================

def calculate_dynamic_cores(num_isomers):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_per_job = 6000 
    
    max_jobs_by_mem = max(1, int((mem_gb * 1024) / mem_per_job))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

def determine_full_opt_extrapolation_level(base_size, num_isomers):
    """
    Bumps the size category by 1 for every 5 isomers to prevent timeframe explosions,
    then assigns the rigorous Double-Hybrid keywords requested. 
    Notice: NO RIGIDBODYOPT keyword is used here to allow full complex relaxation.
    """
    size_map = {"Small": 0, "Medium": 1, "Large": 2}
    inv_map = {0: "Small", 1: "Medium", 2: "Large"}
    
    base_idx = size_map.get(base_size, 0)
    bump = num_isomers // 5
    final_idx = min(2, base_idx + bump)
    final_size = inv_map[final_idx]
    
    if final_size == "Small":
        method = "revDSD-PBEP86-D4"
        basis = "aug-cc-pVTZ"
        header = f"! {method} {basis} aug-cc-pVTZ/JK aug-cc-pVTZ/C RIJCOSX ExtremeSCF TightOPT DEFGRID3\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
    elif final_size == "Medium":
        method = "revDSD-PBEP86-D4"
        basis = "cc-pVTZ"
        header = f"! {method} {basis} cc-pVTZ/JK cc-pVTZ/C RIJCOSX TightSCF OPT DEFGRID2\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
    else:
        method = "revDSD-PBEP86-D4"
        basis = "def2-TZVP"
        header = f"! {method} {basis} def2/J def2-TZVP/C RIJCOSX NormalSCF OPT DEFGRID2\n"
        
    return final_size, method, basis, header

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 4.7: WEAK COMPLEX FULL OPT{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        original_size = item.get("size", "Small")
        num_frags = item.get("fragments_count", 1)
        
        # 1. Complex Bypass Check
        if num_frags == 1:
            print_status(f"Bypass Triggered: {base_name} is a single fragment. Skipping Stage 4.7.", "warning")
            continue
            
        # 2. Dynamic Input Routing (Pulls from 4.6 if available, else 4.4)
        ensemble_file_46 = os.path.join("Isomer_xyz_Stage4_6_Verified", f"Stage4_6_Verified_{base_name}_Final_Ensemble.xyz")
        ensemble_file_44 = os.path.join("Isomer_xyz_Stage4_4_HighAccWeak", f"Stage4_4_HighAccWeak_{base_name}_Final_Ensemble.xyz")
        
        if os.path.exists(ensemble_file_46):
            ensemble_file = ensemble_file_46
            print_status(f"Discovered Stage 4.6 Verified Ensemble for {base_name}.", "success")
        elif os.path.exists(ensemble_file_44):
            ensemble_file = ensemble_file_44
            print_status(f"Stage 4.6 missing. Falling back to Stage 4.4 High-Accuracy Ensemble for {base_name}.", "info")
        else:
            print_status(f"Missing Stage 4.4 and Stage 4.6 ensembles for {base_name}. Skipping.", "error")
            continue
            
        isomers = read(ensemble_file, index=':')
        num_isomers = len(isomers)
        
        if len(isomers[0]) < 4:
            print_status(f"Bypass Triggered: {base_name} complex lacks enough atoms (<4) for an intermolecular dihedral. Skipping.", "warning")
            continue
            
        print(f"\n{Colors.BOLD}--- Full Optimization of Weak Complex: {base_name} ---{Colors.ENDC}")
        work_dir = f"Stage4_7_FullOptWeak_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        # ---------------------------------------------------------
        # PHASE 1: Pre-Optimize with MolSym
        # ---------------------------------------------------------
        print_status("Phase 1: MolSym Symmetrization...", "info")
        preopt_isomers = []
        for idx, iso in enumerate(isomers):
            sym_iso = apply_molsym_preopt(iso, base_name, idx, work_dir, prefix="molsym_fullopt_weak")
            preopt_isomers.append(sym_iso)

        # ---------------------------------------------------------
        # PHASE 2: Config & Execution
        # ---------------------------------------------------------
        final_size, method, basis, block_header = determine_full_opt_extrapolation_level(original_size, num_isomers)
        print_status(f"Isomer Count Scaling: {num_isomers} Isomers -> Shifted size from {original_size} to {Colors.WARNING}{final_size}{Colors.ENDC}", "info")
        print_status(f"Assigned Double Hybrid Method: {Colors.OKGREEN}{method} / {basis} (FULL OPT){Colors.ENDC}", "info")
        
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers)
        
        try:
            maxcore = estimate_memory(isomers[0], method=method, basis_set=basis)
        except Exception:
            maxcore = 3000
            
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job ({maxcore} MB maxcore)", "info")
        
        # Execute Engine Protocol Loop natively from Stage 1.3
        high_level_isomers = engine_revdsd_opt_loop(
            isomers=preopt_isomers,
            base_name=base_name,
            work_dir=work_dir,
            block_header=block_header,
            maxcore=maxcore,
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage4_7",
            job_prefix_mod="weak_fullopt"
        )

        # ---------------------------------------------------------
        # PHASE 3: Crusher & Final Export
        # ---------------------------------------------------------
        if high_level_isomers:
            final_unique = the_crusher(high_level_isomers, system_temperature, silent=True)
            xyz_path, csv_path = export_thermo(
                final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage4_7_FullOptWeak"
            )
        else:
            print_status(f"All calculations failed for {base_name}. No inventory exported.", "error")
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 4.7 COMPLETE: Full Relaxed Weak Complex Optimization Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 4.8: Weak Full Opt Triage

Purpose: Visual verification of the fully relaxed Double-Hybrid Weak Complexes.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 04_8-VISUAL-TRIAGE-WEAK-COMPLEX-FULL-OPT-AA (Stage 4.8)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# Following the fully relaxed High-Level Extrapolations (revDSD-PBEP86-D4) for 
# Weak Interaction Complexes in Stage 4.7, this stage provides the absolute 
# final visual verification. It loads the fully relaxed complex ensembles, 
# groups them via the grouping crusher, and launches the final 3D Visual Triage 
# Board to lock in the "Gold Standard" global minima for weak interactions.
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment/VS Code after Stage 4.7 has 
# finished. If your system was bypassed (single molecule or < 4 atoms), this stage 
# will politely notify you and do nothing. Select your complex from the dropdown 
# menu and click "Launch Triage". Verify the 3D structures and click "Accept Selections" 
# to instantly generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI. Upon acceptance, successfully writes 
# to the 'Isomer_xyz_Stage4_8_Verified' folder using the Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage4_7_FullOptWeak"
OUTPUT_DIR = "Isomer_xyz_Stage4_8_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class WeakComplexFullOptVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for Fully 
    Optimized Weak Complexes.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage4_8_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Gold Standard Weak Complex Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_weak_complex_full_opt_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 4.8: WEAK FULL OPT TRIAGE{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.7.", "info")
        return

    # Specifically search for the output generated by Stage 4.7 Full Relaxed Weak Complex Optimization
    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage4_7_FullOptWeak_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 4.7 Full Opt Ensembles found in {INPUT_DIR}.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.7 (Single fragment or < 4 atoms).", "info")
        return

    # Extract base names for the dropdown menu
    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage4_7_FullOptWeak_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Complex:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 4.7 Fully Optimized Weak Complex to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            
            # Additional safety check against user config flags
            ingested_files = user_config.get("ingested_files", [])
            for item in ingested_files:
                if item.get("file", "").replace('.xyz', '') == base_name:
                    if item.get("fragments_count", 1) == 1:
                        print_status(f"Bypass Enforced: {base_name} is a single fragment.", "warning")
                        return

            print_status(f"Loading fully relaxed weak complex coordinates for {base_name}...", "info")
            
            try:
                # Read all isomers from the Stage 4.7 ensemble
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                if len(isomers[0]) < 4:
                    print_status(f"Bypass Enforced: {base_name} lacks enough atoms (<4) for intermolecular dihedral evaluation.", "warning")
                    return
                
                # Pass them through the grouping crusher to array them for the UI
                print_status("Executing Grouping Crusher alignment on highly refined geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                # Instantiate the dynamically subclassed UI
                WeakComplexFullOptVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_weak_complex_full_opt_triage()


👑 Stage 4.9: Ultimate Weak Complex Extrap

Purpose: Executes fully relaxed CCSD(T) optimizations to secure the absolute minimum for Weak Complexes.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 04_9-ULTIMATE-WEAK-COMPLEX-EXTRAP-AA (Stage 4.9)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes the ultimate High-Level Coupled-Cluster Extrapolations 
# (CCSD(T) / DLPNO-CCSD(T)) specifically for Weak Interaction Complexes. 
# This performs a full optimization (no rigid body constraints) to allow 
# the geometries to gently relax into the "Gold Standard" minimum.
# 
# 2. Use instructions:
# Execute this script after completing Stage 4.6 (or 4.4 if triage was skipped) 
# in VS Code. It utilizes Checkpoint/Restart logic natively via the engine and 
# dynamically scales basis sets based on isomer count.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates absolute "Gold Standard" theoretical weak complex geometries. 
# Outputs specialized Pickett files, and generates the final Stage 4.9 Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_ccsdt_opt_loop' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    estimate_memory = globals()['estimate_orca_memory']
    engine_ccsdt_opt_loop = globals()['engine_ccsdt_opt_loop']
    apply_molsym_preopt = globals()['apply_molsym_preopt']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
OUTPUT_DIR = "Isomer_xyz_Stage4_9_UltimateWeak"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# DYNAMIC EXECUTION PROTOCOLS
# ==============================================================================

def calculate_dynamic_cores(num_isomers, is_ccsd=False):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    # CCSD(T) requires massive memory per thread
    mem_per_job = 8000 if is_ccsd else 4000 
    
    max_jobs_by_mem = max(1, int((mem_gb * 1024) / mem_per_job))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

def determine_ultimate_weak_extrapolation_level(base_size, num_isomers):
    """
    Bumps the size category by 1 for every 5 isomers to prevent timeframe explosions,
    then assigns the rigorous Stage 8 keywords.
    """
    size_map = {"Small": 0, "Medium": 1, "Large": 2}
    inv_map = {0: "Small", 1: "Medium", 2: "Large"}
    
    base_idx = size_map.get(base_size, 0)
    bump = num_isomers // 5
    final_idx = min(2, base_idx + bump)
    final_size = inv_map[final_idx]
    
    if final_size == "Small":
        method = "CCSD(T)"
        basis = "aug-cc-pwCVQZ"
        header = f"! {method} {basis} TightOPT AutoAux ExtremeSCF DEFGRID3 UseSym\n"
        header += "%method\n  Z_Tol 1e-14;\n  FrozenCore FC_NONE;\nend\n"
        header += "%geom\n  TolE 1e-9;\n  TolRMSG 3e-6;\n  TolMaxG 1e-5;\nend\n"
    elif final_size == "Medium":
        method = "DLPNO-CCSD(T)"
        basis = "aug-cc-pVTZ"
        header = f"! {method} TightPNO {basis} TightOPT AutoAux ExtremeSCF DEFGRID3 UseSym\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
        header += "%geom\n  TolE 1e-9;\n  TolRMSG 3e-6;\n  TolMaxG 1e-5;\nend\n"
    else:
        method = "DLPNO-CCSD(T)"
        basis = "cc-pVTZ"
        header = f"! {method} NormalPNO {basis} OPT AutoAux TightSCF DEFGRID2 UseSym\n"
        
    return final_size, method, basis, header

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 4.9: ULTIMATE WEAK CC EXTRAP{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        original_size = item.get("size", "Small")
        num_frags = item.get("fragments_count", 1)
        
        # 1. Complex Bypass Check
        if num_frags == 1:
            print_status(f"Bypass Triggered: {base_name} is a single fragment. Skipping Stage 4.9.", "warning")
            continue
        
        # 2. Dynamic Input Routing (Pulls from 4.6 if available, else 4.4)
        ensemble_file_46 = os.path.join("Isomer_xyz_Stage4_6_Verified", f"Stage4_6_Verified_{base_name}_Final_Ensemble.xyz")
        ensemble_file_44 = os.path.join("Isomer_xyz_Stage4_4_HighAccWeak", f"Stage4_4_HighAccWeak_{base_name}_Final_Ensemble.xyz")
        
        if os.path.exists(ensemble_file_46):
            ensemble_file = ensemble_file_46
            print_status(f"Discovered Stage 4.6 Verified Ensemble for {base_name}.", "success")
        elif os.path.exists(ensemble_file_44):
            ensemble_file = ensemble_file_44
            print_status(f"Stage 4.6 missing. Falling back to Stage 4.4 High-Accuracy Ensemble for {base_name}.", "info")
        else:
            print_status(f"Missing Stage 4.4 and Stage 4.6 ensembles for {base_name}. Skipping.", "error")
            continue
            
        isomers = read(ensemble_file, index=':')
        num_isomers = len(isomers)
        
        if len(isomers[0]) < 4:
            print_status(f"Bypass Triggered: {base_name} complex lacks enough atoms (<4) for an intermolecular dihedral. Skipping.", "warning")
            continue
            
        print(f"\n{Colors.BOLD}--- Extrapolating Weak Complex Ensemble: {base_name} ---{Colors.ENDC}")
        work_dir = f"Stage4_9_UltimateWeak_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)
            
        # ---------------------------------------------------------
        # PHASE 1: Pre-Optimize with MolSym
        # ---------------------------------------------------------
        print_status("Phase 1: MolSym Symmetrization...", "info")
        preopt_isomers = []
        for idx, iso in enumerate(isomers):
            sym_iso = apply_molsym_preopt(iso, base_name, idx, work_dir, prefix="molsym_cc_weak")
            preopt_isomers.append(sym_iso)

        # ---------------------------------------------------------
        # PHASE 2: Config & Execution
        # ---------------------------------------------------------
        final_size, method, basis, block_header = determine_ultimate_weak_extrapolation_level(original_size, num_isomers)
        print_status(f"Isomer Count Scaling: {num_isomers} Isomers -> Shifted size from {original_size} to {Colors.WARNING}{final_size}{Colors.ENDC}", "info")
        print_status(f"Assigned Coupled-Cluster Method: {Colors.OKGREEN}{method} / {basis}{Colors.ENDC}", "info")
        
        is_ccsd = "DLPNO" not in method
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers, is_ccsd)
        
        try:
            maxcore = estimate_memory(isomers[0], method=method, basis_set=basis)
        except Exception:
            maxcore = 3000
            
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job ({maxcore} MB maxcore)", "info")
        
        # Execute Engine Protocol Loop (Passing requires_freq=False for strict Opt structure)
        high_level_isomers = engine_ccsdt_opt_loop(
            isomers=preopt_isomers,
            base_name=base_name,
            work_dir=work_dir,
            block_header=block_header,
            maxcore=maxcore,
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage4_9",
            job_prefix_mod="weak_ccsd",
            requires_freq=False
        )

        # ---------------------------------------------------------
        # PHASE 3: Crusher & Final Export
        # ---------------------------------------------------------
        if high_level_isomers:
            # Sift out any exact duplicates caused by subtle energetic collapses in CCSD(T)
            final_unique = the_crusher(high_level_isomers, system_temperature, silent=True)
            xyz_path, csv_path = export_thermo(
                final_unique, base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage4_9_UltimateWeak"
            )
        else:
            print_status(f"All calculations failed for {base_name}. No inventory exported.", "error")
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 4.9 COMPLETE: Ultimate Weak Complex Coupled-Cluster Extrapolation Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 4.10: Weak Ultimate Triage

Purpose: Final visual triage for "Gold Standard" fully relaxed CCSD(T) Weak Complexes.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 04_10-VISUAL-TRIAGE-WEAK-COMPLEX-ULTIMATE-AA (Stage 4.10)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# Following the ultimate High-Level Coupled-Cluster Extrapolations (CCSD(T) / 
# DLPNO-CCSD(T)) for Weak Interaction Complexes in Stage 4.9, this stage provides 
# the absolute final visual verification. It loads the Gold Standard complex 
# ensembles, groups them via the grouping crusher, and launches the final 
# 3D Visual Triage Board to lock in the true global minima for weak interactions.
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment/VS Code after Stage 4.9 has 
# finished. If your system was bypassed (single molecule or < 4 atoms), this stage 
# will politely notify you and do nothing. Select your complex from the dropdown 
# menu and click "Launch Triage". Verify the 3D structures and click "Accept Selections" 
# to instantly generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI. Upon acceptance, successfully writes 
# to the 'Isomer_xyz_Stage4_10_Verified' folder using the Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage4_9_UltimateWeak"
OUTPUT_DIR = "Isomer_xyz_Stage4_10_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class WeakComplexUltimateVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for Ultimate 
    Coupled-Cluster Weak Complexes.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage4_10_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Ultimate Gold Standard Weak Complex Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_weak_complex_ultimate_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 4.10: WEAK ULTIMATE TRIAGE{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.9.", "info")
        return

    # Specifically search for the output generated by Stage 4.9 Ultimate Weak Complex Extrapolation
    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage4_9_UltimateWeak_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 4.9 Ultimate Ensembles found in {INPUT_DIR}.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 4.9 (Single fragment or < 4 atoms).", "info")
        return

    # Extract base names for the dropdown menu
    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage4_9_UltimateWeak_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    dropdown = widgets.Dropdown(
        options=options,
        description='Complex:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 4.9 Ultimate Weak Complex to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            
            # Additional safety check against user config flags
            ingested_files = user_config.get("ingested_files", [])
            for item in ingested_files:
                if item.get("file", "").replace('.xyz', '') == base_name:
                    if item.get("fragments_count", 1) == 1:
                        print_status(f"Bypass Enforced: {base_name} is a single fragment.", "warning")
                        return

            print_status(f"Loading Gold Standard weak complex coordinates for {base_name}...", "info")
            
            try:
                # Read all isomers from the Stage 4.9 ensemble
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                if len(isomers[0]) < 4:
                    print_status(f"Bypass Enforced: {base_name} lacks enough atoms (<4) for intermolecular dihedral evaluation.", "warning")
                    return
                
                # Pass them through the grouping crusher to array them for the UI
                print_status("Executing Grouping Crusher alignment on highly refined geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                # Instantiate the dynamically subclassed UI
                WeakComplexUltimateVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_weak_complex_ultimate_triage()


⚛️ Stage 5.1: Automated Isotopologue Generation

Purpose: Automatically substitutes isotopic masses (>0.1% terrestrial abundance) and runs high-accuracy optimizations.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 05_1-AUTOMATED-ISOTOPOLOGUE-AA (Stage 5.1)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage automates the generation and optimization of molecular isotopologues 
# based on >0.1% terrestrial abundance limits (and Deuterium). It individually 
# substitutes every applicable atom across the ensemble, runs a high-accuracy 
# optimization (via the Stage 1.3 Engine protocol), and passes the variants 
# through a strict Topographic Funnel. No Pickett files are generated at this stage.
# 
# 2. Use instructions:
# Execute this script. If the 'Initial_Isotopologue_xyz' directory is empty, it 
# will halt and prompt you to add your ensemble(s). Once added, re-run to process.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates structural variants utilizing explicit isotopic masses, 
# executes optimizations utilizing the shared engine loop, and exports a unified 
# Thermodynamic Inventory containing the variants with the _[Atom#-Isotope] tag.
# ==============================================================================

import os
import sys
import json
import glob
import shutil
import numpy as np
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_high_acc_opt_loop' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    engine_high_acc_opt_loop = globals()['engine_high_acc_opt_loop']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Initial_Isotopologue_xyz"
OUTPUT_DIR = "Isomer_xyz_Stage5_1_Isotopologues"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# TERRESTRIAL ABUNDANCE & ISOTOPE DICTIONARY
# ==============================================================================
# Ruleset: > 0.1% natural abundance (plus Deuterium for Hydrogens)
ISOTOPE_RULES = {
    'H': [('D', 2.01410)],
    'C': [('13C', 13.00335)],
    'N': [('15N', 15.00011)],
    'O': [('18O', 17.99916)],
    'S': [('33S', 32.97146), ('34S', 33.96787)],
    'Cl': [('37Cl', 36.96590)],
    'Br': [('81Br', 80.91629)],
    'Si': [('29Si', 28.97649), ('30Si', 29.97377)]
}

def calculate_dynamic_cores(num_isomers):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_mb = mem_gb * 1024
    
    max_jobs_by_mem = max(1, int(mem_mb / 500))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 5.1: ISOTOPOLOGUES    {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    # 1. Directory Integrity Check
    if not os.path.exists(INPUT_DIR):
        os.makedirs(INPUT_DIR)
        print(f"\n{Colors.WARNING}{Colors.BOLD}🚨 ATTENTION REQUIRED 🚨{Colors.ENDC}")
        print(f"Created '{INPUT_DIR}' directory.")
        print("Please place the target `.xyz` ensemble(s) into this folder, then re-run this stage.")
        sys.exit(0)
        
    xyz_files = glob.glob(os.path.join(INPUT_DIR, "*.xyz"))
    if not xyz_files:
        print(f"\n{Colors.WARNING}{Colors.BOLD}🚨 DIRECTORY EMPTY 🚨{Colors.ENDC}")
        print(f"No '.xyz' files found in '{INPUT_DIR}'. Please add them and re-run.")
        sys.exit(0)
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)
    
    ingested_files = user_config.get("ingested_files", [])

    for ensemble_file in xyz_files:
        base_name = os.path.basename(ensemble_file).replace('.xyz', '').replace('_Final_Ensemble', '')
        
        # Recover routing flag for this specific molecule
        routing_flag = "ORCA_CPU"
        for item in ingested_files:
            if item.get("file", "").replace('.xyz', '') == base_name:
                routing_flag = item.get("pyscf_v_orca", "ORCA_CPU")
        
        print(f"\n{Colors.BOLD}--- Processing Isotopologues: {base_name} ---{Colors.ENDC}")
        isomers = read(ensemble_file, index=':')
        
        work_dir = f"Stage5_1_Isotopologues_{base_name}"
        if not os.path.exists(work_dir): os.makedirs(work_dir)

        # ---------------------------------------------------------
        # PHASE 1: Automated Combinatorial Substitution
        # ---------------------------------------------------------
        print_status("Generating exact mass variants based on Terrestrial Abundance parameters...", "info")
        all_variants = []
        for parent_idx, iso in enumerate(isomers):
            # Track Parent Reference
            parent = iso.copy()
            parent.info['iso_label'] = "Parent"
            parent.info['parent_num'] = parent_idx + 1
            all_variants.append(parent)

            # Generate Single Substitutions
            for atom_idx, atom in enumerate(iso):
                sym = atom.symbol
                if sym in ISOTOPE_RULES:
                    for iso_sym, iso_mass in ISOTOPE_RULES[sym]:
                        variant = iso.copy()
                        masses = variant.get_masses()
                        masses[atom_idx] = iso_mass
                        variant.set_masses(masses)
                        
                        # Apply [Atom#-Isotope] naming explicitly required by user
                        variant.info['iso_label'] = f"Atom{atom_idx+1}-{iso_sym}"
                        variant.info['parent_num'] = parent_idx + 1
                        all_variants.append(variant)
                        
        print_status(f"Generated {len(all_variants)} total structural isotopic combinations.", "success")
            
        concurrent_jobs, cores_per_job = calculate_dynamic_cores(len(all_variants))
        print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job", "info")

        # ---------------------------------------------------------
        # PHASE 2: Optimization Loop Protocol (Engine Managed)
        # ---------------------------------------------------------
        print(f"\n{Colors.BOLD}--- Phase 2: Parallel High-Accuracy Optimization Loop ---{Colors.ENDC}")
        
        # Execute Engine Protocol Loop natively from Stage 1.3
        # We ignore the returned list and read the outputs manually to flawlessly attach custom isotopic masses
        engine_high_acc_opt_loop(
            preopt_isomers=all_variants,
            base_name=base_name,
            work_dir=work_dir,
            routing_flag=routing_flag,
            charge=charge,
            mult=mult,
            concurrent_jobs=concurrent_jobs,
            cores_per_job=cores_per_job,
            sys_config=sys_config,
            stage_suffix="Stage5_1",
            job_prefix_mod="iso"
        )
        
        opt_isomers = []
        for idx, variant in enumerate(all_variants):
            if routing_flag == "PySCF_GPU":
                opt_file = os.path.join(work_dir, f"pyscf_iso_out_{idx}.xyz")
            else:
                opt_file = os.path.join(work_dir, f"{base_name}_iso_{idx}_OPT_FREQ.xyz")
            
            if os.path.exists(opt_file):
                try:
                    opt_iso = read(opt_file)
                    # Re-attach custom mass matrix and isotopologue labels
                    opt_iso.info.update(variant.info)
                    opt_iso.set_masses(variant.get_masses())
                    
                    if routing_flag == "ORCA_CPU":
                        out_file = os.path.join(work_dir, f"{base_name}_iso_{idx}.out")
                        opt_iso.info['energy'] = parse_energy(out_file)
                    
                    opt_isomers.append(opt_iso)
                except: pass

        # ---------------------------------------------------------
        # PHASE 3: Strict Topographic Funnel & Filter
        # ---------------------------------------------------------
        print(f"\n{Colors.BOLD}--- Phase 3: The Strict Topographic Funnel (ExtremeOpt Filter) ---{Colors.ENDC}")
        unique_pre_filter = the_crusher(opt_isomers, system_temperature)
        
        strict_isomers = []
        for variant in unique_pre_filter:
            iso_label = variant.info.get('iso_label', 'Unknown')
            parent_num = variant.info.get('parent_num', 1)
            
            job_prefix = f"{base_name}_strict_Iso{parent_num}_{iso_label}"
            inp_name = f"{job_prefix}.inp"
            
            inp_path = os.path.join(work_dir, inp_name)
            iso_path = os.path.join(work_dir, f"{job_prefix}.xyz")
            write(iso_path, variant)
            
            # Very strict variation of Stage 3 (xTB ExtremeOpt filter)
            with open(inp_path, "w") as f:
                f.write(f"! xTB2 ExtremeOpt\n") 
                f.write(f"%pal nprocs {cores_per_job} end\n")
                f.write(f"* xyzfile {charge} {mult} {os.path.basename(iso_path)}\n")
                
            out = run_orca_safely(f"Strict Filter {job_prefix}", inp_name, work_dir, sys_config, exit_on_fail=False)
            if out:
                try:
                    strict_iso = read(os.path.join(work_dir, f"{job_prefix}_ExtremeOpt.xyz"))
                    strict_iso.info['energy'] = parse_energy(out)
                    strict_iso.info.update(variant.info)
                    strict_iso.set_masses(variant.get_masses())
                    strict_isomers.append(strict_iso)
                except: pass

        final_unique = the_crusher(strict_isomers, system_temperature)

        # ---------------------------------------------------------
        # FINAL EXPORT PROTOCOL
        # ---------------------------------------------------------
        export_groups = defaultdict(list)
        
        # Map the unique geometries into distinct buckets based on their isotopic label 
        # so the Engine's exporter cleanly generates separate ledgers and accurate naming
        for f in final_unique:
            label = f.info.get('iso_label', 'Parent')
            p_num = f.info.get('parent_num', 1)
            custom_suffix = f"Iso{p_num}_{label}"
            export_groups[custom_suffix].append(f)

        print_status(f"Exporting {len(export_groups)} distinct isotopic ensembles...", "info")
        for custom_suffix, iso_list in export_groups.items():
            custom_base_name = f"{base_name}_{custom_suffix}"
            xyz_path, csv_path = export_thermo(
                iso_list, custom_base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage5_1"
            )
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 5.1 COMPLETE: High-Accuracy Isotopologues successfully generated.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 5.2: Isotopologue Triage

Purpose: Interactive 3D UI to verify the structural integrity of the generated Isotopologues.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 05_2-VISUAL-TRIAGE-ISOTOPOLOGUES-AA (Stage 5.2)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage provides the visual gatekeeper for the automated isotopologues 
# generated in Stage 5.1. It loads the high-accuracy isotopic ensembles, groups 
# them via the grouping crusher, and launches the final 3D Visual Triage Board 
# to allow the spectroscopist to verify the geometries before finalizing the ledger.
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment/VS Code after Stage 5.1 
# has finished. Select the specific isotopologue from the dropdown menu and click 
# "Launch Triage". Verify the 3D structures and click "Accept Selections" to 
# instantly generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI for the isotopologues. Upon acceptance, 
# successfully writes to the 'Isomer_xyz_Stage5_2_Verified' folder using the 
# Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage5_1_Isotopologues"
OUTPUT_DIR = "Isomer_xyz_Stage5_2_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class IsotopologueVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for Isotopologues.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            # Note: We enforce the specific [Atom#-Isotope] naming explicitly generated in 5.1
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage5_2_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Verified Isotopologue Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_isotopologue_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 5.2: ISOTOPOLOGUE TRIAGE{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 5.1.", "info")
        return

    # Specifically search for the output generated by Stage 5.1
    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage5_1_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 5.1 Isotopologue Ensembles found in {INPUT_DIR}.", "warning")
        return

    # Extract base names for the dropdown menu
    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage5_1_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    # Sort options to keep Atom#-Isotope labels grouped cleanly
    options.sort()

    dropdown = widgets.Dropdown(
        options=options,
        description='Isotopologue:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 5.1 Isotopologue to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value

            print_status(f"Loading isotopologue coordinates for {base_name}...", "info")
            
            try:
                # Read all isomers from the Stage 5.1 ensemble
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                # Pass them through the grouping crusher to array them for the UI
                print_status("Executing Grouping Crusher alignment on highly refined geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                # Instantiate the dynamically subclassed UI
                IsotopologueVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_isotopologue_triage()


🔓 Stage 5.3: Isotopologue Full Opt Extrap

Purpose: Executes fully relaxed Double-Hybrid extrapolations specifically for the Isotopologues.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 05_3-HIGH-LEVEL-ISOTOPOLOGUE-FULL-OPT-AA (Stage 5.3)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes a FULL (relaxed) High-Level Double-Hybrid Extrapolation 
# (revDSD-PBEP86-D4) specifically for unique Isotopologues. It loads the distinct 
# isotopic ensembles from Stage 5.1 / 5.2, symmetrizes them with MolSym, and 
# delegates the structural optimization to the Stage 1.3 Engine Protocol.
# 
# 2. Use instructions:
# Execute this script after completing Stage 5.2 (or Stage 5.1 if triage was skipped).
# It utilizes Checkpoint/Restart logic natively via the engine and dynamically 
# scales basis sets based on isomer count.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates ultra-high accuracy relaxed isotopologue geometries. Outputs 
# specialized Pickett files and exports the final Stage 5.3 Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import shutil
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_revdsd_opt_loop' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    apply_molsym_preopt = globals()['apply_molsym_preopt']
    engine_revdsd_opt_loop = globals()['engine_revdsd_opt_loop']
    estimate_memory = globals()['estimate_orca_memory']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
OUTPUT_DIR = "Isomer_xyz_Stage5_3_IsotopologuesFullOpt"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# DYNAMIC EXECUTION PROTOCOLS
# ==============================================================================

def calculate_dynamic_cores(num_isomers):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_per_job = 6000 
    
    max_jobs_by_mem = max(1, int((mem_gb * 1024) / mem_per_job))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

def determine_full_opt_extrapolation_level(base_size, num_isomers):
    """
    Bumps the size category by 1 for every 5 isomers to prevent timeframe explosions,
    then assigns the rigorous Double-Hybrid keywords requested. 
    """
    size_map = {"Small": 0, "Medium": 1, "Large": 2}
    inv_map = {0: "Small", 1: "Medium", 2: "Large"}
    
    base_idx = size_map.get(base_size, 0)
    bump = num_isomers // 5
    final_idx = min(2, base_idx + bump)
    final_size = inv_map[final_idx]
    
    if final_size == "Small":
        method = "revDSD-PBEP86-D4"
        basis = "aug-cc-pVTZ"
        header = f"! {method} {basis} aug-cc-pVTZ/JK aug-cc-pVTZ/C RIJCOSX ExtremeSCF TightOPT DEFGRID3\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
    elif final_size == "Medium":
        method = "revDSD-PBEP86-D4"
        basis = "cc-pVTZ"
        header = f"! {method} {basis} cc-pVTZ/JK cc-pVTZ/C RIJCOSX TightSCF OPT DEFGRID2\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
    else:
        method = "revDSD-PBEP86-D4"
        basis = "def2-TZVP"
        header = f"! {method} {basis} def2/J def2-TZVP/C RIJCOSX NormalSCF OPT DEFGRID2\n"
        
    return final_size, method, basis, header

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 5.3: ISOTOPOLOGUE FULL OPT{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        original_size = item.get("size", "Small")
        
        # 1. Dynamic Input Routing (Pulls from 5.2 if available, else 5.1)
        ensemble_files = glob.glob(os.path.join("Isomer_xyz_Stage5_2_Verified", f"Stage5_2_Verified_{base_name}_*_Final_Ensemble.xyz"))
        
        if os.path.exists("Isomer_xyz_Stage5_2_Verified") and ensemble_files:
            print_status(f"Discovered Stage 5.2 Verified Ensembles for {base_name}.", "success")
        else:
            ensemble_files = glob.glob(os.path.join("Isomer_xyz_Stage5_1_Isotopologues", f"Stage5_1_Isotopologues_{base_name}_*_Final_Ensemble.xyz"))
            if ensemble_files:
                print_status(f"Stage 5.2 missing. Falling back to Stage 5.1 Ensembles for {base_name}.", "info")
            
        if not ensemble_files:
            print_status(f"Missing Stage 5.1 and Stage 5.2 ensembles for {base_name}. Skipping.", "error")
            continue
            
        for ensemble_file in ensemble_files:
            # Extract the specific isotopologue label (e.g., Molecule_Iso1_Atom1-13C)
            prefix_to_remove = "Stage5_2_Verified_" if "Stage5_2" in ensemble_file else "Stage5_1_Isotopologues_"
            iso_base_name = os.path.basename(ensemble_file).replace(prefix_to_remove, "").replace("_Final_Ensemble.xyz", "")
            
            isomers = read(ensemble_file, index=':')
            num_isomers = len(isomers)
            
            print(f"\n{Colors.BOLD}--- Full Optimization of Isotopologue: {iso_base_name} ---{Colors.ENDC}")
            work_dir = f"Stage5_3_FullOpt_{iso_base_name}"
            if not os.path.exists(work_dir): os.makedirs(work_dir)
                
            # ---------------------------------------------------------
            # PHASE 1: Pre-Optimize with MolSym
            # ---------------------------------------------------------
            print_status("Phase 1: MolSym Symmetrization...", "info")
            preopt_isomers = []
            for idx, iso in enumerate(isomers):
                sym_iso = apply_molsym_preopt(iso, iso_base_name, idx, work_dir, prefix="molsym_fullopt_iso")
                preopt_isomers.append(sym_iso)

            # ---------------------------------------------------------
            # PHASE 2: Config & Execution
            # ---------------------------------------------------------
            final_size, method, basis, block_header = determine_full_opt_extrapolation_level(original_size, num_isomers)
            print_status(f"Isomer Count Scaling: {num_isomers} Isomers -> Shifted size from {original_size} to {Colors.WARNING}{final_size}{Colors.ENDC}", "info")
            print_status(f"Assigned Double Hybrid Method: {Colors.OKGREEN}{method} / {basis} (FULL OPT){Colors.ENDC}", "info")
            
            concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers)
            
            try:
                maxcore = estimate_memory(isomers[0], method=method, basis_set=basis)
            except Exception:
                maxcore = 3000
                
            print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job ({maxcore} MB maxcore)", "info")
            
            # Execute Engine Protocol Loop natively from Stage 1.3
            high_level_isomers = engine_revdsd_opt_loop(
                isomers=preopt_isomers,
                base_name=iso_base_name,
                work_dir=work_dir,
                block_header=block_header,
                maxcore=maxcore,
                charge=charge,
                mult=mult,
                concurrent_jobs=concurrent_jobs,
                cores_per_job=cores_per_job,
                sys_config=sys_config,
                stage_suffix="Stage5_3",
                job_prefix_mod="iso_fullopt"
            )

            # ---------------------------------------------------------
            # PHASE 3: Crusher & Final Export
            # ---------------------------------------------------------
            if high_level_isomers:
                # Preserve the custom isotopic mass configurations and labels for the export engine
                for orig_iso, hl_iso in zip(isomers, high_level_isomers):
                    hl_iso.info.update(orig_iso.info)
                    hl_iso.set_masses(orig_iso.get_masses())

                final_unique = the_crusher(high_level_isomers, system_temperature, silent=True)
                xyz_path, csv_path = export_thermo(
                    final_unique, iso_base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage5_3_FullOpt"
                )
                
                # Aggregate Pickett files securely to Output Directory
                print_status("Harvesting computed Pickett spectral parameter files...", "info")
                for pickett_file in glob.glob(os.path.join(work_dir, "*.pickett")):
                    shutil.copy(pickett_file, OUTPUT_DIR)
                    
            else:
                print_status(f"All calculations failed for {iso_base_name}. No inventory exported.", "error")
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 5.3 COMPLETE: Full Relaxed Isotopologue Optimization Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 5.4: Isotopologue Full Opt Triage

Purpose: Visual verification of the Double-Hybrid extrapolated Isotopologues.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 05_4-VISUAL-TRIAGE-ISOTOPOLOGUES-FULL-OPT-AA (Stage 5.3)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# Following the FULL (relaxed) High-Level Double-Hybrid Extrapolations 
# (revDSD-PBEP86-D4) for isotopologues in Stage 5.3, this stage provides 
# the absolute final visual verification. It loads the fully relaxed isotopic 
# ensembles, groups them via the grouping crusher, and launches the 3D Visual 
# Triage Board to lock in the true global minima for each isotopologue.
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment/VS Code after Stage 5.3 
# has finished. Select the specific isotopologue from the dropdown menu and click 
# "Launch Triage". Verify the 3D structures and click "Accept Selections" to 
# instantly generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI for the isotopologues. Upon acceptance, 
# successfully writes to the 'Isomer_xyz_Stage5_4_Verified' folder using the 
# Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage5_3_IsotopologuesFullOpt"
OUTPUT_DIR = "Isomer_xyz_Stage5_4_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class IsotopologueFullOptVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for Fully Optimized Isotopologues.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            # Note: The custom isotopic labels and masses are preserved in the Atoms info object
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage5_4_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Gold Standard Isotopologue Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_isotopologue_full_opt_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 5.4: ISOTOPOLOGUE FULL OPT TRIAGE{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 5.3.", "info")
        return

    # Specifically search for the output generated by Stage 5.3
    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage5_3_FullOpt_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 5.3 Full Opt Isotopologue Ensembles found in {INPUT_DIR}.", "warning")
        return

    # Extract base names for the dropdown menu
    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage5_3_FullOpt_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    # Sort options to keep Atom#-Isotope labels grouped cleanly
    options.sort()

    dropdown = widgets.Dropdown(
        options=options,
        description='Isotopologue:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 5.3 Fully Optimized Isotopologue to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value

            print_status(f"Loading fully relaxed isotopologue coordinates for {base_name}...", "info")
            
            try:
                # Read all isomers from the Stage 5.3 ensemble
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                # Pass them through the grouping crusher to array them for the UI
                print_status("Executing Grouping Crusher alignment on highly refined geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                # Instantiate the dynamically subclassed UI
                IsotopologueFullOptVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_isotopologue_full_opt_triage()


👑 Stage 5.5: Ultimate Isotopologue Extrap

Purpose: Executes ultimate CCSD(T) extrapolations to secure Isotopologue global minima.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 05_5-ULTIMATE-ISOTOPOLOGUE-EXTRAP-AA (Stage 5.4)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage executes the ultimate High-Level Coupled-Cluster Extrapolations 
# (CCSD(T) / DLPNO-CCSD(T)) specifically for the verified Isotopologues. It loads 
# the distinct isotopic ensembles from Stage 5.4 (or 5.2), symmetrizes them with 
# MolSym, and delegates the rigorous structural optimization to the Stage 1.3 
# Engine Protocol.
# 
# 2. Use instructions:
# Execute this script after completing Stage 5.4 (or Stage 5.2 if triage was skipped).
# It utilizes Checkpoint/Restart logic natively via the engine and dynamically 
# scales basis sets based on isomer count.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates "Gold Standard" ultra-high accuracy isotopologue geometries. 
# Outputs specialized Pickett files and exports the final Stage 5.5 Inventory.
# ==============================================================================

import os
import sys
import json
import glob
import shutil
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'the_crusher' in globals() and 'engine_ccsdt_opt_loop' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    the_crusher = globals()['the_crusher']
    export_thermo = globals()['export_thermodynamic_inventory']
    apply_molsym_preopt = globals()['apply_molsym_preopt']
    engine_ccsdt_opt_loop = globals()['engine_ccsdt_opt_loop']
    estimate_memory = globals()['estimate_orca_memory']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 or 2 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 and Stage 1.3 cells first.")
    sys.exit(1)

try:
    from ase.io import read, write
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
OUTPUT_DIR = "Isomer_xyz_Stage5_5_UltimateIsotopologues"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# DYNAMIC EXECUTION PROTOCOLS
# ==============================================================================

def calculate_dynamic_cores(num_isomers, is_ccsd=False):
    cpu_threads = sys_config.get("hardware", {}).get("cpu_threads", 4)
    mem_gb = sys_config.get("hardware", {}).get("memory_gb", 8.0)
    
    hyperthreads_half = max(1, cpu_threads // 2)
    mem_per_job = 8000 if is_ccsd else 4000 
    
    max_jobs_by_mem = max(1, int((mem_gb * 1024) / mem_per_job))
    target_concurrent = min(hyperthreads_half, max_jobs_by_mem)
    
    if num_isomers >= target_concurrent:
        concurrent_jobs = target_concurrent
        cores_per_job = 1
    else:
        concurrent_jobs = num_isomers
        cores_per_job = max(1, target_concurrent // max(1, concurrent_jobs))
        
    return concurrent_jobs, cores_per_job

def determine_ultimate_extrapolation_level(base_size, num_isomers):
    """
    Bumps the size category by 1 for every 5 isomers to prevent timeframe explosions,
    then assigns the rigorous Stage 8 keywords. 
    """
    size_map = {"Small": 0, "Medium": 1, "Large": 2}
    inv_map = {0: "Small", 1: "Medium", 2: "Large"}
    
    base_idx = size_map.get(base_size, 0)
    bump = num_isomers // 5
    final_idx = min(2, base_idx + bump)
    final_size = inv_map[final_idx]
    
    if final_size == "Small":
        method = "CCSD(T)"
        basis = "aug-cc-pwCVQZ"
        header = f"! {method} {basis} TightOPT AutoAux ExtremeSCF DEFGRID3 UseSym\n"
        header += "%method\n  Z_Tol 1e-14;\n  FrozenCore FC_NONE;\nend\n"
        header += "%geom\n  TolE 1e-9;\n  TolRMSG 3e-6;\n  TolMaxG 1e-5;\nend\n"
    elif final_size == "Medium":
        method = "DLPNO-CCSD(T)"
        basis = "aug-cc-pVTZ"
        header = f"! {method} TightPNO {basis} TightOPT AutoAux ExtremeSCF DEFGRID3 UseSym\n"
        header += "%method\n  Z_Tol 1e-14;\nend\n"
        header += "%geom\n  TolE 1e-9;\n  TolRMSG 3e-6;\n  TolMaxG 1e-5;\nend\n"
    else:
        method = "DLPNO-CCSD(T)"
        basis = "cc-pVTZ"
        header = f"! {method} NormalPNO {basis} OPT AutoAux TightSCF DEFGRID2 UseSym\n"
        
    return final_size, method, basis, header

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 5.5: ULTIMATE ISOTOPOLOGUES{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    system_temperature = user_config.get("temperature", 298.15)
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)

    ingested_files = user_config.get("ingested_files", [])
    
    for item in ingested_files:
        base_name = item.get("file").replace('.xyz', '')
        original_size = item.get("size", "Small")
        
        # 1. Dynamic Input Routing (Pulls from 5.4 if available, else 5.2)
        ensemble_files = glob.glob(os.path.join("Isomer_xyz_Stage5_4_Verified", f"Stage5_4_Verified_{base_name}_*_Final_Ensemble.xyz"))
        
        if os.path.exists("Isomer_xyz_Stage5_4_Verified") and ensemble_files:
            print_status(f"Discovered Stage 5.4 Verified Ensembles for {base_name}.", "success")
        else:
            ensemble_files = glob.glob(os.path.join("Isomer_xyz_Stage5_2_Verified", f"Stage5_2_Verified_{base_name}_*_Final_Ensemble.xyz"))
            if ensemble_files:
                print_status(f"Stage 5.4 missing. Falling back to Stage 5.2 Ensembles for {base_name}.", "info")
            
        if not ensemble_files:
            print_status(f"Missing Stage 5.2 and Stage 5.4 ensembles for {base_name}. Skipping.", "error")
            continue
            
        for ensemble_file in ensemble_files:
            # Extract the specific isotopologue label (e.g., Molecule_Iso1_Atom1-13C)
            prefix_to_remove = "Stage5_4_Verified_" if "Stage5_4" in ensemble_file else "Stage5_2_Verified_"
            iso_base_name = os.path.basename(ensemble_file).replace(prefix_to_remove, "").replace("_Final_Ensemble.xyz", "")
            
            isomers = read(ensemble_file, index=':')
            num_isomers = len(isomers)
            
            print(f"\n{Colors.BOLD}--- Coupled-Cluster Extrapolation: {iso_base_name} ---{Colors.ENDC}")
            work_dir = f"Stage5_5_Ultimate_{iso_base_name}"
            if not os.path.exists(work_dir): os.makedirs(work_dir)
                
            # ---------------------------------------------------------
            # PHASE 1: Pre-Optimize with MolSym
            # ---------------------------------------------------------
            print_status("Phase 1: MolSym Symmetrization...", "info")
            preopt_isomers = []
            for idx, iso in enumerate(isomers):
                sym_iso = apply_molsym_preopt(iso, iso_base_name, idx, work_dir, prefix="molsym_cc_iso")
                preopt_isomers.append(sym_iso)

            # ---------------------------------------------------------
            # PHASE 2: Config & Execution
            # ---------------------------------------------------------
            final_size, method, basis, block_header = determine_ultimate_extrapolation_level(original_size, num_isomers)
            print_status(f"Isomer Count Scaling: {num_isomers} Isomers -> Shifted size from {original_size} to {Colors.WARNING}{final_size}{Colors.ENDC}", "info")
            print_status(f"Assigned Coupled-Cluster Method: {Colors.OKGREEN}{method} / {basis}{Colors.ENDC}", "info")
            
            is_ccsd = "DLPNO" not in method
            concurrent_jobs, cores_per_job = calculate_dynamic_cores(num_isomers, is_ccsd)
            
            try:
                maxcore = estimate_memory(isomers[0], method=method, basis_set=basis)
            except Exception:
                maxcore = 3000
                
            print_status(f"Resource Matrix: {concurrent_jobs} concurrent jobs @ {cores_per_job} cores/job ({maxcore} MB maxcore)", "info")
            
            # Execute Engine Protocol Loop natively from Stage 1.3 (requires_freq=False per instructions)
            high_level_isomers = engine_ccsdt_opt_loop(
                isomers=preopt_isomers,
                base_name=iso_base_name,
                work_dir=work_dir,
                block_header=block_header,
                maxcore=maxcore,
                charge=charge,
                mult=mult,
                concurrent_jobs=concurrent_jobs,
                cores_per_job=cores_per_job,
                sys_config=sys_config,
                stage_suffix="Stage5_5",
                job_prefix_mod="iso_ccsd",
                requires_freq=False
            )

            # ---------------------------------------------------------
            # PHASE 3: Crusher & Final Export
            # ---------------------------------------------------------
            if high_level_isomers:
                # Preserve the custom isotopic mass configurations and labels for the export engine
                for orig_iso, hl_iso in zip(isomers, high_level_isomers):
                    hl_iso.info.update(orig_iso.info)
                    hl_iso.set_masses(orig_iso.get_masses())

                final_unique = the_crusher(high_level_isomers, system_temperature, silent=True)
                xyz_path, csv_path = export_thermo(
                    final_unique, iso_base_name, system_temperature, OUTPUT_DIR, stage_prefix="Stage5_5_Ultimate"
                )
                
                # Aggregate Pickett files securely to Output Directory
                print_status("Harvesting computed Pickett spectral parameter files...", "info")
                for pickett_file in glob.glob(os.path.join(work_dir, "*.pickett")):
                    shutil.copy(pickett_file, OUTPUT_DIR)
                    
            else:
                print_status(f"All calculations failed for {iso_base_name}. No inventory exported.", "error")
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 5.5 COMPLETE: Ultimate Coupled-Cluster Isotopologue Optimization Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


👁️ Stage 5.6: Ultimate Isotopologue Triage

Purpose: Final visual triage for "Gold Standard" CCSD(T) Isotopologues.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 05_6-VISUAL-TRIAGE-ISOTOPOLOGUES-ULTIMATE-AA (Stage 5.6)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# Following the ultimate High-Level Coupled-Cluster Extrapolations (CCSD(T) / 
# DLPNO-CCSD(T)) for Isotopologues in Stage 5.5, this stage provides the absolute 
# final visual verification. It loads the Gold Standard complex ensembles, 
# groups them via the grouping crusher, and launches the final 3D Visual Triage 
# Board to lock in the true global minima.
# 
# 2. Use instructions:
# Execute this cell in your active Jupyter environment/VS Code after Stage 5.5 has 
# finished. Select your specific isotopologue from the dropdown menu and click 
# "Launch Triage". Verify the 3D structures and click "Accept Selections" to 
# instantly generate the final Verified XYZ Ensemble and a formatted CSV Inventory.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates a highly interactive UI. Upon acceptance, successfully writes 
# to the 'Isomer_xyz_Stage5_6_Verified' folder using the Inventory Export Protocol.
# ==============================================================================

import os
import sys
import json
import glob
import numpy as np

try:
    from ase.io import read
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("❌ ERROR: Required libraries (ASE, ipywidgets) not found. Run Stage 0.")
    sys.exit(1)

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'VisualIsomerTriage' in globals() and 'export_thermodynamic_inventory' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded UI protocols.")
    VisualIsomerTriage = globals()['VisualIsomerTriage']
    export_thermo = globals()['export_thermodynamic_inventory']
    the_grouping_crusher = globals()['the_grouping_crusher']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 2 missing from memory.")
    print("ACTION REQUIRED: Ensure you have executed the Stage 1.3 cell first to load the UI protocols.")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Isomer_xyz_Stage5_5_UltimateIsotopologues"
OUTPUT_DIR = "Isomer_xyz_Stage5_6_Verified"

# ==============================================================================
# DYNAMIC TRIAGE SUBCLASS
# ==============================================================================
class IsotopologueUltimateVerifiedTriage(VisualIsomerTriage):
    """
    Subclasses the Stage 1.3 VisualIsomerTriage to inject the Inventory Export 
    Protocol directly into the 'Accept Selections' state-machine for Ultimate 
    Coupled-Cluster Isotopologues.
    """
    def __init__(self, grouped_isomers, base_name, temperature, output_dir):
        self.temperature = temperature
        self.output_dir = output_dir
        super().__init__(grouped_isomers, base_name)
        
    def on_accept(self, btn):
        super().on_accept(btn)
        final_reps = [g[0] for g in self.groups]
        with self.out:
            xyz_path, csv_path = export_thermo(
                final_unique=final_reps, 
                base_name=self.base_name, 
                temperature=self.temperature, 
                output_dir=self.output_dir, 
                stage_prefix="Stage5_6_Verified"
            )
            print(f"\n{Colors.OKBLUE}--- Ultimate Gold Standard Isotopologue Inventory Export Complete ---{Colors.ENDC}")
            print(f" • Coordinate Data: {xyz_path}")
            print(f" • Tabular Ledger:  {csv_path}")

# ==============================================================================
# INTERACTIVE LAUNCHER
# ==============================================================================
def launch_isotopologue_ultimate_triage():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 5.6: ULTIMATE ISOTOPOLOGUE TRIAGE{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(USER_CONFIG):
        print_status("Configuration file missing. Ensure Stage 1.1 is complete.", "error")
        return

    with open(USER_CONFIG, 'r') as f: user_config = json.load(f)
    temperature = user_config.get("temperature", 298.15)

    if not os.path.exists(INPUT_DIR):
        print_status(f"Input directory '{INPUT_DIR}' not found.", "warning")
        print_status("Bypass Logic: Your system likely did not qualify for Stage 5.5.", "info")
        return

    # Specifically search for the output generated by Stage 5.5 Ultimate Isotopologue Extrapolation
    ensemble_files = glob.glob(os.path.join(INPUT_DIR, "Stage5_5_Ultimate_*_Final_Ensemble.xyz"))
    if not ensemble_files:
        print_status(f"No Stage 5.5 Ultimate Ensembles found in {INPUT_DIR}.", "warning")
        return

    # Extract base names for the dropdown menu
    options = []
    for f in ensemble_files:
        base_name = os.path.basename(f).replace("Stage5_5_Ultimate_", "").replace("_Final_Ensemble.xyz", "")
        options.append((base_name, f))

    # Sort options to keep Atom#-Isotope labels grouped cleanly
    options.sort()

    dropdown = widgets.Dropdown(
        options=options,
        description='Isotopologue:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    
    launch_btn = widgets.Button(description='Launch Triage', button_style='info', icon='eye')
    
    ui_container = widgets.VBox([
        widgets.HTML("<b>Select a Stage 5.5 Ultimate Isotopologue to Verify:</b>"),
        widgets.HBox([dropdown, launch_btn])
    ])
    
    triage_output = widgets.Output()
    
    def on_launch_clicked(b):
        with triage_output:
            clear_output(wait=True)
            base_name, file_path = dropdown.value
            
            print_status(f"Loading Gold Standard isotopologue coordinates for {base_name}...", "info")
            
            try:
                # Read all isomers from the Stage 5.5 ensemble
                isomers = read(file_path, index=':')
                if not isomers:
                    print_status("Failed to read isomers from file.", "error")
                    return
                
                # Pass them through the grouping crusher to array them for the UI
                print_status("Executing Grouping Crusher alignment on highly refined geometries...", "info")
                grouped_isomers = the_grouping_crusher(isomers, temperature, silent=True)
                
                print_status("Initializing 3D Visual Triage Board...", "success")
                # Instantiate the dynamically subclassed UI
                IsotopologueUltimateVerifiedTriage(grouped_isomers, base_name, temperature, OUTPUT_DIR)
                
            except Exception as e:
                print_status(f"Failed to launch triage: {e}", "error")

    launch_btn.on_click(on_launch_clicked)
    display(ui_container, triage_output)

if __name__ == "__main__":
    launch_isotopologue_ultimate_triage()


💧 Stage 7: Solvation

Purpose: Implements advanced Implicit (CPCM) and Explicit (MACE-MD + CCSD(T) Binding) Solvation modeling.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 07-SOLVATION-AA (Stage 7)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage implements advanced Implicit and Explicit solvation modeling. 
# For Implicit solvation, it dynamically constructs CPCM queries based on ORCA 6.1.1.
# For Explicit solvation, it merges the solute and solvent, leverages MACE-OFF23 
# (via ASE Langevin Dynamics) on the GPU to perform rapid liquid-state MD, 
# explicitly extracts the solvent shell cluster, optimizes the local minima 
# (via PySCF-GPU or r²SCAN-3c), and performs scaled High-Accuracy CC energetics.
# 
# 2. Use instructions:
# Execute this script directly. If the `Solvent_System` folder is empty, it will 
# halt and prompt you to add your `.xyz` files. Once added, interact with the 
# terminal prompts to select your solvent modeling strategy.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Generates optimized solvated clusters or implicit models. Checkpoints 
# are natively respected. Pickett files and highly accurate solvent binding 
# energies are produced.
# ==============================================================================

import os
import sys
import json
import glob
import subprocess
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'run_orca_subprocess_safely' in globals():
    print("➡️ Active memory engine detected. Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    parse_energy = globals()['parse_orca_energy']
    estimate_memory = globals()['estimate_orca_memory']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
else:
    print(f"❌ ERROR: Engine Part 1 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 cell first.")
    sys.exit(1)

try:
    from ase.io import read, write
    from ase import Atoms
    from ase.neighborlist import natural_cutoffs, NeighborList
    import networkx as nx
except ImportError:
    print_status("Required libraries (ASE, NetworkX) not found. Run Stage 0.", "error")
    sys.exit(1)

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Solvent_System"
OUTPUT_DIR = "Solvent_Stage7_Outputs"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# DYNAMIC SOLVATION PROTOCOLS
# ==============================================================================

def execute_pyscf_gpu_solvation_opt(xyz_file, out_xyz, charge, spin_mult):
    """Executes a PySCF GPU process to optimize the explicit solvent cluster."""
    spin_val = spin_mult - 1
    script_content = f"""
import sys
from pyscf import gto, dft
try:
    from pyscf.geomopt.geometric_solver import optimize
except ImportError:
    print("geomeTRIC optimizer not installed. PySCF geometry optimization aborted.")
    sys.exit(1)

mol = gto.M(atom='{xyz_file}', basis='def2-tzvp', charge={charge}, spin={spin_val})
mol.build()

mf = dft.RKS(mol) if {spin_val} == 0 else dft.UKS(mol)
mf.xc = 'wB97X-D4'

try:
    import gpu4pyscf
    mf = mf.to_gpu()
except ImportError: pass

try:
    mol_eq = optimize(mf, maxsteps=200)
    mol_eq.tofile('{out_xyz}')
except Exception as e:
    print(f"PySCF Optimization Failed: {{e}}")
    sys.exit(1)
"""
    script_name = f"pyscf_solvent_runner_{os.path.basename(xyz_file)}.py"
    with open(script_name, "w") as f: f.write(script_content)
    res = subprocess.run([sys.executable, script_name], capture_output=True, text=True)
    if os.path.exists(script_name): os.remove(script_name)
    return res.returncode == 0 and os.path.exists(out_xyz)

def extract_solvent_shell(cluster, solute_len, shell_radius=4.5):
    """Uses ASE graphing to extract only solvent molecules within the defined vdW shell radius."""
    solute_indices = list(range(solute_len))
    
    cutoffs = [c * 1.2 for c in natural_cutoffs(cluster)]
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(cluster)
    matrix = nl.get_connectivity_matrix()
    graph = nx.from_scipy_sparse_array(matrix)
    
    components = list(nx.connected_components(graph))
    solute_comps = set()
    solvent_comps = []
    
    for comp in components:
        if any(idx in solute_indices for idx in comp):
            solute_comps.update(comp)
        else:
            solvent_comps.append(comp)
            
    extracted_indices = list(solute_comps)
    pos = cluster.get_positions()
    solute_pos = pos[list(solute_comps)]
    
    for comp in solvent_comps:
        comp_pos = pos[list(comp)]
        # Calculate minimum distance from any atom in this solvent molecule to any solute atom
        dist_matrix = np.linalg.norm(comp_pos[:, np.newaxis, :] - solute_pos[np.newaxis, :, :], axis=-1)
        if np.min(dist_matrix) < shell_radius:
            extracted_indices.extend(comp)
            
    return cluster[extracted_indices]

def run_explicit_mace_md(solute_atoms, solvent_atoms):
    """Merges solute and solvent, relaxes structural clashes, and executes MACE-MD."""
    print_status("Merging solute and explicit solvent matrices...", "info")
    solvent_atoms.center()
    solvent_atoms.translate(solute_atoms.get_center_of_mass() - solvent_atoms.get_center_of_mass())
    cluster = solute_atoms + solvent_atoms
    
    try:
        from mace.calculators import mace_mp
        calc = mace_mp(model="small", device="cuda", default_dtype="float32")
        cluster.calc = calc
        print_status("MACE-OFF23 GPU Calculator engaged for Liquid-State MD.", "success")
    except Exception as e:
        print_status(f"MACE not available on GPU: {e}. Falling back to xTB...", "warning")

    print_status("Phase 1: Repulsive Clash Relaxation (LBFGS)...", "info")
    from ase.optimize import LBFGS
    opt = LBFGS(cluster, logfile=None)
    opt.run(fmax=1.0, steps=75)
    
    print_status("Phase 2: Langevin Molecular Dynamics (298.15 K, 200 fs)...", "info")
    from ase.md.langevin import Langevin
    from ase import units
    dyn = Langevin(cluster, 1.0 * units.fs, temperature_K=298.15, friction=0.01, logfile=None)
    dyn.run(200)
    
    return cluster

def determine_sp_scaling(num_atoms):
    """Gracefully scales the Coupled-Cluster execution based on the extracted shell size."""
    if num_atoms < 30:
        return "! CCSD(T) aug-cc-pVTZ AutoAux TightSCF DEFGRID3", 8000
    elif num_atoms <= 80:
        return "! DLPNO-CCSD(T) aug-cc-pVTZ TightPNO AutoAux TightSCF DEFGRID3", 5000
    else:
        return "! DLPNO-CCSD(T) cc-pVTZ NormalPNO AutoAux NormalSCF DEFGRID2", 4000

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 7: SOLVATION SYSTEMS  {Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    # 1. Directory Integrity Check
    if not os.path.exists(INPUT_DIR):
        os.makedirs(INPUT_DIR)
        print(f"\n{Colors.WARNING}{Colors.BOLD}🚨 DIRECTORY CREATED 🚨{Colors.ENDC}")
        print(f"Created '{INPUT_DIR}' directory.")
        print("Please place your `molecule.xyz` file(s) into this folder.")
        print("If using explicit solvation, you must ALSO place a file named exactly `solvent.xyz` here.")
        sys.exit(0)
        
    xyz_files = glob.glob(os.path.join(INPUT_DIR, "*.xyz"))
    if not xyz_files:
        print(f"\n{Colors.WARNING}{Colors.BOLD}🚨 DIRECTORY EMPTY 🚨{Colors.ENDC}")
        print(f"No '.xyz' files found in '{INPUT_DIR}'. Please add them and re-run.")
        sys.exit(0)
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)
    cores = sys_config.get("hardware", {}).get("cpu_threads", 4)
    ingested_files = user_config.get("ingested_files", [])

    # 2. Interactive Terminal Query
    print(f"\n{Colors.OKCYAN}--- Solvation Configuration ---{Colors.ENDC}")
    solv_type = input("Implicit or Explicit solvation? (I/E): ").strip().upper()

    # ==========================================================================
    # PATH A: IMPLICIT SOLVATION
    # ==========================================================================
    if solv_type == 'I':
        print(f"\n{Colors.OKBLUE}Documentation Reference: https://www.faccts.de/docs/orca/6.1/tutorials/prop/cpcm.html{Colors.ENDC}")
        print_status("Remind to check the ORCA manual for method/solvent compatibilities.", "info")
        
        solvent_name = input("What solvent? (e.g., Water, CH2Cl2): ").strip()
        method_name = input("Method to use? (e.g., r2SCAN-3c, wB97X-D4 def2-TZVP): ").strip()
        
        for f in xyz_files:
            if os.path.basename(f).lower() == "solvent.xyz": continue
            
            base_name = os.path.basename(f).replace('.xyz', '')
            print(f"\n{Colors.BOLD}--- Executing Implicit Solvation: {base_name} ---{Colors.ENDC}")
            
            work_dir = f"Stage7_Implicit_{base_name}"
            if not os.path.exists(work_dir): os.makedirs(work_dir)
            
            job_name = f"{base_name}_Implicit_{solvent_name}"
            inp_path = os.path.join(work_dir, f"{job_name}.inp")
            out_path = os.path.join(work_dir, f"{job_name}.out")
            pickett_name = f"{base_name}_{solvent_name}_Stage7.pickett"
            
            # Checkpoint recovery
            if os.path.exists(out_path):
                with open(out_path, 'r') as check_f:
                    if "ORCA TERMINATED NORMALLY" in check_f.read():
                        print_status(f"Implicit optimization already complete. Skipping.", "success")
                        continue
            
            shutil.copy(f, os.path.join(work_dir, f"{job_name}.xyz"))
            
            with open(inp_path, "w") as inp_f:
                inp_f.write(f"! {method_name} CPCM({solvent_name}) OPT FREQ\n")
                inp_f.write(f"%pal nprocs {cores} end\n")
                inp_f.write(f"%output\n  Pickettname \"{pickett_name}\"\nend\n")
                inp_f.write(f"* xyzfile {charge} {mult} {job_name}.xyz\n")
                
            run_orca_safely(f"Implicit Optimization ({solvent_name})", f"{job_name}.inp", work_dir, sys_config, exit_on_fail=False)

    # ==========================================================================
    # PATH B: EXPLICIT SOLVATION
    # ==========================================================================
    elif solv_type == 'E':
        print(f"\n{Colors.OKBLUE}Documentation Reference: https://www.faccts.de/docs/orca/6.1/tutorials/prop/solvator.html{Colors.ENDC}")
        
        solvent_file = os.path.join(INPUT_DIR, "solvent.xyz")
        if not os.path.exists(solvent_file):
            print_status(f"Missing exactly named `solvent.xyz` inside `{INPUT_DIR}`.", "error")
            print("Please place the initial solvent box/cluster file and re-run.")
            sys.exit(0)
            
        solvent_atoms = read(solvent_file)

        for f in xyz_files:
            if os.path.basename(f).lower() == "solvent.xyz": continue
            
            base_name = os.path.basename(f).replace('.xyz', '')
            print(f"\n{Colors.BOLD}--- Executing Explicit Solvation: {base_name} ---{Colors.ENDC}")
            
            routing_flag = "ORCA_CPU"
            for item in ingested_files:
                if item.get("file", "").replace('.xyz', '') == base_name:
                    routing_flag = item.get("pyscf_v_orca", "ORCA_CPU")
            
            work_dir = f"Stage7_Explicit_{base_name}"
            if not os.path.exists(work_dir): os.makedirs(work_dir)
            
            # Step 1: Sampling & MD Shell Extraction
            shell_xyz_path = os.path.join(work_dir, f"{base_name}_Explicit_Shell.xyz")
            if not os.path.exists(shell_xyz_path):
                solute_atoms = read(f)
                solute_len = len(solute_atoms)
                
                md_cluster = run_explicit_mace_md(solute_atoms, solvent_atoms)
                print_status("MD Complete. Extracting inner solvent shell cluster...", "info")
                shell_cluster = extract_solvent_shell(md_cluster, solute_len, shell_radius=4.5)
                write(shell_xyz_path, shell_cluster)
                print_status(f"Solvent shell extracted ({len(shell_cluster)} atoms total).", "success")
            else:
                shell_cluster = read(shell_xyz_path)
                print_status("Checkpointed solvent shell cluster found.", "success")

            # Step 2: High-Accuracy Cluster Optimization
            opt_xyz_path = os.path.join(work_dir, f"{base_name}_Explicit_OPT.xyz")
            if not os.path.exists(opt_xyz_path):
                print_status(f"Optimizing cluster via hardware router ({routing_flag})...", "info")
                
                if routing_flag == "PySCF_GPU":
                    execute_pyscf_gpu_solvation_opt(shell_xyz_path, opt_xyz_path, charge, mult)
                else:
                    inp_name = f"{base_name}_Explicit_OPT.inp"
                    pickett_name = f"{base_name}_Explicit_Stage7.pickett"
                    with open(os.path.join(work_dir, inp_name), "w") as inp_f:
                        inp_f.write(f"! r2SCAN-3c OPT FREQ\n")
                        inp_f.write(f"%pal nprocs {cores} end\n")
                        inp_f.write(f"%output\n  Pickettname \"{pickett_name}\"\nend\n")
                        inp_f.write(f"* xyzfile {charge} {mult} {os.path.basename(shell_xyz_path)}\n")
                    run_orca_safely(f"Explicit Cluster Opt", inp_name, work_dir, sys_config, exit_on_fail=False)
            else:
                print_status("Checkpointed optimized cluster found.", "success")
                
            # Step 3: High-Accuracy Energetics (Coupled-Cluster Binding)
            sp_inp_name = f"{base_name}_Explicit_SP.inp"
            sp_out_name = f"{base_name}_Explicit_SP.out"
            sp_out_path = os.path.join(work_dir, sp_out_name)
            
            if os.path.exists(sp_out_path):
                with open(sp_out_path, 'r') as check_f:
                    if "ORCA TERMINATED NORMALLY" in check_f.read():
                        print_status("Coupled-Cluster SP already complete. Skipping.", "success")
                        continue
                        
            opt_atoms = read(opt_xyz_path)
            num_cluster_atoms = len(opt_atoms)
            
            header, maxcore = determine_sp_scaling(num_cluster_atoms)
            print_status(f"Graceful Scaling Matrix applied for {num_cluster_atoms} atoms: {Colors.OKGREEN}{header}{Colors.ENDC}", "info")
            
            with open(os.path.join(work_dir, sp_inp_name), "w") as inp_f:
                inp_f.write(f"{header}\n")
                inp_f.write(f"%pal nprocs {cores} end\n%maxcore {maxcore}\n")
                inp_f.write(f"* xyzfile {charge} {mult} {os.path.basename(opt_xyz_path)}\n")
                
            run_orca_safely("Explicit High-Accuracy Energetics SP", sp_inp_name, work_dir, sys_config, exit_on_fail=False)
            
    else:
        print_status("Invalid selection. Please run the stage again and enter 'I' or 'E'.", "error")
        sys.exit(1)

    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 7 COMPLETE: Solvation Calculations Finished.{Colors.ENDC}")

if __name__ == "__main__":
    main()


📊 Stage 9: Spectroscopy Export

Purpose: Evaluates final VPT2 frequencies and dynamically generates highly specialized Pickett parameter files for SPCAT.

In [ ]:
#!/usr/bin/env python3

# ==============================================================================
# 09_SPECTROSCOPY_EXPORT_AA (Stage 9)
# ==============================================================================
# 
# 1. Detailed Purpose: 
# This stage generates the final spectroscopic outputs (Pickett files for SPCAT) 
# by performing rigorous VPT2 frequency extrapolations. It automatically packages 
# verified outputs from previous stages, parses their optimized geometries and 
# inheritance keywords, evaluates point group symmetries via MolSym, and routes 
# them through the high-accuracy ORCA execution loop.
# 
# 2. Use instructions:
# Execute this script directly. It will generate the 'Ini_Spec_Opt-xyz' folder 
# and prompt you to package data. Select your desired stages. Answer the symmetry 
# handling prompts, and the pipeline will generate the definitive Pickett files.
# 
# 3. Expected Execution & Success Criteria:
# SUCCESS: Parses all `.out` files, successfully detects C1 vs higher symmetries, 
# applies MolSym alignments if requested, strips optimization keywords to strictly 
# freeze the geometry, and generates `.pickett` files via ORCA VPT2.
# ==============================================================================

import os
import sys
import json
import glob
import shutil
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# ENGINE INJECTION (MEMORY REVERSION)
# ==============================================================================
if 'run_orca_subprocess_safely' in globals():
    print("➡️ Active memory engine detected (Jupyter). Utilizing loaded protocols.")
    run_orca_safely = globals()['run_orca_subprocess_safely']
    Colors = globals()['Colors']
    print_status = globals()['print_status']
    
    # Load MolSym Preopt if available from Stage 1.3
    if 'apply_molsym_preopt' in globals():
        apply_molsym_preopt = globals()['apply_molsym_preopt']
    else:
        def apply_molsym_preopt(atoms, *args, **kwargs): return atoms
else:
    print(f"❌ ERROR: Engine Part 1 missing from memory.")
    print("ACTION REQUIRED: Execute Stage 1.2 cell first.")
    sys.exit(1)

try:
    from ase.io import read, write
    from ase import Atoms
except ImportError:
    print_status("Required libraries (ASE) not found. Run Stage 0.", "error")
    sys.exit(1)

try:
    import molsym
    MOLSYM_AVAILABLE = True
except ImportError:
    MOLSYM_AVAILABLE = False

SYSTEM_CONFIG = "opi_system_config.json"
USER_CONFIG = "opi_user_config.json"
INPUT_DIR = "Ini_Spec_Opt-xyz"
OUTPUT_DIR = "OPI-ORCA_Spec"

with open(SYSTEM_CONFIG, 'r') as f: sys_config = json.load(f)
with open(USER_CONFIG, 'r') as f: user_config = json.load(f)

# ==============================================================================
# SYMMETRY & PARSING PROTOCOLS
# ==============================================================================

def get_point_group(atoms):
    """Detects the Point Group using the MolSym API."""
    if not MOLSYM_AVAILABLE: return "C1"
    temp_xyz = "temp_pg.xyz"
    write(temp_xyz, atoms)
    try:
        mol = molsym.Molecule.from_file(temp_xyz)
        pg = getattr(mol, 'pg', None)
        if not pg: pg = getattr(mol, 'point_group', "C1")
        if os.path.exists(temp_xyz): os.remove(temp_xyz)
        return str(pg)
    except Exception:
        if os.path.exists(temp_xyz): os.remove(temp_xyz)
        return "C1"

def parse_orca_out_for_spec(out_path):
    """Deep parses an ORCA .out file to extract the exact final geometry and inherited header block."""
    with open(out_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
        
    energy = 0.0
    header_block = ""
    capture_header = False
    temp_header_lines = []
    
    geom_lines = []
    capture_geom = False
    
    for line in lines:
        if "FINAL SINGLE POINT ENERGY" in line:
            try: energy = float(line.split()[4])
            except: pass
            
        if "INPUT FILE" in line:
            capture_header = True
            temp_header_lines = []
            continue
            
        if capture_header:
            clean_line = line.split(">", 1)[-1].strip() if ">" in line else line.strip()
            if clean_line.startswith("!") or clean_line.startswith("%"):
                temp_header_lines.append(clean_line)
            if clean_line.startswith("* xyz") or clean_line.startswith("*xyz"):
                capture_header = False
                header_block = "\n".join(temp_header_lines)
                continue
                
        if "CARTESIAN COORDINATES (ANGSTROEM)" in line:
            capture_geom = True
            geom_lines = []
            continue
            
        if capture_geom:
            if "------" in line: continue
            if line.strip() == "":
                capture_geom = False
                continue
            geom_lines.append(line.strip())
            
    symbols = []
    positions = []
    for gl in geom_lines:
        parts = gl.split()
        if len(parts) >= 4:
            symbols.append(parts[0])
            positions.append([float(parts[1]), float(parts[2]), float(parts[3])])
            
    atoms = Atoms(symbols=symbols, positions=positions) if symbols else None
    
    clean_header = []
    for hl in header_block.split('\n'):
        if hl.startswith("!"):
            # Strip Optimization keywords to enforce Frozen Geometry VPT2 rules
            hl = hl.replace("ExtremeOpt", "").replace("TightOPT", "").replace("OPT", "").replace("Opt", "")
            hl = " ".join(hl.split())
        if hl.strip():
            clean_header.append(hl)
        
    return energy, "\n".join(clean_header), atoms

def query_and_package_stages(input_dir):
    print(f"\n{Colors.OKCYAN}--- Data Packaging Query ---{Colors.ENDC}")
    print("Which previous stage(s) would you like to package data from for Spectroscopy?")
    
    stages = [
        {"id": "2.6", "name": "Extrapolated Isomers", "search": "Stage2_6_HighLevel_*/*.out", "desc": "High-accuracy monomer extrapolations"},
        {"id": "3.7", "name": "Ultimate Strong Complexes", "search": "Stage3_7_UltimateComplex_*/*.out", "desc": "CCSD(T) strong interaction clusters"},
        {"id": "4.9", "name": "Ultimate Weak Complexes", "search": "Stage4_9_UltimateWeak_*/*.out", "desc": "CCSD(T) weak interaction conformers"},
        {"id": "5.5", "name": "Ultimate Isotopologues", "search": "Stage5_5_Ultimate_*/*.out", "desc": "High-accuracy isotopic structural variants"},
        {"id": "7", "name": "Solvation Shells", "search": "Stage7_*/*.out", "desc": "Optimized solvent shell systems"}
    ]
    
    valid_stages = []
    for s in stages:
        if glob.glob(s['search']): valid_stages.append(s)
            
    if not valid_stages:
        print_status("No automated stage outputs detected on disk.", "warning")
        print("Please manually place your .out files into the 'Ini_Spec_Opt-xyz' folder.")
        return
        
    for i, s in enumerate(valid_stages):
        print(f" [{i+1}] Stage {s['id']} ({s['name']}) - {s['desc']}")
    print(" [M] Manual Placement (I will place .out files myself)")
    
    ans = input("Select stages to package (e.g., 1,3 or M): ").strip().upper()
    if ans == 'M':
        print_status(f"Awaiting manual placement of .out files into '{input_dir}'.", "info")
        return
        
    selected_indices = [int(x.strip())-1 for x in ans.split(',') if x.strip().isdigit()]
    
    for idx in selected_indices:
        if 0 <= idx < len(valid_stages):
            s = valid_stages[idx]
            print_status(f"Packaging {s['name']}...", "info")
            
            out_files = glob.glob(s['search'])
            count = 0
            for out_f in out_files:
                with open(out_f, 'r', encoding='utf-8', errors='ignore') as check:
                    if "ORCA TERMINATED NORMALLY" in check.read():
                        shutil.copy(out_f, os.path.join(input_dir, os.path.basename(out_f)))
                        count += 1
            print_status(f"Packaged {count} successfully completed .out files.", "success")

# ==============================================================================
# MAIN EXECUTION ROUTINE
# ==============================================================================
def main():
    print(f"\n{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD} ChemComp_OPI-ORCA_v3-1 - STAGE 9: SPECTROSCOPY EXPORT{Colors.ENDC}")
    print(f"{Colors.HEADER}{Colors.BOLD}======================================================{Colors.ENDC}")

    if not os.path.exists(INPUT_DIR):
        os.makedirs(INPUT_DIR)
        print_status(f"Created '{INPUT_DIR}' directory.", "info")
        query_and_package_stages(INPUT_DIR)
    else:
        out_files = glob.glob(os.path.join(INPUT_DIR, "*.out"))
        if not out_files:
            print_status(f"'{INPUT_DIR}' is empty.", "warning")
            query_and_package_stages(INPUT_DIR)

    out_files = glob.glob(os.path.join(INPUT_DIR, "*.out"))
    if not out_files:
        print_status(f"No '.out' files found in '{INPUT_DIR}'. Please add them and re-run.", "error")
        sys.exit(0)
        
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    # ---------------------------------------------------------
    # INVENTORY & SYMMETRY DETECTION
    # ---------------------------------------------------------
    print(f"\n{Colors.BOLD}--- Target Inventory & Symmetry Detection ---{Colors.ENDC}")
    print(f"{Colors.OKCYAN}Informational: These spectroscopy calculations will NOT optimize the geometry.{Colors.ENDC}")
    print(f"{Colors.OKCYAN}They will explicitly inherit the exact keywords from your source files.{Colors.ENDC}\n")
    
    inventory = []
    has_non_c1 = False
    
    for out_f in out_files:
        base_name = os.path.basename(out_f).replace('.out', '')
        energy, header, atoms = parse_orca_out_for_spec(out_f)
        
        if not atoms:
            print_status(f"Failed to extract geometry from {os.path.basename(out_f)}. Skipping.", "warning")
            continue
            
        pg = get_point_group(atoms)
        if pg != "C1" and "C1" not in pg:
            has_non_c1 = True
            
        inventory.append({
            'name': base_name,
            'energy': energy,
            'header': header,
            'pg': pg,
            'atoms': atoms
        })
        
        print(f" • File: {Colors.OKCYAN}{os.path.basename(out_f)}{Colors.ENDC}")
        print(f"   Point Group: {Colors.WARNING}{pg}{Colors.ENDC}")
        print(f"   Header Block:\n{Colors.OKGREEN}{header}{Colors.ENDC}\n")

    # ---------------------------------------------------------
    # SYMMETRY INTERACTION LOGIC
    # ---------------------------------------------------------
    sym_choice = 3 
    
    if has_non_c1:
        print(f"{Colors.WARNING}--- Symmetry Interaction Alert ---{Colors.ENDC}")
        print("Symmetry (Point Groups > C1) has been detected in your isomers.")
        print("In ORCA, utilizing symmetry (UseSym) can massively accelerate calculations")
        print("by reducing the number of displaced geometries evaluated during VPT2.")
        print("However, numerical noise in near-symmetric structures can sometimes cause")
        print("imaginary frequencies, optimization failures, or mode-mixing artifacts.")
        print("\nHow would you like to handle symmetry?")
        print(" [1] Enforce Symmetry (Strict alignment via MolSym, append 'UseSym')")
        print(" [2] Allow Symmetry (Maintain current coords, append 'UseSym')")
        print(" [3] Force C1 (No symmetry, append 'NoSym')")
        
        while True:
            ans = input("Select an option [1/2/3]: ").strip()
            if ans in ['1', '2', '3']:
                sym_choice = int(ans)
                break
            else:
                print("Invalid selection.")
    else:
        print_status("Only C1 symmetry detected across all isomers. MolSym pre-optimization bypassed.", "info")

    # ---------------------------------------------------------
    # EXECUTION
    # ---------------------------------------------------------
    inventory.sort(key=lambda x: x['energy'])
    charge = user_config.get("charge", 0)
    mult = user_config.get("multiplicity", 1)
    
    print(f"\n{Colors.BOLD}--- Executing VPT2 Spectroscopic Predictions ---{Colors.ENDC}")
    
    for item in inventory:
        iso_name = item['name']
        atoms = item['atoms']
        header = item['header']
        pg = item['pg']
        
        print_status(f"Processing {iso_name}...", "info")
        
        # Checkpoint Validation
        vpt2_out = os.path.join(OUTPUT_DIR, f"{iso_name}_VPT2.out")
        if os.path.exists(vpt2_out):
            with open(vpt2_out, 'r', encoding='utf-8', errors='ignore') as check:
                if "ORCA TERMINATED NORMALLY" in check.read():
                    print_status(f"VPT2 calculation already complete for {iso_name}. Skipping.", "success")
                    continue
        
        if sym_choice in [1, 2] and pg != "C1" and "C1" not in pg:
            print_status(f"MolSym strict pre-optimization applied to {iso_name} to guarantee idealized alignment.", "info")
            atoms = apply_molsym_preopt(atoms, iso_name, 0, OUTPUT_DIR, prefix="molsym_vpt2")
            
        sym_flag = "UseSym" if sym_choice in [1, 2] else "NoSym"
        
        header_lines = header.split("\n")
        new_header_lines = []
        for hl in header_lines:
            if hl.startswith("!"):
                hl = hl.replace("UseSym", "").replace("NoSym", "")
                hl = f"{hl} {sym_flag} NumFreq VPT2"
                hl = " ".join(hl.split())
            new_header_lines.append(hl)
                
        final_header = "\n".join(new_header_lines)
        pickett_name = f"{iso_name}_Pickett.txt"
        
        inp_name = f"{iso_name}_VPT2.inp"
        xyz_name = f"{iso_name}_VPT2.xyz"
        
        write(os.path.join(OUTPUT_DIR, xyz_name), atoms)
        
        # Construct the specialized VPT2 / Pickett output block
        inp_content = f"{final_header}\n"
        inp_content += "%elprop\n  Dipole true\nend\n"
        inp_content += f"%output\n  Pickettname \"{pickett_name}\"\nend\n"
        inp_content += f"%freq\n  NumFreq true\n  DX 0.005\nend\n"
        inp_content += f"* xyzfile {charge} {mult} {xyz_name}\n"
        
        with open(os.path.join(OUTPUT_DIR, inp_name), "w") as f:
            f.write(inp_content)
            
        run_orca_safely(f"VPT2: {iso_name}", inp_name, OUTPUT_DIR, sys_config, exit_on_fail=False)
        
    print(f"\n{Colors.OKGREEN}{Colors.BOLD}🏁 STAGE 9 COMPLETE: Spectroscopic Pickett Files Generated in '{OUTPUT_DIR}'.{Colors.ENDC}")

if __name__ == "__main__":
    main()
